<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/05_cx_agentic_graph_rag_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️ CX — Investigating a Product-Launch Crisis with Agentic RAG + GraphRAG
## One question, three retrieval architectures — each one answers what the last could not

> **The scenario.** You are on the management team of **Vantage Instruments**, an electronics
> company. **Project Orion**, the flagship launch of the year, has just slipped six weeks. The CEO
> walks into the room and asks one question:
>
> > **"Why was Orion delayed, what supplier or component caused it, and which of our other
> > programmes are exposed to the same risk?"**
>
> The evidence is scattered across status reports, procurement registers, engineering BOMs, supplier
> incident bulletins and meeting minutes. **No single document contains the answer.** That is not an
> accident of this notebook — that is what corporate knowledge actually looks like.

You build **four** retrieval architectures against that one question, and watch each one fail in a *different, informative* way:

| | **Part 1 · Classical** | **Part 2 · Agentic** | **Parts 4–5 · GraphRAG** | **Part 6 · Combined** |
|---|---|---|---|---|
| **The question it answers** | "Find me passages about Orion." | "What am I missing, and what should I search for next?" | "How are these things *connected*?" | "Which of those is this?" |
| **Retrieval shape** | one shot, top-k | a loop that reacts to what it found | traversal over explicit relations | routed, per turn |
| **Breaks when** | the answer spans several documents | the fact lives in a **join**, not in any text | you need narrative, dates and causes | (costs the most) |

*Instructors: Parts 1–6 are the core arc. Part 7 (local vs global) is a self-contained extension —
drop it if you are short on time.*

**⚠️ The trap you should watch for.** The obvious answer to the CEO — *"a supplier had a factory
shutdown"* — is **true and insufficient**. There is a second-order exposure in this company that
**no amount of semantic search will ever surface**, because no document in the corpus mentions the two
entities in the same sentence. Finding it is the whole point of Part 4.

**How to read this notebook** — cells marked **🔧 PROVIDED — run, don't edit** hand you the plumbing
(corpus, embeddings, LLM wrapper, visuals). Cells marked **🎯** are yours to write. There are only
eight of them, and they are short: **the 🎯 cells are the architecture**, and everything else is
scaffolding so that you never spend the session debugging embeddings or entity extraction.

Inside a 🎯 cell, every `...` is a blank you have to fill, and each one is preceded by a
`# 🎯 TODO` comment telling you the signature of the helper to call and the shape it returns.
Everything else in those cells — bookkeeping, prints, dict keys the provided code depends on —
is already written: leave it as it is. An unfilled `...` raises rather than failing quietly, so
run the cell and read the error if you are unsure whether you are done.

In [1]:
#@title 🗺️ Roadmap — one question, four retrieval architectures { display-mode: "form" }
from IPython.display import HTML, display
_steps = [("🔍", "Part 1 · Classical RAG", "one shot, top-k", "retrieve once, answer once — and hit a wall"),
          ("🔄", "Part 2 · Agentic RAG", "search → reflect → search", "let the query evolve as evidence arrives"),
          ("🕸️", "Part 4 · GraphRAG", "traverse relations", "follow explicit edges no passage states"),
          ("🧭", "Part 6 · Combined", "an agent that routes", "choose the right retriever per question")]
_grad = ["#667eea", "#7d5fd0", "#9a63d4", "#c05fb0"]
_b = ""
for (_ic, _t, _q, _d), _g in zip(_steps, _grad):
    _b += (f'<div class="cxr-step"><div class="cxr-ic" style="background:linear-gradient(135deg,{_g},{_g}cc)">{_ic}</div>'
           f'<div class="cxr-t">{_t}</div><div class="cxr-q">{_q}</div><div class="cxr-d">{_d}</div></div>'
           '<div class="cxr-ar">➜</div>')
_b = _b.rsplit('<div class="cxr-ar">➜</div>', 1)[0]
display(HTML(f'''
<style>
.cxr{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff}}
.cxr-h{{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}}
.cxr-s{{font-size:12px;color:#6b6685;margin:0 0 16px}}
.cxr-row{{display:flex;align-items:stretch;flex-wrap:wrap}}
.cxr-step{{flex:1 1 170px;min-width:160px;text-align:center;padding:0 8px}}
.cxr-ic{{width:54px;height:54px;border-radius:50%;margin:0 auto 8px;display:flex;align-items:center;justify-content:center;font-size:24px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}}
.cxr-t{{font-weight:800;font-size:13px;color:#2c2350}}
.cxr-q{{font-size:11px;color:#8b5cf6;margin-top:3px;font-weight:700}}
.cxr-d{{font-size:10.5px;color:#8b86a6;margin-top:5px;line-height:1.3}}
.cxr-ar{{display:flex;align-items:center;font-size:18px;color:#b9a9e6;flex:0 0 16px}}
</style>
<div class="cxr"><div class="cxr-h">🗺️ Same question, four architectures</div>
<div class="cxr-s">The CEO's question never changes. What changes is <b>how the system is allowed to look for the answer</b> — and that alone decides whether it finds the hidden risk.</div>
<div class="cxr-row">{_b}</div></div>'''))

---
## 0. Setup

Nothing to clone. You need **one OpenRouter API key** and about a minute of downloads. No GPU
required — the embedding model is tiny and runs on CPU.

**0.1 — Install dependencies.** `sentence-transformers` gives us real dense embeddings (so the
retrieval failures you see later are *genuine* semantic failures, not a weak keyword matcher),
`networkx` gives us the graph.

In [2]:
%pip install -q "sentence-transformers>=3.0" "networkx>=3.0" "requests>=2.31" "numpy>=1.24"

**0.2 — 🔧 PROVIDED — the LLM wrapper.** Every LLM call in this notebook goes through `llm()` (free
text) or `llm_json()` (structured output). Both hit **OpenRouter**. The cell asks for your key, then
picks the first model from a preference list that OpenRouter actually serves today — so the notebook
keeps working when model names change.

`LLM_CALLS` is a call meter. Keep an eye on it: agentic retrieval buys its extra recall with extra
calls, and at the end we compare the bill.

In [3]:
#@title 🔧 PROVIDED — OpenRouter LLM wrapper (run, don't edit) { display-mode: "form" }
import os, json, re, getpass, requests

# --- key ---------------------------------------------------------------------
if not os.environ.get("OPENROUTER_API_KEY"):
    try:                                            # Colab secret, if you set one
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (sk-or-...): ")

API_KEY  = os.environ["OPENROUTER_API_KEY"]
BASE_URL = "https://openrouter.ai/api/v1"
HEADERS  = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# --- model selection: first preference that OpenRouter actually serves --------
PREFERRED = ["anthropic/claude-sonnet-4.5",
             "anthropic/claude-3.7-sonnet",
             "openai/gpt-4.1-mini",
             "google/gemini-2.0-flash-001",
             "meta-llama/llama-3.3-70b-instruct"]

def _pick_model():
    try:
        available = {m["id"] for m in requests.get(f"{BASE_URL}/models", timeout=20).json()["data"]}
        for m in PREFERRED:
            if m in available:
                return m
    except Exception as e:
        print("⚠️  could not list models:", e)
    return PREFERRED[0]

MODEL = _pick_model()
LLM_CALLS = {"n": 0, "in_chars": 0}

def llm(prompt, system=None, temperature=0.0, max_tokens=900):
    """One chat completion. Returns the assistant's text."""
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    body = {"model": MODEL, "messages": msgs,
            "temperature": temperature, "max_tokens": max_tokens}
    last = None
    for _ in range(3):                                       # brief retry: routers hiccup
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS,
                              json=body, timeout=120)
            r.raise_for_status()
            LLM_CALLS["n"] += 1
            LLM_CALLS["in_chars"] += len(prompt) + len(system or "")
            return r.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            last = e
    raise RuntimeError(f"OpenRouter call failed after 3 tries: {last}")

def llm_json(prompt, system=None, temperature=0.0, max_tokens=700):
    """Same, but insists on a single JSON object and parses it (tolerates code fences)."""
    sys = (system or "") + "\nReply with ONE JSON object and nothing else. No prose, no code fences."
    raw = llm(prompt, system=sys.strip(), temperature=temperature, max_tokens=max_tokens)
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.S)                  # last resort: first balanced-looking blob
        if m:
            return json.loads(m.group(0))
        raise ValueError(f"model did not return JSON:\n{raw[:400]}")

print(f"✅ LLM ready · model = {MODEL}")

OpenRouter API key (sk-or-...): ··········
✅ LLM ready · model = anthropic/claude-sonnet-4.5


**0.3 — 🔧 PROVIDED — the corpus.** 25 short documents, the kind of thing that actually lives in a
company: status reports, engineering BOMs, procurement registers, supplier bulletins, an old risk
review, meeting minutes. Every document carries a **date** and a **source type** — the metadata
any production retrieval system leans on.

Read a few. Notice that each one is individually unremarkable.

In [4]:
#@title 🔧 PROVIDED — the corpus (run, don't edit) { display-mode: "form" }
def _d(id, title, date, kind, text): return dict(id=id, title=title, date=date, kind=kind, text=text)

DOCS = [
    # ---------- programme status ----------
    _d("D01", "Project Orion — status report", "2026-03-12", "status report",
       "Project Orion (AR field headset) has slipped its launch window by six weeks. Hardware "
       "validation and firmware are complete and the team is idle. The programme is blocked on "
       "inbound optics: the OS-17 optical sensor is the single long-lead item on the critical path "
       "and no confirmed delivery date is available. All other subassemblies are in stock."),
    _d("D02", "Project Nova — status report", "2026-03-10", "status report",
       "Project Nova (industrial inspection camera) has entered final validation and remains on "
       "schedule for a Q3 release. Engineering reports no blocking issues this cycle. Procurement "
       "has raised no exceptions."),
    _d("D03", "Project Helios — status report", "2026-03-11", "status report",
       "Project Helios (thermal drone payload) completed thermal qualification ahead of plan. "
       "Schedule is green. No outstanding supplier actions are recorded against this programme."),

    # ---------- engineering dependencies (BOMs) ----------
    _d("D10", "Engineering BOM extract — Orion", "2025-09-02", "engineering dependency",
       "Project Orion bill of materials, optical and mainboard sections: OS-17 optical sensor (1x), "
       "PCB-K3 mainboard (1x). OS-17 is flagged as a critical single-source part."),
    _d("D11", "Engineering BOM extract — Nova", "2025-09-02", "engineering dependency",
       "Project Nova bill of materials: OS-17 optical sensor (2x), PCB-K3 mainboard (1x). Nova "
       "shares the OS-17 sensor line with other programmes."),
    _d("D12", "Engineering BOM extract — Helios", "2025-09-02", "engineering dependency",
       "Project Helios bill of materials: IR-9 infrared imaging module (1x), GIM-3 gimbal assembly "
       "(1x). No optical sensor of the OS series is used on this platform, and no part is shared "
       "with the Orion or Nova programmes."),
    _d("D13", "Engineering BOM extract — Luna", "2025-09-02", "engineering dependency",
       "Project Luna bill of materials: BAT-4 battery module (1x), PCB-K3 mainboard (1x)."),
    _d("D14", "Engineering BOM extract — Vega", "2025-09-02", "engineering dependency",
       "Project Vega bill of materials: OS-22 optical sensor (4x), BAT-4 battery module (2x), "
       "CS-9 telemetry SDK."),
    _d("D15", "Engineering BOM extract — Atlas", "2025-09-02", "engineering dependency",
       "Project Atlas is a software-only platform. Dependencies: CS-9 telemetry SDK. No bespoke "
       "hardware."),

    # ---------- procurement registers ----------
    _d("D20", "Procurement register — optical sensors", "2025-06-18", "procurement note",
       "The OS-17 and OS-22 optical sensors are procured from Apex Components (Kaohsiung). Apex is "
       "sole-source for both parts; no qualified second source exists as of this revision."),
    _d("D21", "Procurement register — imaging modules", "2025-06-18", "procurement note",
       "The IR-9 infrared imaging module is procured from Meridian Optics (Penang) under a "
       "three-year framework agreement."),
    _d("D22", "Procurement register — power", "2025-06-18", "procurement note",
       "The BAT-4 battery module is procured from Voltix Energy. Dual-sourcing was evaluated and "
       "deferred on cost grounds."),
    _d("D23", "Procurement register — software", "2025-06-18", "procurement note",
       "The CS-9 telemetry SDK is licensed from CloudSync Ltd under an annual subscription."),
    _d("D24", "Procurement register — mainboards", "2025-06-18", "procurement note",
       "The PCB-K3 mainboard is manufactured by Kestrel Fabrication."),
    _d("D25", "Procurement register — mechanical assemblies", "2025-06-18", "procurement note",
       "The GIM-3 gimbal assembly is procured from Aerodyne Mechanics under a bespoke agreement. "
       "This part is used on one programme only."),

    # ---------- supplier incidents & performance ----------
    _d("D30", "Supplier incident report — Apex Components", "2026-03-05", "supplier incident",
       "Apex Components suspended production at its Kaohsiung facility on 4 March 2026 following an "
       "upstream materials shortage. All outstanding shipments are on hold. Apex has not issued a "
       "recovery date and has declined to confirm allocation priorities."),
    _d("D31", "Apex Components — supplier disclosure letter", "2026-03-09", "supplier incident",
       "In its disclosure to customers, Apex Components attributes the production suspension to the "
       "loss of photonic wafer supply from Sanko Photonics, which has been its sole wafer vendor "
       "since the 2025 sourcing consolidation."),
    _d("D32", "Regional incident bulletin — Sanko Photonics", "2026-02-21", "supplier incident",
       "A cleanroom fire on 19 February 2026 destroyed Facility 2 at Sanko Photonics. Photonic "
       "wafer output is expected to be interrupted for 10 to 14 weeks. Sanko has invoked force "
       "majeure with its downstream customers."),
    _d("D33", "Sanko Photonics — regional customer register", "2026-01-15", "procurement note",
       "Sanko Photonics supplies photonic wafers to two qualified regional integrators: Apex "
       "Components and Meridian Optics. Both hold sole-source status for the wafer grades listed."),
    _d("D34", "Supplier incident report — CloudSync Ltd", "2026-03-02", "supplier incident",
       "CloudSync Ltd reported a four-hour API gateway outage affecting telemetry ingestion for all "
       "subscribers. Service was restored the same day with no data loss. Root cause: a failed "
       "certificate rotation."),
    _d("D35", "Supplier performance note — Voltix Energy", "2026-02-27", "supplier incident",
       "Voltix Energy delivered 100% on time in the trailing quarter with zero quality escapes. No "
       "production interruptions reported."),

    # ---------- risk reviews (the temporal trap) ----------
    _d("D40", "Annual supplier risk review", "2024-11-20", "risk review",
       "Apex Components is rated LOW RISK. Apex has recorded no production interruptions in five "
       "years and maintains dual-sourced wafer supply, which materially reduces single-point "
       "failure exposure. No mitigation actions are recommended at this time."),
    _d("D41", "Sourcing consolidation memo", "2025-11-04", "procurement note",
       "Effective Q1 2026, photonic wafer sourcing across our optical supply base is consolidated "
       "to a single qualified vendor to secure volume pricing. This supersedes the dual-source "
       "assumptions recorded in the 2024 annual supplier risk review."),

    # ---------- management ----------
    _d("D50", "Management meeting minutes", "2026-03-13", "meeting minutes",
       "The CEO asked for a written explanation of the Orion slip and an assessment of which other "
       "programmes carry the same exposure. Owner: COO. Due: end of week. The board briefing is on "
       "the 20th."),
    _d("D51", "Product roadmap 2026", "2026-01-08", "roadmap",
       "Launch windows for 2026: Orion (Q2), Nova (Q3), Helios (Q3), Luna (Q4), Vega (Q4), "
       "Atlas (rolling releases). Orion and Nova are the revenue-critical programmes."),
]

BY_ID = {d["id"]: d for d in DOCS}
print(f"{len(DOCS)} documents loaded · kinds: {sorted({d['kind'] for d in DOCS})}")
print("longest document:", max(len(d['text']) for d in DOCS), "characters — everything fits in one chunk")

25 documents loaded · kinds: ['engineering dependency', 'meeting minutes', 'procurement note', 'risk review', 'roadmap', 'status report', 'supplier incident']
longest document: 346 characters — everything fits in one chunk


In [5]:
#@title 🗂️ The corpus at a glance — and the four documents that matter { display-mode: "form" }
from IPython.display import HTML, display
import json as _json

_KIND_COLOR = {"status report": "#667eea", "engineering dependency": "#7d5fd0",
               "procurement note": "#9a63d4", "supplier incident": "#e0796d",
               "risk review": "#e0a23c", "meeting minutes": "#4c8dd8", "roadmap": "#39b36a"}
# the minimal set of documents that, read together, answer the CEO — revealed on click
_CHAIN = ["D01", "D10", "D20", "D30", "D31", "D32", "D33", "D21", "D12", "D11"]

_cards = ""
for _d0 in DOCS:
    _c = _KIND_COLOR[_d0["kind"]]
    _cards += (f'<div class="cm-card" data-id="{_d0["id"]}" style="border-top:3px solid {_c}">'
               f'<div class="cm-id">{_d0["id"]} · {_d0["date"]}</div>'
               f'<div class="cm-t">{_d0["title"]}</div>'
               f'<div class="cm-k" style="color:{_c}">{_d0["kind"]}</div></div>')

display(HTML('''
<style>
.cm{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:940px;color:#2c2350}
.cm-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.cm-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.cm-grid{display:flex;flex-wrap:wrap;gap:8px}
.cm-card{flex:1 1 148px;min-width:140px;background:#fff;border:1px solid #e7e4f6;border-radius:11px;padding:9px 11px;transition:.25s;opacity:1}
.cm-card.dim{opacity:.2}
.cm-card.hit{box-shadow:0 6px 18px rgba(224,121,109,.35);transform:translateY(-2px)}
.cm-id{font-size:9.5px;color:#a9a3c4;font-family:ui-monospace,Menlo,monospace}
.cm-t{font-size:11.5px;font-weight:700;color:#2c2350;line-height:1.3;margin:3px 0 4px}
.cm-k{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.4px}
.cm-btn{cursor:pointer;border:none;border-radius:10px;padding:8px 14px;font-size:12.5px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);margin:0 8px 12px 0}
.cm-btn.alt{background:#fff;color:#4b3f7a;border:1px solid #d9d5ee}
.cm-foot{margin-top:14px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:9px 12px;line-height:1.55}
</style>
<div class="cm">
 <div class="cm-h">🗂️ 25 documents. None of them is "the answer."</div>
 <div class="cm-s">This is the corpus your systems will search. Each document is individually boring and locally true. The CEO's answer exists only in the <b>relationship between them</b>. Colour = source type.</div>
 <button class="cm-btn" onclick="cmChain()">🔦 Show the documents that form the causal chain</button>
 <button class="cm-btn alt" onclick="cmReset()">reset</button>
 <div class="cm-grid" id="cmGrid">__CARDS__</div>
 <div class="cm-foot" id="cmFoot">💡 Click the button: <b>ten</b> of these 25 documents have to be read <i>together</i>, in the right order, to answer the CEO. A top-5 retriever gets to pick five.</div>
</div>
<script>
const cmChainIds = __CHAIN__;
function cmReset(){
  document.querySelectorAll('#cmGrid .cm-card').forEach(c=>{c.classList.remove('dim');c.classList.remove('hit');});
  document.getElementById('cmFoot').innerHTML = '💡 Click the button: <b>ten</b> of these 25 documents have to be read <i>together</i>, in the right order, to answer the CEO. A top-5 retriever gets to pick five.';
}
function cmChain(){
  document.querySelectorAll('#cmGrid .cm-card').forEach(c=>{
    if(cmChainIds.includes(c.dataset.id)){c.classList.add('hit');c.classList.remove('dim');}
    else {c.classList.add('dim');c.classList.remove('hit');}
  });
  document.getElementById('cmFoot').innerHTML = '🔦 The chain: <b>D01</b> Orion is blocked on OS-17 → <b>D20</b> OS-17 comes from Apex → <b>D30</b> Apex halted production → <b>D31</b> because its wafer vendor Sanko failed → <b>D32</b> Sanko had a cleanroom fire → <b>D33</b> Sanko also supplies Meridian → <b>D12</b> Meridian makes IR-9, which Helios uses. Notice that <b>no single document</b> mentions both <i>Orion</i> and <i>Helios</i> as sharing a risk. That fact is not written down anywhere.';
}
</script>
'''.replace("__CARDS__", _cards).replace("__CHAIN__", _json.dumps(_CHAIN))))

**0.4 — 🔧 PROVIDED — the vector store and the two search tools.** Real dense embeddings
(`all-MiniLM-L6-v2`, 384-dim, runs on CPU in seconds). `vector_search` is cosine similarity over
the whole corpus; `keyword_search` is a simple lexical fallback, because a good agent should have
more than one way to look.

In [6]:
#@title 🔧 PROVIDED — embeddings · vector_search · keyword_search (run, don't edit) { display-mode: "form" }
import numpy as np
from sentence_transformers import SentenceTransformer

_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def _doc_text(d):
    return f"{d['title']}. {d['text']}"

EMB = _embedder.encode([_doc_text(d) for d in DOCS],
                       normalize_embeddings=True, show_progress_bar=False)

SEARCH_CALLS = {"vector": 0, "keyword": 0, "graph": 0}

def vector_search(query, top_k=5):
    """Dense semantic retrieval. Returns [{doc, score}] sorted by cosine similarity."""
    SEARCH_CALLS["vector"] += 1
    q = _embedder.encode([query], normalize_embeddings=True)[0]
    sims = EMB @ q
    order = np.argsort(-sims)[:top_k]
    return [{"doc": DOCS[i], "score": float(sims[i])} for i in order]

def keyword_search(query, top_k=5):
    """Lexical retrieval: score = how many query terms appear in the document."""
    SEARCH_CALLS["keyword"] += 1
    terms = [t for t in re.findall(r"[A-Za-z0-9\-]+", query.lower()) if len(t) > 2]
    scored = []
    for d in DOCS:
        hay = _doc_text(d).lower()
        hits = sum(1 for t in terms if t in hay)
        if hits:
            scored.append({"doc": d, "score": hits / max(len(terms), 1)})
    scored.sort(key=lambda r: -r["score"])
    return scored[:top_k]

print(f"✅ vector store ready · {EMB.shape[0]} docs × {EMB.shape[1]} dims")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ vector store ready · 25 docs × 384 dims


**0.5 — 🔧 PROVIDED — the scorecard, and the visual helpers.**

This is the measuring stick for the whole notebook. The CEO's question has **six** checkable facts in
its answer. Every system you build gets scored on the same six, so at the end the comparison is
*numbers from your run*, not vibes.

In [7]:
#@title 🔧 PROVIDED — gold facts + scoring + visual helpers (run, don't edit) { display-mode: "form" }
from IPython.display import HTML, display

PURPLE, PURPLE2 = "#667eea", "#764ba2"
GREEN, RED, AMBER, BLUE = "#39b36a", "#e0796d", "#e0a23c", "#4c8dd8"

# --- the six facts a complete answer must contain ----------------------------
GOLD = [
    dict(key="component", label="States that the blocking component is the OS-17 optical sensor"),
    dict(key="supplier",  label="States that OS-17 is supplied by Apex Components"),
    dict(key="proximate", label="States the proximate cause: Apex halted/suspended production"),
    dict(key="root",      label="Goes past the proximate cause to the UNDERLYING one: Apex stopped "
                                "because its own sub-tier wafer vendor, Sanko Photonics, lost "
                                "output. Credit if Sanko is named as the cause behind Apex's "
                                "stoppage, whether or not the cleanroom fire is mentioned"),
    dict(key="exposure1", label="Identifies Project Nova as exposed BECAUSE it also uses OS-17 "
                                "from Apex"),
    dict(key="exposure2", label="Identifies Project Helios as exposed BECAUSE it depends on "
                                "Meridian Optics / IR-9, which shares the sub-tier supplier Sanko "
                                "(not merely listing Helios as another programme)"),
]

_SCORE_CACHE = {}

def score_answer(text):
    """Which of the six gold facts does this answer actually ESTABLISH?

    Graded by an LLM judge, not by substring matching. That is not gold-plating: a good RAG
    answer often *names* an entity precisely in order to say it cannot be confirmed ("Apex is
    the likely supplier, but this is not established"). A keyword scorer counts that as a hit
    and reports a failure as a success — which would make this whole notebook's scoreboard lie."""
    key = (text or "").strip()[:6000]
    if key in _SCORE_CACHE:
        return _SCORE_CACHE[key]
    rubric = "\n".join(f'- {g["key"]}: {g["label"]}' for g in GOLD)
    out = llm_json(
        f"ANSWER UNDER REVIEW:\n\"\"\"\n{key}\n\"\"\"\n\nFACTS TO CHECK:\n{rubric}\n\n"
        "For each fact, does the answer AFFIRMATIVELY ESTABLISH it as a conclusion?\n"
        "Mark FALSE if the answer: says the evidence is insufficient; hedges ('likely', 'probably', "
        "'not definitively established'); merely mentions the entity without asserting the "
        "relationship; or raises it only as an open question or a recommendation to investigate.\n"
        "Mark TRUE only for a clear, committed claim that the fact holds.\n"
        "Return one boolean per fact key: "
        + json.dumps({g["key"]: True for g in GOLD}))
    got = {g["key"]: bool(out.get(g["key"])) for g in GOLD}
    _SCORE_CACHE[key] = got
    return got

ANSWERS = {}          # system name -> answer text, filled in as you go

def explain(title, body, tone="info", icon="💡"):
    """Themed explanation card; `body` is raw HTML."""
    edge = {"info": PURPLE, "warn": AMBER, "risk": RED, "good": GREEN}[tone]
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f7f8ff,#fbf5ff);
     border:1px solid #ecebff;border-left:5px solid {edge};border-radius:14px;padding:15px 18px;margin:8px 0;max-width:880px">
  <div style="font-size:15.5px;font-weight:800;color:#3b2d6b;margin-bottom:7px">{icon} {title}</div>
  <div style="font-size:13px;color:#3a3357;line-height:1.6">{body}</div>
</div>'''))

def _esc(s):
    return (s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def show_hits(hits, title="retrieved passages", note=""):
    """Render a ranked list of retrieval hits as cards."""
    rows = ""
    for i, h in enumerate(hits, 1):
        d, s = h["doc"], h["score"]
        w = int(max(4, min(100, s * 100)))
        rows += (f'<div style="background:#fff;border:1px solid #e7e4f6;border-radius:11px;padding:9px 12px;margin:6px 0">'
                 f'<div style="display:flex;align-items:center;gap:9px">'
                 f'<span style="font-weight:800;color:#b9a9e6;font-size:12px">#{i}</span>'
                 f'<span style="font-weight:700;font-size:12.5px;color:#2c2350">{_esc(d["title"])}</span>'
                 f'<span style="font-family:ui-monospace,Menlo,monospace;font-size:10px;color:#a9a3c4">{d["id"]} · {d["date"]}</span>'
                 f'<span style="margin-left:auto;display:flex;align-items:center;gap:6px">'
                 f'<span style="width:70px;height:7px;background:#efedf8;border-radius:4px;overflow:hidden;display:inline-block">'
                 f'<span style="display:block;height:100%;width:{w}%;background:{PURPLE}"></span></span>'
                 f'<span style="font-size:10.5px;color:#6b6685;font-variant-numeric:tabular-nums">{s:.2f}</span></span></div>'
                 f'<div style="font-size:11px;color:#5b5578;line-height:1.5;margin-top:5px">{_esc(d["text"][:230])}…</div></div>')
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
     border:1px solid #ecebff;border-radius:16px;padding:16px 18px;max-width:900px">
 <div style="font-size:14.5px;font-weight:800;color:#3b2d6b">🔍 {title}</div>
 <div style="font-size:11.5px;color:#6b6685;margin:2px 0 8px">{note}</div>{rows}</div>'''))

def fact_coverage(text, title="what this answer establishes", subtitle=""):
    """Score one answer against the six gold facts and render it."""
    got = score_answer(text)
    rows = ""
    for g in GOLD:
        ok = got[g["key"]]
        c, ic = (GREEN, "✓") if ok else (RED, "✗")
        rows += (f'<div style="display:flex;align-items:center;gap:10px;background:#fff;border:1px solid #e7e4f6;'
                 f'border-left:4px solid {c};border-radius:9px;padding:7px 11px;margin:5px 0">'
                 f'<span style="font-weight:800;color:{c};font-size:14px;width:14px">{ic}</span>'
                 f'<span style="font-size:12px;color:#2c2350">{g["label"]}</span></div>')
    n = sum(got.values())
    ring = GREEN if n == len(GOLD) else (AMBER if n >= 4 else RED)
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:#fff;border:1px solid #ecebff;
     border-top:4px solid {ring};border-radius:15px;padding:15px 18px;margin:8px 0;max-width:700px">
 <div style="display:flex;align-items:baseline;gap:10px">
  <div style="font-size:15px;font-weight:800;color:#3b2d6b">{title}</div>
  <div style="margin-left:auto;font-size:22px;font-weight:800;color:{ring};font-variant-numeric:tabular-nums">{n}/{len(GOLD)}</div></div>
 <div style="font-size:11.5px;color:#6b6685;margin:2px 0 9px">{subtitle}</div>{rows}</div>'''))
    return got

# --- architecture diagrams: every system in this notebook, drawn the same way -
_ARCH_C = {"q": BLUE, "vec": PURPLE, "llm": PURPLE2, "graph": "#9a63d4",
           "text": GREEN, "ans": "#2f9e5c", "route": RED, "off": AMBER}

_ARCH_TMPL = """
<style>
.ar{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f7f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:14px;padding:15px 18px;margin:8px 0;max-width:880px;color:#2c2350}
.ar-h{font-size:15.5px;font-weight:800;color:#3b2d6b}
.ar-s{font-size:11.5px;color:#6b6685;margin:2px 0 12px;line-height:1.5}
.ar-flow{display:flex;align-items:stretch;flex-wrap:wrap}
.ar-box{position:relative;box-sizing:border-box;flex:1 1 104px;min-width:96px;max-width:210px;background:#fff;border:1px solid #e7e4f6;border-radius:11px;padding:9px 9px 8px;text-align:center}
.ar-hot{box-shadow:0 5px 15px rgba(118,75,162,.25)}
.ar-fan{flex:1.6 1 152px;max-width:236px}
.ar-new{position:absolute;top:-7px;right:-5px;font-size:8px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;color:#fff;background:#e0796d;border-radius:5px;padding:1px 5px}
.ar-ic{font-size:16px;line-height:1.25}
.ar-t{font-size:11.5px;font-weight:800;color:#2c2350;margin-top:3px;line-height:1.3}
.ar-alt{font-family:ui-monospace,Menlo,monospace;font-size:9px;overflow-wrap:anywhere;font-weight:700;color:#4b3f7a;background:#f7f6fc;border-radius:5px;padding:3px 6px;margin-top:3px;text-align:left}
.ar-d{font-size:9.5px;color:#8b86a6;line-height:1.4;margin-top:5px;overflow-wrap:anywhere}
.ar-ar{display:flex;align-items:center;justify-content:center;color:#b9a9e6;font-size:14px;flex:0 0 15px}
.ar-loop{margin-top:9px;font-size:10.5px;font-weight:700;color:#764ba2;background:#f2f0fc;border:1px dashed #c3b5e8;border-radius:9px;padding:7px 11px;text-align:center}
.ar-note{margin-top:10px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:9px 12px;line-height:1.55}
.ar code{background:#fff;border-radius:4px;padding:1px 4px;font-size:9px}
</style>
<div class="ar">
 <div class="ar-h">🏗️ __TITLE__</div>
 <div class="ar-s">__SUB__</div>
 <div class="ar-flow">__FLOW__</div>
 __LOOP__
 __NOTE__
</div>"""

def architecture(title, steps, subtitle="", loop=None, note="", new=()):
    """Draw one retrieval system as a pipeline — same visual language every time.

    steps : [(kind, icon, label, sublabel)]. `label` may instead be a list of
            (kind, text) pairs, which renders as a fan of alternative branches.
    loop  : text for the dashed feedback bar under the row (None = no loop).
    new   : indices of the steps this architecture adds over the previous one.
    `sublabel`, `subtitle`, `loop` and `note` may contain HTML; `label` is escaped."""
    flow = ""
    for i, (kind, icon, label, sub) in enumerate(steps):
        c = _ARCH_C.get(kind, PURPLE)
        if isinstance(label, (list, tuple)):
            body = "".join(f'<div class="ar-alt" style="border-left:3px solid '
                           f'{_ARCH_C.get(k, PURPLE)}">{_esc(t)}</div>' for k, t in label)
        else:
            body = f'<div class="ar-t">{_esc(label)}</div>'
        fan = " ar-fan" if isinstance(label, (list, tuple)) else ""
        badge = '<span class="ar-new">new</span>' if i in new else ''
        hot = " ar-hot" if i in new else ""
        flow += (f'<div class="ar-box{hot}{fan}" style="border-top:3px solid {c}">{badge}'
                 f'<div class="ar-ic">{icon}</div>{body}'
                 f'<div class="ar-d">{sub}</div></div>')
        if i < len(steps) - 1:
            flow += '<div class="ar-ar">➜</div>'
    display(HTML(_ARCH_TMPL
                 .replace("__TITLE__", title)
                 .replace("__SUB__", subtitle)
                 .replace("__FLOW__", flow)
                 .replace("__LOOP__", f'<div class="ar-loop">↻ {loop}</div>' if loop else "")
                 .replace("__NOTE__", f'<div class="ar-note">{note}</div>' if note else "")))

print("✅ scorecard + visuals ready")

✅ scorecard + visuals ready


---
# Part 1 — Classical RAG hits a wall

The standard pipeline: **embed the question → take the top-5 → hand them to the model → answer.**
It is the right architecture for a huge number of problems. Let's watch it fail honestly.

In [8]:
CEO_QUESTION = ("Why was Project Orion delayed, what supplier or component caused it, "
                "and which other programmes are exposed to the same risk?")

hits = vector_search(CEO_QUESTION, top_k=5)
show_hits(hits, "Classical RAG · top-5 for the CEO's question",
          note="One embedding of the question, one pass over the corpus, five passages. This is all the model will see.")

Look carefully at what came back before reading on.

The retriever is not broken — every passage is *topically relevant*. It found Orion. It probably
found an incident report. But notice the shape of the failure: the question contains **three
sub-questions** (*why*, *who*, *who else*) and a single embedding is an average of all three. You get
passages that are moderately related to the blend, rather than the specific passages that each
sub-question needs.

In [9]:
#@title 🧱 The wall — one embedding, three information needs { display-mode: "form" }
from IPython.display import HTML, display

# Each hop of the causal chain, and the document(s) that actually assert it.
# Nothing below is hardcoded: we check the REAL top-5 from the cell above.
HOPS = [
    (1, "Orion is blocked on <b>OS-17</b>",                      ["D01", "D10"], "any"),
    (2, "OS-17 is supplied by <b>Apex Components</b>",           ["D20"],        "all"),
    (3, "Apex halted because <b>Sanko</b> lost wafer output",    ["D31"],        "all"),
    (4, "Sanko also supplies <b>Meridian Optics</b>",            ["D33"],        "all"),
    (5, "Meridian makes IR-9 → <b>Helios</b> is exposed",        ["D21", "D12"], "all"),
]
_top_ids = [h["doc"]["id"] for h in hits]

_needs = ""
_covered = 0
for _hop, _claim, _need, _mode in HOPS:
    _present = [d for d in _need if d in _top_ids]
    _ok = bool(_present) if _mode == "any" else len(_present) == len(_need)
    _covered += _ok
    _mark, _col = ("✓", "#39b36a") if _ok else (("~", "#e0a23c") if _present else ("✗", "#e0796d"))
    _got = (f'<span class="wl-src wl-hit">{" ".join(_present)}</span>' if _present else '') + \
           "".join(f'<span class="wl-src wl-miss">{d}</span>' for d in _need if d not in _top_ids)
    _needs += (f'<div class="wl-need"><span class="wl-hop" style="background:{_col}">HOP {_hop}</span>'
               f'<span class="wl-txt">{_claim}<br><span class="wl-lbl">asserted by</span>{_got}</span>'
               f'<span class="wl-x" style="color:{_col}">{_mark}</span></div>')

display(HTML('''
<style>
.wl{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:900px;color:#2c2350}
.wl-src{display:inline-block;font-family:ui-monospace,Menlo,monospace;font-size:9px;font-weight:700;border-radius:4px;padding:1px 5px;margin:2px 3px 0 0}
.wl-hit{background:#e7f6ee;color:#2f9e5c}
.wl-miss{background:#fdecea;color:#c0392b;text-decoration:line-through}
.wl-lbl{font-size:8.5px;text-transform:uppercase;letter-spacing:.4px;color:#b9a9e6;margin-right:4px}
.wl-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.wl-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.55}
.wl-row{display:flex;gap:14px;flex-wrap:wrap;align-items:center;justify-content:center}
.wl-q{flex:0 1 250px;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #667eea;border-radius:12px;padding:11px 13px;font-size:12px;line-height:1.5}
.wl-ar{font-size:22px;color:#b9a9e6}
.wl-col{flex:1 1 300px;display:flex;flex-direction:column;gap:7px}
.wl-need{background:#fff;border:1px solid #e7e4f6;border-radius:10px;padding:8px 11px;font-size:11.5px;display:flex;gap:9px;align-items:center}
.wl-hop{font-family:ui-monospace,Menlo,monospace;font-size:9.5px;font-weight:800;color:#fff;border-radius:5px;padding:2px 7px}
.wl-txt{flex:1;line-height:1.4}
.wl-x{font-weight:800;font-size:14px}
.wl-score{margin-top:13px;text-align:center;font-size:12px;color:#4b3f7a;background:#fff;border:1px solid #e7e4f6;border-radius:10px;padding:8px}
.wl-foot{margin-top:10px;font-size:11.5px;color:#7a3d34;background:#fdf3f1;border-left:3px solid #e0796d;border-radius:9px;padding:10px 12px;line-height:1.6}
.wl-vec{display:flex;align-items:center;gap:6px;justify-content:center;margin:12px 0 4px}
.wl-vchip{font-size:10.5px;font-weight:700;border-radius:6px;padding:3px 9px;background:#eef0ff;color:#4b3f7a}
.wl-plus{color:#b9a9e6;font-weight:800}
.wl-avg{background:#e0796d;color:#fff}
</style>
<div class="wl">
 <div class="wl-h">🧱 Why one embedding cannot do this</div>
 <div class="wl-s">A dense retriever compresses the whole question into <b>one point in space</b> and asks "what lies near it?". But the CEO asked three things at once, and the answer to each lives in a different corner of the corpus.</div>
 <div class="wl-vec">
   <span class="wl-vchip">why delayed?</span><span class="wl-plus">+</span>
   <span class="wl-vchip">which supplier?</span><span class="wl-plus">+</span>
   <span class="wl-vchip">who else is exposed?</span><span class="wl-plus">=</span>
   <span class="wl-vchip wl-avg">one blurry average vector</span>
 </div>
 <div class="wl-row">
  <div class="wl-q"><b>“Why was Orion delayed, what supplier/component caused it, and which other programmes are exposed?”</b><div style="font-size:10.5px;color:#8b86a6;margin-top:6px">one query · one vector · top-5<br>your run retrieved: <b>__TOP__</b></div></div>
  <div class="wl-ar">&rarr;</div>
  <div class="wl-col">__NEEDS__</div>
 </div>
 <div class="wl-score">Your single-shot retrieval covered <b>__COVERED__ of 5</b> hops in the causal chain.</div>
 <div class="wl-foot">🔑 <b>The lesson to take away:</b> this is <i>not</i> a similarity problem. Better embeddings, a bigger <code>top_k</code>, or a reranker will not fix it — because hop 3 is only <i>findable</i> once you know the word "Apex", and hop 5 is only findable once you know the word "Sanko". <b>The query you need next depends on what the last search returned.</b> A single-shot architecture structurally cannot do that.</div>
</div>
'''.replace("__NEEDS__", _needs)
   .replace("__TOP__", " ".join(_top_ids))
   .replace("__COVERED__", str(_covered))))

### 🎯 1.1 — Name what is missing

Before you write any agent, do the thing the agent will have to do: **audit the evidence gap**. Look
at the five passages above and fill in the two lists. The `missing` entries are the searches your
agent will eventually have to invent for itself.

In [10]:
known = [
    "Project Orion has slipped by six weeks",
    "Orion is blocked on the OS-17 optical sensor",
    "There is some kind of supplier disruption in the corpus",
]

# 🎯 TODO: what does the answer still require that the top-5 did not give you?
#          The first gap is filled in as an example of the format: short, concrete,
#          and phrased the way you would actually type it into a search box.
#          Add at least three more — these are the queries your agent will
#          have to invent for itself in Part 2.
missing = [
    "which supplier actually makes OS-17",      # ← example: keep this one
    "what Apex Components depends on",     # 🎯 hop 3 in the diagram above — what does OS-17 depend on?
    "other customers of Sanko Photonics",     # 🎯 hop 4 — and what sits behind THAT?
    "programmes using Meridian Optics parts",     # 🎯 hop 5 — who else is downstream of it?
]

assert len(missing) >= 4, "there are at least four distinct gaps — look again at the hop diagram"
print(f"✅ {len(known)} facts known · {len(missing)} gaps identified\n")
for i, m in enumerate(missing, 1):
    print(f"  gap {i}: {m}")

✅ 3 facts known · 4 gaps identified

  gap 1: which supplier actually makes OS-17
  gap 2: what Apex Components depends on
  gap 3: other customers of Sanko Photonics
  gap 4: programmes using Meridian Optics parts


### 1.2 — Let the classical system answer anyway

Now give those five passages to the LLM and demand the CEO's answer. This is the baseline every
later system has to beat. Watch what it does with the gaps: a good model will hedge; a bad one will
guess.

In [11]:
def build_context(docs):
    """Format documents into a numbered evidence block for the LLM."""
    return "\n\n".join(
        f"[{d['id']} · {d['date']} · {d['kind']}] {d['title']}\n{d['text']}" for d in docs)

ANSWER_SYSTEM = (
    "You are an analyst for Vantage Instruments. Answer ONLY from the evidence provided. "
    "If the evidence does not establish something, say so explicitly rather than guessing. "
    "Be concise and concrete: name components, suppliers and programmes by their exact identifiers.")

classical_answer = llm(
    f"Evidence:\n{build_context([h['doc'] for h in hits])}\n\nQuestion: {CEO_QUESTION}",
    system=ANSWER_SYSTEM)

ANSWERS["Classical RAG"] = classical_answer
print(classical_answer)

Based on the evidence provided:

## Why Project Orion was delayed

Project Orion slipped its launch window by six weeks because it is **blocked on inbound optics**. Specifically, the **OS-17 optical sensor** is the single long-lead item on the critical path with no confirmed delivery date. All other subassemblies are in stock, and hardware validation and firmware are complete.

## Supplier or component that caused the delay

**Component:** OS-17 optical sensor

**Supplier:** The evidence does not explicitly identify which supplier provides the OS-17. However, document D30 reports that **Apex Components** suspended production at its Kaohsiung facility on 4 March 2026 due to upstream materials shortage, with all outstanding shipments on hold and no recovery date issued.

The timing correlation is suggestive (Apex suspension on 4 March, Orion status report on 12 March citing OS-17 unavailability), but the evidence does not definitively establish that Apex supplies the OS-17.

## Other pro

In [12]:
fact_coverage(classical_answer, "Classical RAG · fact coverage",
              subtitle="Scored against the six facts a complete answer to the CEO must contain.")

{'component': True,
 'supplier': False,
 'proximate': False,
 'root': False,
 'exposure1': False,
 'exposure2': False}

In [13]:
#@title 🧠 Lesson 1 — what actually went wrong { display-mode: "form" }
architecture(
    "Part 1 · Classical RAG — one shot, one pass",
    [("q",   "❓", "the question",  "one string, fixed before anything is known"),
     ("vec", "🧮", "embed",         "the whole question &rarr; <b>one</b> 384-dim point"),
     ("vec", "📚", "vector search", "cosine over 25 docs &middot; keep the top 5"),
     ("llm", "🤖", "LLM",           "1 call &middot; answers from those 5 passages"),
     ("ans", "📝", "answer",        "scored against the six gold facts")],
    subtitle="Every arrow points forward. Nothing that comes back is allowed to change what was asked.",
    note="🔑 The query is written <b>once</b>, before the first document is seen — so a fact that only "
         "becomes searchable <i>after</i> you have read something else is structurally out of reach.")

explain("Retrieval failure ≠ bad similarity",
        "Every passage the retriever returned was <b>relevant</b>. The system still failed, because the "
        "question needed a <i>sequence</i> of retrievals where each query is only writable after the "
        "previous result comes back. You cannot search for <code>Sanko</code> before you have learned "
        "that the word <code>Sanko</code> exists.", tone="warn", icon="🧱")
explain("The diagnostic question",
        "Ask this of any RAG failure: <b>“is the answer <i>in</i> a passage, or <i>between</i> passages?”</b> "
        "If it is in a passage, fix retrieval — better embeddings, reranking, chunking. If it is between "
        "passages, no amount of retrieval tuning helps; you need a different <b>architecture</b>.",
        tone="info", icon="🔑")
explain("What we do next",
        "Give the system the ability to <b>search again</b>, with a query it writes itself based on what it "
        "just learned. That single change is the whole of agentic RAG.", tone="good", icon="🔄")

---
# Part 2 — Make retrieval agentic

The fix is almost embarrassingly simple: **put the retriever in a loop, and let the model write the
next query.**

The system now has a state (what it has found so far) and a decision to make at every step (what do
I still not know?). That is what makes it an *agent* rather than a pipeline — not the model, the
**control flow**.

In [14]:
#@title 🔄 The investigation loop — state, decision, action { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r'''
<style>
.lp{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.lp-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.lp-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.55}
.lp-wrap{display:flex;gap:18px;flex-wrap:wrap;align-items:stretch}
.lp-cycle{flex:1 1 330px;display:flex;flex-direction:column;gap:0}
.lp-node{background:#fff;border:1px solid #e7e4f6;border-radius:11px;padding:9px 13px;font-size:12px;display:flex;gap:10px;align-items:center}
.lp-ic{width:26px;height:26px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:13px;color:#fff;flex:0 0 26px}
.lp-nt{font-weight:800;font-size:12px}
.lp-nd{font-size:10.5px;color:#8b86a6;line-height:1.35}
.lp-down{text-align:center;color:#b9a9e6;font-size:15px;line-height:1;margin:3px 0}
.lp-back{margin-top:6px;text-align:center;font-size:11px;color:#764ba2;font-weight:700;background:#f2f0fc;border-radius:8px;padding:6px}
.lp-right{flex:1 1 300px;display:flex;flex-direction:column;gap:9px}
.lp-cmp{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:11px 13px}
.lp-ct{font-weight:800;font-size:12px;color:#3b2d6b;margin-bottom:5px}
.lp-cd{font-size:11px;color:#5b5578;line-height:1.5}
.lp-code{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;background:#f7f6fc;color:#4b3f7a;border-radius:5px;padding:1px 5px}
.lp-foot{margin-top:15px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="lp">
 <div class="lp-h">🔄 Agentic RAG is a control-flow change, not a model change</div>
 <div class="lp-s">Same corpus, same embeddings, same LLM as Part 1. The only difference is that retrieval now happens <b>inside a loop whose next input depends on its last output</b>.</div>
 <div class="lp-wrap">
  <div class="lp-cycle">
   <div class="lp-node"><span class="lp-ic" style="background:#667eea">?</span><span><span class="lp-nt">Question + evidence so far</span><div class="lp-nd">the agent's state — starts empty</div></span></div>
   <div class="lp-down">&darr;</div>
   <div class="lp-node"><span class="lp-ic" style="background:#7d5fd0">🧠</span><span><span class="lp-nt">What am I still missing?</span><div class="lp-nd">the reflection step — an LLM call over the state</div></span></div>
   <div class="lp-down">&darr;</div>
   <div class="lp-node"><span class="lp-ic" style="background:#9a63d4">✍️</span><span><span class="lp-nt">Write the next query</span><div class="lp-nd">a <i>new</i> query, using words learned from the last hop</div></span></div>
   <div class="lp-down">&darr;</div>
   <div class="lp-node"><span class="lp-ic" style="background:#c05fb0">🔍</span><span><span class="lp-nt">Search &amp; merge into evidence</span><div class="lp-nd">deduplicate — the same doc keeps coming back</div></span></div>
   <div class="lp-down">&darr;</div>
   <div class="lp-node"><span class="lp-ic" style="background:#39b36a">✅</span><span><span class="lp-nt">Enough to answer?</span><div class="lp-nd">the stopping rule — the hardest part to get right</div></span></div>
   <div class="lp-back">&#8630; if not: loop, with a smarter query than last time</div>
  </div>
  <div class="lp-right">
   <div class="lp-cmp"><div class="lp-ct">🔧 provided for you</div><div class="lp-cd">
     <span class="lp-code">propose_next_search(question, evidence)</span> — the reflection + query-writing step, as one structured LLM call.<br>
     <span class="lp-code">enough_evidence(question, evidence)</span> — the stopping rule.</div></div>
   <div class="lp-cmp" style="border-left:4px solid #e0796d"><div class="lp-ct">🎯 you write</div><div class="lp-cd">
     <span class="lp-code">investigate(question, max_steps)</span> — the loop itself. Four lines. That is the entire architecture.</div></div>
   <div class="lp-cmp"><div class="lp-ct">⚠️ the two failure modes</div><div class="lp-cd">
     <b>Never stops:</b> no stopping rule, or a rule that always says "one more". Bound it with <span class="lp-code">max_steps</span>.<br>
     <b>Loops on itself:</b> proposes the same query forever because you did not show it what it already searched.</div></div>
  </div>
 </div>
 <div class="lp-foot">💰 <b>The cost you are buying recall with:</b> classical RAG = 1 LLM call. Agentic RAG = 2–3 calls <i>per step</i>. Watch the <code>LLM_CALLS</code> meter at the end — that is the real trade-off you present to a budget owner.</div>
</div>
'''))

**🔧 PROVIDED — the two LLM helpers.** `propose_next_search` is the reflection step: it sees the
question, what has been found, and *what has already been searched*, and returns a structured plan.
`enough_evidence` is the stopping rule. Both use `llm_json`, so their output is a dict you can branch
on — this is what "structured output" buys you in agent design.

In [15]:
#@title 🔧 PROVIDED — propose_next_search · enough_evidence (run, don't edit) { display-mode: "form" }
def _evidence_digest(evidence, limit=1200):
    """Compact view of gathered evidence for the reflection prompts."""
    if not evidence:
        return "(nothing gathered yet)"
    return "\n".join(f"- [{d['id']}] {d['title']}: {d['text'][:limit]}" for d in evidence)

def propose_next_search(question, evidence, past_queries=()):
    """Reflect on the gap and write the next query. Returns dict with keys:
       missing (list[str]), query (str), reasoning (str)."""
    out = llm_json(
        f"INVESTIGATION QUESTION:\n{question}\n\n"
        f"EVIDENCE GATHERED SO FAR:\n{_evidence_digest(evidence)}\n\n"
        f"QUERIES ALREADY TRIED (do not repeat these):\n"
        f"{chr(10).join('- ' + q for q in past_queries) or '(none)'}\n\n"
        "You are running a multi-hop investigation over a document corpus. Identify what the "
        "question still requires that the evidence does not establish, then write ONE new search "
        "query targeting the single most important gap.\n"
        "Rules for the query: use the specific named entities you have just learned (part numbers, "
        "supplier names, project names) — that is how you make progress. Keep it under 12 words. "
        "It must be materially different from every query already tried.\n"
        'Return: {"missing": ["..."], "query": "...", "reasoning": "one sentence"}')
    out.setdefault("missing", []); out.setdefault("reasoning", "")
    return out

def enough_evidence(question, evidence):
    """Stopping rule. Returns dict with keys: sufficient (bool), why (str)."""
    out = llm_json(
        f"QUESTION:\n{question}\n\nEVIDENCE:\n{_evidence_digest(evidence)}\n\n"
        "Can this question be answered COMPLETELY and specifically from this evidence alone — every "
        "sub-question, with named entities and no guessing? Be strict: partial answers are not "
        "sufficient.\n"
        'Return: {"sufficient": true|false, "why": "one sentence"}')
    return {"sufficient": bool(out.get("sufficient")), "why": out.get("why", "")}

print("✅ reflection helpers ready")

✅ reflection helpers ready


### 🎯 2.1 — Write the loop

Everything above was scaffolding. **This function is the architecture.** Four things happen per step:

- **`propose_next_search(question, evidence, past_queries)`** → `{"query": ..., "missing": [...], "reasoning": ...}`
  Pass `past_queries` — without it the agent re-proposes its favourite query forever.
- **`vector_search(query, top_k=4)`** → `[{"doc":…, "score":…}]`
- **accumulate the new documents into `evidence`.** The dedup filter and `seen_ids` are
  written for you; what you add is the line that makes the evidence pile *grow* across steps.
- **`enough_evidence(question, evidence)`** → `{"sufficient": bool}` — break when it says yes.

The `trace` bookkeeping is written for you so the next cell can visualise the run.

In [27]:
def investigate(question, max_steps=5, top_k=4, verbose=True):
    """Iterative retrieval: search, reflect, search again with a better query."""
    evidence, seen_ids, past_queries, trace = [], set(), [], []

    for step in range(max_steps):
        # 🎯 TODO 1: reflect on the gap and get the next query.
        #   propose_next_search(question, evidence, past_queries)
        #       -> {"query": str, "missing": [str], "reasoning": str}
        #   Pass past_queries AND record the new query in it — without that the
        #   agent re-proposes its favourite query until max_steps.
        plan  = propose_next_search(question, evidence, past_queries)
        query = plan["query"]
        past_queries.append(query)

        # 🎯 TODO 2: run the search.
        #   vector_search(query, top_k=top_k) -> [{"doc":{...}, "score":float}]
        results = vector_search(query, top_k=top_k)

        # 🎯 TODO 3: `new_docs` below is this step's haul with anything already
        #   seen filtered out — only newly-seen documents count as progress.
        #   Accumulate it into `evidence`, the pile the agent reasons over and
        #   finally answers from. This one line is what makes the loop cumulative:
        #   skip it and every step throws away what the last one found.
        new_docs = [r["doc"] for r in results if r["doc"]["id"] not in seen_ids]
        evidence.extend(new_docs)
        seen_ids.update(d["id"] for d in new_docs)

        # --- bookkeeping for the trace visual (provided — leave it alone) ---
        trace.append(dict(step=step + 1, query=query, missing=plan["missing"],
                          reasoning=plan.get("reasoning", ""),
                          hits=[(r["doc"]["id"], r["doc"]["title"], r["score"]) for r in results],
                          new_ids=[d["id"] for d in new_docs]))
        if verbose:
            print(f"STEP {step + 1}  🔍 {query!r}")
            print(f"         → {len(new_docs)} new: {[d['id'] for d in new_docs] or '—'}")

        # 🎯 TODO 4: stop early if the evidence is already sufficient.
        #   enough_evidence(question, evidence) -> {"sufficient": bool, "why": str}
        verdict = enough_evidence(question, evidence)

        trace[-1]["sufficient"] = verdict["sufficient"]
        trace[-1]["why"] = verdict["why"]
        if verdict["sufficient"]:
            if verbose:
                print(f"         ✅ stopping: {verdict['why']}")
            break

    return evidence, trace
print("✅ investigate() defined")

✅ investigate() defined


### 2.2 — Run the investigation

Watch the **queries**, not the answers. The interesting thing is that step 3's query contains a word
that did not exist anywhere in step 1's world.

In [28]:
agentic_evidence, agentic_trace = investigate(CEO_QUESTION, max_steps=5)
print(f"\n📚 gathered {len(agentic_evidence)} distinct documents in {len(agentic_trace)} steps")

STEP 1  🔍 'Project Orion delay cause supplier component'
         → 4 new: ['D30', 'D01', 'D50', 'D31']
STEP 2  🔍 'OS-17 optical sensor Apex Components other projects programmes'
         → 3 new: ['D20', 'D11', 'D10']
         ✅ stopping: The evidence explicitly states Orion was delayed by the OS-17 optical sensor from Apex Components due to their Kaohsiung facility suspension caused by loss of photonic wafer supply from Sanko Photonics, and Project Nova is exposed to the same risk as it also uses the OS-17 sensor from the same sole-source supplier.

📚 gathered 7 distinct documents in 2 steps


In [29]:
#@title 🧭 Your investigation trace — watch the vocabulary grow { display-mode: "form" }
from IPython.display import HTML, display
_rows = ""
for _t in agentic_trace:
    _new = "".join(f'<span class="tr-new">{i}</span>' for i in _t["new_ids"]) or '<span class="tr-none">nothing new</span>'
    _miss = "".join(f'<li>{_esc(m)}</li>' for m in _t["missing"][:4])
    _stop = ('<span class="tr-stop yes">✅ sufficient — stop</span>' if _t.get("sufficient")
             else '<span class="tr-stop no">↻ still missing something — loop</span>')
    _rows += f'''
    <div class="tr-step">
      <div class="tr-n">{_t["step"]}</div>
      <div class="tr-body">
        <div class="tr-q">🔍 <b>{_esc(_t["query"])}</b></div>
        <div class="tr-r">{_esc(_t.get("reasoning",""))}</div>
        <div class="tr-lbl">it believed it was missing</div><ul class="tr-miss">{_miss}</ul>
        <div class="tr-lbl">new documents pulled in</div><div>{_new}</div>
        <div style="margin-top:7px">{_stop}</div>
      </div>
    </div>'''
display(HTML('''
<style>
.tr{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:900px;color:#2c2350}
.tr-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.tr-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.55}
.tr-step{display:flex;gap:12px;margin-bottom:10px}
.tr-n{flex:0 0 30px;height:30px;border-radius:50%;background:linear-gradient(135deg,#667eea,#764ba2);color:#fff;font-weight:800;display:flex;align-items:center;justify-content:center;font-size:13px}
.tr-body{flex:1;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px}
.tr-q{font-size:12.5px;color:#2c2350;font-family:ui-monospace,Menlo,monospace}
.tr-r{font-size:11px;color:#8b86a6;font-style:italic;margin:3px 0 7px;line-height:1.45}
.tr-lbl{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;color:#b9a9e6;margin-top:6px}
.tr-miss{margin:3px 0 0;padding-left:17px;font-size:11px;color:#5b5578;line-height:1.5}
.tr-new{display:inline-block;font-family:ui-monospace,Menlo,monospace;font-size:10px;font-weight:700;background:#e7f6ee;color:#2f9e5c;border-radius:5px;padding:2px 7px;margin:3px 3px 0 0}
.tr-none{font-size:10.5px;color:#c9c4dd;font-style:italic}
.tr-stop{font-size:10.5px;font-weight:800;border-radius:6px;padding:3px 9px}
.tr-stop.yes{background:#e7f6ee;color:#2f9e5c}
.tr-stop.no{background:#f2f0fc;color:#764ba2}
.tr-foot{margin-top:6px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="tr">
 <div class="tr-h">🧭 The trace — each query was unwritable one step earlier</div>
 <div class="tr-s">Read the queries top to bottom. Entity names appear that the agent had no way of knowing at the start. That is <b>iterative retrieval</b> doing its job.</div>
 __ROWS__
 <div class="tr-foot">🔑 This is the mechanism: <b>retrieval produces vocabulary, and vocabulary enables the next retrieval.</b> Classical RAG gets exactly one draw from that process.</div>
</div>'''.replace("__ROWS__", _rows)))

In [30]:
agentic_answer = llm(
    f"Evidence:\n{build_context(agentic_evidence)}\n\nQuestion: {CEO_QUESTION}",
    system=ANSWER_SYSTEM)
ANSWERS["Agentic RAG"] = agentic_answer
print(agentic_answer)

Based on the evidence provided:

## Why Project Orion was delayed

Project Orion was delayed by six weeks because it is blocked on inbound optics, specifically the **OS-17 optical sensor**, which is the single long-lead item on the critical path. Hardware validation and firmware are complete, but no confirmed delivery date is available for this component [D01].

## Supplier and component that caused the delay

- **Component**: OS-17 optical sensor
- **Direct supplier**: Apex Components (Kaohsiung facility) [D20]
- **Root cause**: Apex Components suspended production on 4 March 2026 due to loss of photonic wafer supply from **Sanko Photonics**, which has been Apex's sole wafer vendor since 2025 [D30, D31]

Apex is the sole-source supplier for the OS-17, with no qualified second source [D20].

## Other programmes exposed to the same risk

**Project Nova** is exposed to the same risk. Nova's bill of materials includes **2x OS-17 optical sensors** (compared to Orion's 1x), and the evidence

In [31]:
_cov_agentic = fact_coverage(agentic_answer, "Agentic RAG · fact coverage",
              subtitle="Same corpus, same embeddings, same LLM. The only change was putting retrieval in a loop.")

In [32]:
#@title 🧠 Lesson 2 — what the loop bought, and what it did not { display-mode: "form" }
from IPython.display import HTML, display
import numpy as np

architecture(
    "Part 2 · Agentic RAG — the same retriever, put in a loop",
    [("q",    "❓", "the question",  "the loop's starting state, and all it has"),
     ("llm",  "🧠", "reflect",       "<code>propose_next_search</code> &middot; what is still missing?"),
     ("vec",  "📚", "vector search", "top-k for the <b>new</b> query, not the original one"),
     ("text", "🗂️", "evidence pile", "merged and deduplicated by document id"),
     ("llm",  "⚖️", "enough?",       "<code>enough_evidence</code> &middot; the stopping rule"),
     ("ans",  "📝", "answer",        "one final call over everything gathered")],
    subtitle="Identical embeddings, identical corpus, identical model as Part 1. "
             "The architecture change is the arrow that goes <b>backwards</b>.",
    loop="not sufficient &rarr; reflect again, now able to spell the entity names the last hop revealed",
    new=(1, 3, 4),
    note="💰 Cost: <b>2–3 LLM calls per step</b> where Part 1 spent 1 in total. "
         "That is the price of the backwards arrow.")

# ── every number below is measured on YOUR run — nothing here is hardcoded ────
_qs   = [t["query"] for t in agentic_trace]
_V    = _embedder.encode([CEO_QUESTION] + _qs, normalize_embeddings=True)
_S    = EMB @ _V.T                                  # 25 docs × (question + one column per step)
_ids  = [d["id"] for d in DOCS]
_seen = {d["id"] for d in agentic_evidence}
_rk   = lambda col, i: int((_S[:, col] > _S[i, col]).sum()) + 1
_base = _S[:, 0]                                    # cosine against the CEO's question
_best = _S[:, 1:].max(axis=1) if _qs else _base.copy()   # best cosine over the queries ACTUALLY issued
_mv   = int(np.argmax(_best - _base))               # the document the loop rescued
_bar  = lambda v, c: (f'<span class="l2-bw"><span class="l2-bf" style="width:{max(2,min(100,v*100)):.0f}%;'
                      f'background:{c}"></span></span><span class="l2-num">{v:.2f}</span>')

# ── panel 1 · the query is a point, and the loop moved the point ─────────────
_moves = ""
for _c, _label in enumerate(["the CEO's question, embedded as-is"] + [f"step {i}" for i in range(1, len(_qs) + 1)]):
    _txt  = (CEO_QUESTION if _c == 0 else _qs[_c - 1])
    _top  = int(np.argmax(_S[:, _c]))
    _dq   = float(_V[0] @ _V[_c])                                  # cosine to the original question
    _dp   = float(_V[_c - 1] @ _V[_c]) if _c > 0 else 1.0          # cosine to the previous query
    _drift = ('<span class="l2-tag l2-same">same region as the question</span>' if _dq >= .60 else
              '<span class="l2-tag l2-far">a different region of the space</span>')
    _cos = "" if _c == 0 else (f'<span class="l2-lbl">cos → question</span>{_bar(_dq, "#667eea")}'
                               f'<span class="l2-lbl">cos → previous query</span>{_bar(_dp, "#9a63d4")}')
    _moves += (f'<div class="l2-mv"><div class="l2-mvh"><span class="l2-step">{_label}</span>'
               f'{"" if _c == 0 else _drift}</div>'
               f'<div class="l2-q">{_esc(_txt[:120])}</div>'
               f'<div class="l2-mg">{_cos}<span class="l2-lbl">its top hit</span>'
               f'<span class="l2-hit">{_ids[_top]} · {_S[_top, _c]:.2f}</span></div></div>')

_mvid, _mfrom, _mto = _ids[_mv], float(_base[_mv]), float(_best[_mv])
_mcol = int(_S[_mv, 1:].argmax()) + 1 if _qs else 0
_rescue = (f'<b>{_mvid}</b> — “{_esc(BY_ID[_mvid]["title"])}” — scored <b>{_mfrom:.2f}</b> against the CEO\'s '
           f'question (rank {_rk(0, _mv)} of 25: unreachable at any sane <code>top_k</code>). Against your '
           f'step-{_mcol} query it scores <b>{_mto:.2f}</b> — rank {_rk(_mcol, _mv)}. Same embedder, same index, '
           f'same document, <b>+{_mto - _mfrom:.2f} cosine</b>. Nothing about the retriever improved: the '
           f'<b>query moved</b>, and the words that moved it were produced by the previous retrieval.')

# ── panel 2 · the chain that was never in the running ────────────────────────
_CHAIN = [("D33", "Sanko also supplies <b>Meridian Optics</b>"),
          ("D21", "Meridian Optics makes <b>IR-9</b>"),
          ("D12", "<b>Helios</b> depends on IR-9")]
_chain_rows = ""
for _cid, _claim in _CHAIN:
    _i = _ids.index(_cid)
    _c2 = int(_S[_i, 1:].argmax()) + 1 if _qs else 0
    _hit = _cid in _seen
    _chain_rows += (f'<div class="l2-cr"><div class="l2-cid">{_cid}</div>'
                    f'<div class="l2-cc">{_claim}<div class="l2-cs">'
                    f'<span class="l2-lbl">vs the question</span>{_bar(float(_base[_i]), "#c9c4dd")}'
                    f'<span class="l2-lbl">vs your best query</span>{_bar(float(_best[_i]), "#e0a23c")}'
                    f'<span class="l2-lbl">best rank</span><span class="l2-hit">#{_rk(_c2, _i)} of 25</span>'
                    f'</div></div>'
                    f'<div class="l2-cv" style="color:{"#2f9e5c" if _hit else "#c0392b"}">'
                    f'{"retrieved" if _hit else "never retrieved"}</div></div>')

# the queries the agent never wrote — one embedding call each, no LLM, no new documents
_CF = ["who else does Sanko Photonics supply",
       "which programmes use the IR-9 module from Meridian Optics"]
_CV = _embedder.encode(_CF, normalize_embeddings=True)
_cf_rows = ""
for _q, _v in zip(_CF, _CV):
    _s = EMB @ _v
    _o = np.argsort(-_s)[:2]
    _cf_rows += (f'<div class="l2-cf"><div class="l2-q">🔍 {_esc(_q)}</div><div class="l2-cfr">'
                 + "".join(f'<span class="l2-hit l2-win">#{k+1} {_ids[i]} · {_s[i]:.2f}</span>'
                           for k, i in enumerate(_o)) + '</div></div>')

_missed = [c for c, _ in _CHAIN if c not in _seen]
_verdict = (f'Those documents were never <i>far away</i>. {"They were" if len(_missed) > 1 else "It was"} '
            f'<b>one unwritten query away</b> — and the query is trivial to write once you decide to ask it. '
            f'Distance in embedding space was never the blocker.'
            if _missed else
            'Your run did retrieve the chain — rarer, and worth noticing. Check the scorecard above: '
            'retrieving the documents and <i>concluding</i> the join are still two different things.')

# ── panel 3 · what a cosine can and cannot tell you ──────────────────────────
_nova = _ids.index("D11")
display(HTML('''
<style>
.l2{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.l2-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.l2-s{font-size:12px;color:#6b6685;margin:0 0 15px;line-height:1.55}
.l2-sec{font-size:11px;font-weight:800;text-transform:uppercase;letter-spacing:.5px;color:#b9a9e6;margin:16px 0 7px}
.l2-mv{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:9px 12px;margin:6px 0}
.l2-mvh{display:flex;align-items:center;gap:8px;flex-wrap:wrap}
.l2-step{font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;color:#764ba2}
.l2-tag{font-size:9.5px;font-weight:800;border-radius:6px;padding:2px 8px}
.l2-same{background:#f2f0fc;color:#764ba2}
.l2-far{background:#fdf3f1;color:#c0392b}
.l2-q{font-family:ui-monospace,Menlo,monospace;font-size:11px;color:#2c2350;margin-top:4px;line-height:1.45}
.l2-mg{display:flex;align-items:center;gap:6px;flex-wrap:wrap;margin-top:6px}
.l2-lbl{font-size:9.5px;color:#a9a3c4;font-weight:700;text-transform:uppercase;letter-spacing:.3px;margin-left:6px}
.l2-bw{width:64px;height:7px;background:#efedf8;border-radius:4px;overflow:hidden;display:inline-block}
.l2-bf{display:block;height:100%}
.l2-num{font-size:10.5px;color:#5b5578;font-variant-numeric:tabular-nums;font-family:ui-monospace,Menlo,monospace}
.l2-hit{font-family:ui-monospace,Menlo,monospace;font-size:10px;font-weight:700;background:#f2f0fc;color:#4b3f7a;border-radius:5px;padding:2px 7px}
.l2-win{background:#e7f6ee;color:#2f9e5c}
.l2-key{margin-top:9px;font-size:11.5px;color:#1e6b40;background:#eafaf0;border-left:4px solid #39b36a;border-radius:9px;padding:10px 12px;line-height:1.6}
.l2-cr{display:flex;gap:10px;align-items:center;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #e0796d;border-radius:11px;padding:9px 12px;margin:6px 0}
.l2-cid{font-family:ui-monospace,Menlo,monospace;font-size:11px;font-weight:800;color:#c0392b;flex:0 0 34px}
.l2-cc{flex:1;font-size:11.5px;line-height:1.45}
.l2-cs{display:flex;align-items:center;gap:5px;flex-wrap:wrap;margin-top:5px}
.l2-cv{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;flex:0 0 92px;text-align:right}
.l2-cf{background:#fff;border:1px solid #e7e4f6;border-left:4px solid #39b36a;border-radius:11px;padding:9px 12px;margin:6px 0}
.l2-cfr{display:flex;gap:6px;flex-wrap:wrap;margin-top:5px}
.l2-bad{margin-top:9px;font-size:11.5px;color:#7a3d34;background:#fdf3f1;border-left:4px solid #e0796d;border-radius:9px;padding:10px 12px;line-height:1.6}
.l2-grid{display:flex;gap:9px;flex-wrap:wrap}
.l2-b{flex:1 1 260px;background:#fff;border:1px solid #e7e4f6;border-top:3px solid #9a63d4;border-radius:12px;padding:11px 13px}
.l2-bt{font-size:12px;font-weight:800;color:#2c2350;margin-bottom:4px}
.l2-bd{font-size:10.8px;color:#5b5578;line-height:1.55}
.l2-foot{margin-top:12px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:4px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
.l2 code{background:#fff;border-radius:4px;padding:1px 5px;font-size:10.5px}
</style>
<div class="l2">
 <div class="l2-h">🧠 Lesson 2 — the loop moved the query. It could not count the answer.</div>
 <div class="l2-s">Both halves of this lesson are visible in the <b>cosine numbers from your own run</b>. A dense retriever turns a query into <b>one point</b> and returns the passages nearest to it. Everything the loop can and cannot do follows from that one sentence.</div>

 <div class="l2-sec">✅ what the loop bought — the query became a moving point</div>
 __MOVES__
 <div class="l2-key">🔑 <b>The mechanism, in one measurement.</b> __RESCUE__</div>

 <div class="l2-sec">❌ what it did not buy — the chain to Helios</div>
 __CHAIN__
 <div class="l2-sec" style="margin-top:10px">two queries your agent never wrote (embedded here, no LLM call)</div>
 __CF__
 <div class="l2-bad">🕳️ <b>It was not a distance problem.</b> __VERDICT__ What was missing is a <i>reason to ask</i>: nothing in the retrieved evidence flags a sub-tier vendor's <b>other customers</b> as the biggest open gap, the CEO never mentioned tier 2, and <code>propose_next_search</code> is told to close “the single most important gap” — a greedy policy with no notion of sweeping an entity's neighbourhood.</div>

 <div class="l2-sec">🧮 why the completeness half of the question survived all of this</div>
 <div class="l2-grid">
  <div class="l2-b"><div class="l2-bt">A cosine scores one pair</div><div class="l2-bd">
    <code>cos(q, d)</code> is a number about <b>a query and one passage</b>. There is no operation on that number, at any threshold, that reports <i>how many</i> documents ought to have come back. <code>top_k</code> returns k rows whether the true answer set has one member or seven.</div></div>
  <div class="l2-b"><div class="l2-bt">No passage asserts closure</div><div class="l2-bd">
    Documents state facts, not the <i>absence</i> of facts. Nothing in this corpus says “<i>and those are all the exposed programmes</i>”. So “which <b>other</b> programmes” is a claim about a <b>set</b>, and sets are not what a similarity ranking is about.</div></div>
  <div class="l2-b"><div class="l2-bt">The judge reads the pile</div><div class="l2-bd">
    <code>enough_evidence</code> sees only what was <i>retrieved</i>. A document that was never fetched has no representation in that prompt — no low score, no warning, no gap. <b>Coherence becomes indistinguishable from completeness</b>, and the loop stops.</div></div>
 </div>
 <div class="l2-foot">💡 <b>Concretely, in your run:</b> D11 (Nova's BOM) came back at <b>__NOVA__</b> and answered “which other programmes” with one name. Nothing scored <i>low</i> to tell you a second name existed — a missing document produces no signal at all. That is why the loop stopped where it did, and it is not a bug you can prompt your way out of: <b>you cannot verify a completeness claim from a ranked list of passages.</b> You verify it against a structure that is exhaustive by construction — which is Part 4.</div>
</div>'''
 .replace("__MOVES__", _moves).replace("__RESCUE__", _rescue)
 .replace("__CHAIN__", _chain_rows).replace("__CF__", _cf_rows)
 .replace("__VERDICT__", _verdict).replace("__NOVA__", f"{float(_best[_nova]):.2f}")))

explain("What iteration genuinely fixed — the causal chain",
        "Orion → OS-17 → Apex → Sanko is a sequence of lookups where each query is <i>writable only from "
        "the previous answer</i>, and a loop is exactly the right shape for that. In embedding terms: "
        "retrieval returns text, text supplies entity names, and those names <b>relocate the query vector</b> "
        "to a region the original question never pointed at. This is the highest-leverage upgrade to a RAG "
        "system and it costs nothing but latency and tokens.", tone="good", icon="✅")
explain("What it did not fix — and the honest version of why",
        "Not because the evidence is unreachable: the Helios exposure <i>is</i> derivable from text, as the "
        "chain <b>D33 → D21 → D12</b>, and the numbers above show those documents sitting one well-aimed query "
        "away. It failed because that query is <b>unmotivated</b> — no retrieved passage makes a sub-tier "
        "supplier's other customers look like the most important gap — and because the stopping rule cannot "
        "distinguish “I have <i>an</i> answer” from “I have <i>the whole</i> answer”. Iteration deepens a "
        "search; it does not make it exhaustive.", tone="risk", icon="🕳️")
explain("The question to carry into Part 3",
        "If a fact only exists as a <b>path across several documents</b>, and finding it depends on the agent "
        "volunteering a line of enquiry nobody asked for — <b>how would you ever know it was missing?</b> "
        "Not by searching harder. By representing the relationships explicitly, so that “everything downstream "
        "of X” becomes a traversal with a <i>guaranteed</i> answer set rather than a ranked guess.",
        tone="warn", icon="🤔")

---
# Part 3 — The question that iteration cannot answer

Ask the harder half of the CEO's question on its own, and give the agentic system every advantage:
a direct, well-phrased query using the exact entity names it just learned.

In [33]:
EXPOSURE_QUESTION = "Which other programmes are exposed to the same supplier risk as Orion?"

for q in ["other programmes exposed to Apex Components risk",
          "projects affected by the Sanko Photonics wafer shortage",
          "programmes at risk from the optical supply chain disruption"]:
    print(f"\n🔍 {q}")
    for h in vector_search(q, top_k=3):
        print(f"   {h['score']:.2f}  {h['doc']['id']}  {h['doc']['title']}")


🔍 other programmes exposed to Apex Components risk
   0.64  D40  Annual supplier risk review
   0.44  D31  Apex Components — supplier disclosure letter
   0.43  D30  Supplier incident report — Apex Components

🔍 projects affected by the Sanko Photonics wafer shortage
   0.74  D33  Sanko Photonics — regional customer register
   0.68  D32  Regional incident bulletin — Sanko Photonics
   0.57  D31  Apex Components — supplier disclosure letter

🔍 programmes at risk from the optical supply chain disruption
   0.52  D41  Sourcing consolidation memo
   0.44  D31  Apex Components — supplier disclosure letter
   0.42  D40  Annual supplier risk review


Three well-aimed queries. Every one of them returns Orion, Nova, Apex and Sanko documents — and
**none of them returns anything about Helios**, because from the retriever's point of view Helios has
nothing to do with this story.

Let's prove that claim rather than assert it.

In [34]:
# Is there ANY document that mentions Helios together with the disruption?
for term in ["Apex", "Sanko", "OS-17", "wafer"]:
    co = [d["id"] for d in DOCS
          if "helios" in _doc_text(d).lower() and term.lower() in _doc_text(d).lower()]
    print(f"documents containing 'Helios' AND '{term}':  {co or '—  none'}")

print("\nDocuments mentioning Helios at all:",
      [d["id"] for d in DOCS if "helios" in _doc_text(d).lower()])

documents containing 'Helios' AND 'Apex':  —  none
documents containing 'Helios' AND 'Sanko':  —  none
documents containing 'Helios' AND 'OS-17':  —  none
documents containing 'Helios' AND 'wafer':  —  none

Documents mentioning Helios at all: ['D03', 'D12', 'D51']


In [35]:
#@title 🕳️ The fact that is not in any document { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r'''
<style>
.gp{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:900px;color:#2c2350}
.gp-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.gp-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.55}
.gp-chain{display:flex;align-items:center;gap:5px;flex-wrap:wrap;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:12px;margin-bottom:10px}
.gp-n{font-size:11px;font-weight:800;border-radius:8px;padding:6px 10px}
.gp-doc{font-family:ui-monospace,Menlo,monospace;font-size:9px;color:#a9a3c4;display:block;font-weight:400;margin-top:2px}
.gp-ar{color:#b9a9e6;font-size:14px;font-weight:800}
.gp-proj{background:#e7f0ff;color:#2f5fa8}
.gp-comp{background:#f2f0fc;color:#5b4a86}
.gp-sup{background:#fdf3f1;color:#a8483c}
.gp-t2{background:#e0796d;color:#fff}
.gp-cut{display:flex;align-items:center;gap:10px;margin:12px 0;font-size:11.5px;color:#7a3d34}
.gp-scissors{font-size:16px}
.gp-line{flex:1;border-top:2px dashed #e0796d}
.gp-foot{margin-top:8px;font-size:11.5px;color:#7a3d34;background:#fdf3f1;border-left:3px solid #e0796d;border-radius:9px;padding:10px 12px;line-height:1.6}
.gp-key{margin-top:10px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="gp">
 <div class="gp-h">🕳️ Helios is exposed. No document says so.</div>
 <div class="gp-s">The connection between Orion's delay and Helios's risk is <b>six hops long</b>, and every hop is stated in a <i>different</i> document. Each document is locally true and globally silent.</div>
 <div class="gp-chain">
  <span class="gp-n gp-proj">Project Orion<span class="gp-doc">D10</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-comp">OS-17<span class="gp-doc">D20</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-sup">Apex Components<span class="gp-doc">D31</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-t2">Sanko Photonics<span class="gp-doc">D33</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-sup">Meridian Optics<span class="gp-doc">D21</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-comp">IR-9<span class="gp-doc">D12</span></span><span class="gp-ar">&rarr;</span>
  <span class="gp-n gp-proj">Project Helios</span>
 </div>
 <div class="gp-cut"><span class="gp-scissors">✂️</span><span>a retriever can only cut this chain into passage-sized pieces</span><span class="gp-line"></span></div>
 <div class="gp-foot">❌ <b>Why no query works.</b> A dense retriever scores <i>text against text</i>. "Helios is exposed to Apex" is not text — it is a <b>path</b>. There is no passage to be similar to, so similarity search has nothing to rank. More steps do not help either: the agentic loop would have to guess the word "Meridian" out of thin air, and even then nothing connects Meridian to the incident in prose.</div>
 <div class="gp-key">🔑 <b>The GraphRAG thesis, in one line:</b> when your information need is <i>relational</i> — shared dependencies, blast radius, "who else", "what connects X and Y" — represent the relations <b>explicitly</b> and traverse them. Do not ask an embedding to approximate a join.</div>
</div>
'''))

---
# Part 4 — Explore the knowledge graph

We are handing you the graph. In a real system this is built by an **entity-and-relation extraction
pass** over the corpus — an LLM reads each document and emits triples — and that build step is a
notebook of its own. What matters here is what the graph *does for retrieval* once you have it.

**One non-negotiable design rule, and it is the one people get wrong:** every edge keeps the **id of
the document it was extracted from**. A graph without provenance is a pile of assertions your system
cannot cite, cannot date, and cannot audit. Part 5 is entirely about cashing that in.

In [36]:
#@title 🔧 PROVIDED — the knowledge graph, with provenance on every edge (run, don't edit) { display-mode: "form" }
import networkx as nx

# (subject, relation, object, source_document_id) — extracted from the corpus
EDGES = [
    # programmes → components  (from the engineering BOMs)
    ("Project Orion",  "depends_on", "OS-17",  "D10"),
    ("Project Orion",  "depends_on", "PCB-K3", "D10"),
    ("Project Nova",   "depends_on", "OS-17",  "D11"),
    ("Project Nova",   "depends_on", "PCB-K3", "D11"),
    ("Project Helios", "depends_on", "IR-9",   "D12"),
    ("Project Helios", "depends_on", "GIM-3",  "D12"),
    ("Project Luna",   "depends_on", "BAT-4",  "D13"),
    ("Project Luna",   "depends_on", "PCB-K3", "D13"),
    ("Project Vega",   "depends_on", "OS-22",  "D14"),
    ("Project Vega",   "depends_on", "BAT-4",  "D14"),
    ("Project Vega",   "depends_on", "CS-9",   "D14"),
    ("Project Atlas",  "depends_on", "CS-9",   "D15"),

    # tier-1 suppliers → components  (from the procurement registers)
    ("Apex Components",      "supplies", "OS-17",  "D20"),
    ("Apex Components",      "supplies", "OS-22",  "D20"),
    ("Meridian Optics",      "supplies", "IR-9",   "D21"),
    ("Voltix Energy",        "supplies", "BAT-4",  "D22"),
    ("CloudSync Ltd",        "supplies", "CS-9",   "D23"),
    ("Kestrel Fabrication",  "supplies", "PCB-K3", "D24"),
    ("Aerodyne Mechanics",   "supplies", "GIM-3",  "D25"),

    # tier-2 (sub-tier) supplier → tier-1 suppliers  ← the edge nobody thinks to look for
    ("Sanko Photonics", "supplies_wafers_to", "Apex Components", "D33"),
    ("Sanko Photonics", "supplies_wafers_to", "Meridian Optics", "D33"),

    # events
    ("Apex Components",     "disrupted_by", "Apex production halt",  "D30"),
    ("Apex production halt", "caused_by",   "Sanko cleanroom fire",  "D31"),
    ("Sanko Photonics",     "disrupted_by", "Sanko cleanroom fire",  "D32"),
]

NODE_TYPE = {}
for _n in ["Project Orion", "Project Nova", "Project Helios", "Project Luna", "Project Vega", "Project Atlas"]:
    NODE_TYPE[_n] = "programme"
for _n in ["OS-17", "OS-22", "IR-9", "BAT-4", "CS-9", "PCB-K3", "GIM-3"]:
    NODE_TYPE[_n] = "component"
for _n in ["Apex Components", "Meridian Optics", "Voltix Energy", "CloudSync Ltd",
           "Kestrel Fabrication", "Aerodyne Mechanics"]:
    NODE_TYPE[_n] = "tier-1 supplier"
NODE_TYPE["Sanko Photonics"] = "tier-2 supplier"
for _n in ["Apex production halt", "Sanko cleanroom fire"]:
    NODE_TYPE[_n] = "event"

G = nx.Graph()                                   # undirected: we traverse relations both ways
for _s, _r, _o, _src in EDGES:
    G.add_node(_s, kind=NODE_TYPE.get(_s, "other"))
    G.add_node(_o, kind=NODE_TYPE.get(_o, "other"))
    G.add_edge(_s, _o, relation=_r, subject=_s, object=_o, source=_src)

print(f"✅ graph: {G.number_of_nodes()} nodes · {G.number_of_edges()} edges · "
      f"every edge carries a source document id")
print("   e.g.", G.edges['Sanko Photonics', 'Apex Components'])

✅ graph: 22 nodes · 24 edges · every edge carries a source document id
   e.g. {'relation': 'supplies_wafers_to', 'subject': 'Sanko Photonics', 'object': 'Apex Components', 'source': 'D33'}


In [37]:
#@title 🕸️ The knowledge graph — click a node · trace the blast radius { display-mode: "form" }
from IPython.display import HTML, display
import json as _json

_POS = {
    "Sanko cleanroom fire":  (78,  92),  "Sanko Photonics":     (78,  232),
    "Apex production halt":  (272, 46),  "Apex Components":     (272, 150),
    "Meridian Optics":       (272, 232), "Voltix Energy":       (272, 306),
    "CloudSync Ltd":         (272, 364), "Kestrel Fabrication": (272, 420),
    "Aerodyne Mechanics":    (272, 476),
    "OS-17":  (498, 112), "OS-22":  (498, 176), "IR-9":   (498, 240),
    "BAT-4":  (498, 306), "CS-9":   (498, 364), "PCB-K3": (498, 420), "GIM-3": (498, 476),
    "Project Orion":  (726, 92),  "Project Nova":  (726, 150), "Project Vega":   (726, 212),
    "Project Helios": (726, 274), "Project Luna":  (726, 350), "Project Atlas":  (726, 412),
}
_KC = {"programme": "#4c8dd8", "component": "#9a63d4", "tier-1 supplier": "#e0796d",
       "tier-2 supplier": "#c0392b", "event": "#e0a23c"}
_nodes = [dict(id=n, x=_POS[n][0], y=_POS[n][1], kind=NODE_TYPE.get(n, "other"),
               color=_KC.get(NODE_TYPE.get(n, "other"), "#999")) for n in _POS]
_edges = [dict(s=s, o=o, r=r, src=src) for (s, r, o, src) in EDGES]

display(HTML('''
<style>
.kg{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:960px;color:#2c2350}
.kg-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.kg-s{font-size:12px;color:#6b6685;margin:0 0 12px;line-height:1.55}
.kg-ctrl{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:10px;align-items:center}
.kg-btn{cursor:pointer;border:none;border-radius:10px;padding:8px 13px;font-size:12px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2)}
.kg-btn.fire{background:linear-gradient(135deg,#e0796d,#c0392b)}
.kg-btn.alt{background:#fff;color:#4b3f7a;border:1px solid #d9d5ee}
.kg-legend{display:flex;gap:10px;flex-wrap:wrap;font-size:10.5px;color:#5b5578;margin-bottom:8px}
.kg-lg{display:flex;align-items:center;gap:5px}
.kg-dot{width:9px;height:9px;border-radius:50%;display:inline-block}
.kg-svgwrap{background:#fff;border:1px solid #e7e4f6;border-radius:14px;padding:6px;overflow-x:auto}
.kg-panel{margin-top:10px;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #667eea;border-radius:11px;padding:11px 13px;font-size:11.5px;color:#3a3357;line-height:1.6;min-height:56px}
.kg-panel b{color:#3b2d6b}
.kg-trip{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;background:#f7f6fc;border-radius:5px;padding:2px 6px;margin:2px 3px 0 0;display:inline-block;color:#4b3f7a}
.kg-src{color:#a9a3c4;font-size:9.5px}
text{font-family:system-ui,Segoe UI,Roboto,sans-serif;pointer-events:none}
.kgnode{cursor:pointer}
</style>
<div class="kg">
 <div class="kg-h">🕸️ The same 25 documents, as 22 nodes and 24 edges</div>
 <div class="kg-s">Nothing new was added — this is the <i>same information</i>, re-represented so that relationships are first-class objects you can walk instead of sentences you have to match. <b>Click any node</b> to see its neighbours, or trace the blast radius.</div>
 <div class="kg-ctrl">
   <button class="kg-btn fire" onclick="kgBlast()">💥 Trace the blast radius from the fire</button>
   <button class="kg-btn" onclick="kgPath()">🔦 Orion → Helios: the invisible link</button>
   <button class="kg-btn alt" onclick="kgReset()">reset</button>
 </div>
 <div class="kg-legend">
   <span class="kg-lg"><i class="kg-dot" style="background:#e0a23c"></i>event</span>
   <span class="kg-lg"><i class="kg-dot" style="background:#c0392b"></i>tier-2 supplier</span>
   <span class="kg-lg"><i class="kg-dot" style="background:#e0796d"></i>tier-1 supplier</span>
   <span class="kg-lg"><i class="kg-dot" style="background:#9a63d4"></i>component</span>
   <span class="kg-lg"><i class="kg-dot" style="background:#4c8dd8"></i>programme</span>
 </div>
 <div class="kg-svgwrap"><svg id="kgSvg" viewBox="0 0 810 512" style="width:100%;min-width:640px;height:auto"></svg></div>
 <div class="kg-panel" id="kgPanel">👆 Click <b>Sanko Photonics</b> (far left) — the node that never appears in an Orion document, and never appears in a Helios document, but sits underneath both.</div>
</div>
<script>
(function(){
 const N = __NODES__, E = __EDGES__;
 const svg = document.getElementById('kgSvg'), panel = document.getElementById('kgPanel');
 const NS='http://www.w3.org/2000/svg';
 const pos={}; N.forEach(n=>pos[n.id]=n);
 const adj={}; N.forEach(n=>adj[n.id]=[]);
 E.forEach((e,i)=>{adj[e.s].push({to:e.o,i:i,e:e});adj[e.o].push({to:e.s,i:i,e:e});});
 let timers=[];
 function clearTimers(){timers.forEach(t=>clearTimeout(t));timers=[];}

 // ---- render ----
 const gE=document.createElementNS(NS,'g'), gN=document.createElementNS(NS,'g');
 svg.appendChild(gE); svg.appendChild(gN);
 const lines=E.map(e=>{
   const l=document.createElementNS(NS,'line');
   l.setAttribute('x1',pos[e.s].x);l.setAttribute('y1',pos[e.s].y);
   l.setAttribute('x2',pos[e.o].x);l.setAttribute('y2',pos[e.o].y);
   l.setAttribute('stroke','#c9c4dd');l.setAttribute('stroke-width','1.4');l.setAttribute('opacity','.55');
   gE.appendChild(l);return l;});
 const shapes={};
 N.forEach(n=>{
   const g=document.createElementNS(NS,'g'); g.setAttribute('class','kgnode');
   g.addEventListener('click',()=>select(n.id));
   const w=Math.max(70,n.id.length*6.4+16);
   const r=document.createElementNS(NS,'rect');
   r.setAttribute('x',n.x-w/2);r.setAttribute('y',n.y-13);r.setAttribute('width',w);r.setAttribute('height',26);
   r.setAttribute('rx',13);r.setAttribute('fill',n.color);r.setAttribute('opacity','.92');
   const t=document.createElementNS(NS,'text');
   t.setAttribute('x',n.x);t.setAttribute('y',n.y+4);t.setAttribute('text-anchor','middle');
   t.setAttribute('font-size','10.5');t.setAttribute('font-weight','700');t.setAttribute('fill','#fff');
   t.textContent=n.id.replace('Project ','');
   g.appendChild(r);g.appendChild(t);gN.appendChild(g);shapes[n.id]={g:g,r:r,t:t,w:w};});

 function reset(){clearTimers();
   lines.forEach(l=>{l.setAttribute('stroke','#c9c4dd');l.setAttribute('stroke-width','1.4');l.setAttribute('opacity','.55');});
   N.forEach(n=>{shapes[n.id].r.setAttribute('fill',n.color);shapes[n.id].g.setAttribute('opacity','1');});}
 function dimAll(){N.forEach(n=>shapes[n.id].g.setAttribute('opacity','.18'));
   lines.forEach(l=>l.setAttribute('opacity','.12'));}
 function lightEdge(i,color,w){lines[i].setAttribute('stroke',color);lines[i].setAttribute('stroke-width',w||3);lines[i].setAttribute('opacity','1');}
 function lightNode(id,color){shapes[id].g.setAttribute('opacity','1');if(color)shapes[id].r.setAttribute('fill',color);}

 window.kgReset=function(){reset();panel.innerHTML='👆 Click <b>Sanko Photonics</b> (far left) — the node that never appears in an Orion document, and never appears in a Helios document, but sits underneath both.';};

 function select(id){
   reset();dimAll();lightNode(id);
   let rows='';
   adj[id].forEach(a=>{lightEdge(a.i,'#667eea',2.6);lightNode(a.to);
     const dir = a.e.s===id ? (id+' —'+a.e.r+'→ '+a.to) : (a.to+' —'+a.e.r+'→ '+id);
     rows+='<span class="kg-trip">'+dir+' <span class="kg-src">'+a.e.src+'</span></span>';});
   panel.innerHTML='<b>'+id+'</b> · '+pos[id].kind+' · '+adj[id].length+' relation(s). Each triple carries the document it came from:<br>'+rows;}

 window.kgBlast=function(){
   reset();dimAll();
   const start='Sanko cleanroom fire';
   let frontier=[start], seen={}; seen[start]=0; const byDepth=[[start]];
   for(let d=1;d<=4;d++){const nxt=[];   // stop at 4: that is where the programmes light up
     frontier.forEach(u=>adj[u].forEach(a=>{if(!(a.to in seen)){seen[a.to]=d;nxt.push(a.to);}}));
     if(!nxt.length)break; byDepth.push(nxt); frontier=nxt;}
   const cols=['#c0392b','#e0796d','#e0a23c','#9a63d4','#7d5fd0','#4c8dd8'];
   byDepth.forEach((layer,d)=>{
     timers.push(setTimeout(()=>{
       layer.forEach(u=>{lightNode(u,cols[Math.min(d,5)]);
         adj[u].forEach(a=>{if(seen[a.to]!==undefined&&seen[a.to]<d)lightEdge(a.i,cols[Math.min(d,5)],3);});});
       const projs=Object.keys(seen).filter(k=>pos[k].kind==='programme'&&seen[k]<=d);
       panel.innerHTML='💥 <b>Blast radius, hop '+d+'</b> — reached '+layer.length+' new entities.<br>'+
         'Programmes touched so far: <b>'+(projs.map(p=>p.replace('Project ','')).join(', ')||'none yet')+'</b>'+
         (d>=4?'<br><span style="color:#c0392b">⬆ All four optical programmes arrive at once. Orion, Nova and Vega came through <b>Apex</b> — the supplier everyone is already talking about. <b>Helios</b> came through <b>Meridian Optics</b>, which appears in <i>no</i> Orion document and <i>no</i> incident report. Same distance from the fire, completely different chance of anyone noticing.</span>':'');
     },d*900));});};

 window.kgPath=function(){
   reset();dimAll();
   const path=['Project Orion','OS-17','Apex Components','Sanko Photonics','Meridian Optics','IR-9','Project Helios'];
   const docs=['D10','D20','D31/D33','D33','D21','D12'];
   path.forEach((u,k)=>timers.push(setTimeout(()=>{
     lightNode(u,k===0||k===path.length-1?'#2f9e5c':'#764ba2');
     if(k>0){const a=adj[path[k-1]].find(z=>z.to===u); if(a)lightEdge(a.i,'#2f9e5c',3.4);}
     panel.innerHTML='🔦 <b>Orion → Helios</b>, hop '+k+' of 6:<br>'+
       path.slice(0,k+1).map((p,j)=>'<span class="kg-trip">'+p+(j<k?' <span class="kg-src">'+docs[j]+'</span>':'')+'</span>').join('<span style="color:#b9a9e6">→</span> ')+
       (k===path.length-1?'<br><br><b style="color:#2f9e5c">Six hops, six different source documents, zero passages containing both endpoints.</b> A traversal found in microseconds what no query could express.':'');
   },k*750)));};
})();
</script>
'''.replace("__NODES__", _json.dumps(_nodes)).replace("__EDGES__", _json.dumps(_edges))))

### 🎯 4.1 — The primitive: `get_neighbors`

Graph retrieval bottoms out in one operation: *given an entity, what is it connected to, how, and
where did we learn that?* Everything else — paths, blast radius, subgraph extraction — is this
function in a loop.

Return one dict **per incident edge**, with:

- `neighbor` — the node on the other end
- `relation` — `G.edges[entity, nb]["relation"]`
- `direction` — `"out"` if `entity` is the edge's `subject`, else `"in"` (relations are directional
  even though we traverse both ways: *Apex supplies OS-17* is not *OS-17 supplies Apex*)
- `source` — the document id, carried through **every** hop
- `triple` — the relation written in its natural reading order, for prompts and for humans

Use `G.neighbors(entity)` and `G.edges[entity, nb]`.

In [39]:
def get_neighbors(entity):
    """All relations incident to `entity`, with direction and provenance."""
    if entity not in G:
        return []
    SEARCH_CALLS["graph"] += 1
    out = []
    for nb in G.neighbors(entity):
        e = G.edges[entity, nb]     # keys: relation · subject · object · source
        # ✉️ TODO 1: is `entity` the subject of this edge, or the object?
        outgoing = (e["subject"] == entity)   # ✉️ compare e["subject"] with entity
        # ✉️ TODO 2: write the triple in its natural reading order, e.g.
        #   "Apex Components --supplies--> OS-17"   (never "OS-17 --supplies--> Apex")
        triple = f"{entity} --{e['relation']}--> {nb}" if outgoing else f"{nb} --{e['relation']}--> {entity}"     # ✉️ f-string over entity, e["relation"], nb — order depends on `outgoing`
        # ✉️ TODO 3: assemble the record — and never drop the source id.
        #   direction is "out" when `entity` is the subject, "in" otherwise.
        # ✉️ relation and source both come off `e`; direction is "out" when outgoing else "in"
        out.append(dict(neighbor=nb, relation=e["relation"], direction="out" if outgoing else "in", source=e["source"], triple=triple))
    return out

for _e in ["Apex Components", "OS-17", "Sanko Photonics"]:
    print(f"\n📍 {_e}")
    for _n in get_neighbors(_e):
        print(f"   {_n['triple']:<62} [{_n['source']}]")


📍 Apex Components
   Apex Components --supplies--> OS-17                            [D20]
   Apex Components --supplies--> OS-22                            [D20]
   Sanko Photonics --supplies_wafers_to--> Apex Components        [D33]
   Apex Components --disrupted_by--> Apex production halt         [D30]

📍 OS-17
   Project Orion --depends_on--> OS-17                            [D10]
   Project Nova --depends_on--> OS-17                             [D11]
   Apex Components --supplies--> OS-17                            [D20]

📍 Sanko Photonics
   Sanko Photonics --supplies_wafers_to--> Apex Components        [D33]
   Sanko Photonics --supplies_wafers_to--> Meridian Optics        [D33]
   Sanko Photonics --disrupted_by--> Sanko cleanroom fire         [D32]


Read those three blocks again — that is the whole "aha".

`get_neighbors("OS-17")` answers **"which programmes use this part?"** exactly, instantly and
exhaustively. No ranking, no threshold, no top-k, no chance of a near-miss. And
`get_neighbors("Sanko Photonics")` answers a question that has **no textual formulation at all**:
*who sits downstream of this company?*

### 🎯 4.2 — Traversal: `graph_search`

Now generalise to *n* hops. Breadth-first from a starting entity, recording for every reachable node
**how far away it is** and **the exact chain of edges that got you there**. That chain is what you
will cite in Part 5 — a path you cannot reconstruct is a path you cannot defend to a COO.

Fill the three lines inside the loop:

- skip nodes already in `seen` (BFS visits each node once, at its *shortest* depth)
- record `depth=d` and `path = seen[node]["path"] + [that edge]`
- push the neighbour onto the next frontier

In [40]:
#@title 🎬 What a breadth-first traversal actually does — step through it { display-mode: "form" }
from IPython.display import HTML, display
import json as _json

# Same layout and palette as the graph explorer above, so this is recognisably the same graph.
_BFS_STARTS = ["Sanko cleanroom fire", "Apex Components", "OS-17", "Project Orion"]
_bf_nodes = [dict(id=n, x=_POS[n][0], y=_POS[n][1], kind=NODE_TYPE.get(n, "other")) for n in _POS]
_bf_edges = [dict(s=s, o=o, r=r, src=src) for (s, r, o, src) in EDGES]
_bf_btns = "".join(f'<button class="bf-st" onclick="bfStart(\'{s}\')">{s}</button>' for s in _BFS_STARTS)

display(HTML('''
<style>
.bf{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:960px;color:#2c2350}
.bf-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.bf-s{font-size:12px;color:#6b6685;margin:0 0 12px;line-height:1.55}
.bf-idea{background:#fff;border:1px solid #e7e4f6;border-left:4px solid #667eea;border-radius:12px;padding:11px 13px;font-size:11.5px;line-height:1.6;margin-bottom:12px}
.bf-lbl{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;color:#b9a9e6;margin:0 0 5px}
.bf-ctrl{display:flex;gap:7px;flex-wrap:wrap;align-items:center;margin-bottom:9px}
.bf-btn{cursor:pointer;border:none;border-radius:10px;padding:8px 14px;font-size:12.5px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2)}
.bf-btn.go{background:linear-gradient(135deg,#39b36a,#2f9e5c)}
.bf-btn.alt{background:#fff;color:#4b3f7a;border:1px solid #d9d5ee}
.bf-btn:disabled{opacity:.4;cursor:default}
.bf-st{cursor:pointer;border:1px solid #d9d5ee;background:#fff;color:#4b3f7a;border-radius:8px;padding:5px 9px;font-size:10.5px;font-weight:700}
.bf-st.on{background:#667eea;color:#fff;border-color:#667eea}
.bf-legend{display:flex;gap:9px;flex-wrap:wrap;font-size:10px;color:#5b5578;margin:2px 0 8px;align-items:center}
.bf-lg{display:flex;align-items:center;gap:4px}
.bf-dot{width:9px;height:9px;border-radius:50%;display:inline-block}
.bf-dash{width:16px;height:0;border-top:2px dashed #e0a23c;display:inline-block}
.bf-svgwrap{background:#fff;border:1px solid #e7e4f6;border-radius:14px;padding:6px;overflow-x:auto}
.bf-narr{margin-top:10px;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #9a63d4;border-radius:11px;padding:10px 13px;font-size:11.5px;line-height:1.6;min-height:42px}
.bf-code{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;background:#f7f6fc;color:#4b3f7a;border-radius:5px;padding:1px 5px}
.bf-grid{display:flex;gap:9px;flex-wrap:wrap;margin-top:10px}
.bf-p{flex:1 1 220px;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 12px;min-height:96px}
.bf-pt{font-size:11px;font-weight:800;color:#3b2d6b;margin-bottom:2px}
.bf-pd{font-size:9.5px;color:#a9a3c4;line-height:1.4;margin-bottom:7px}
.bf-chip{display:inline-block;font-family:ui-monospace,Menlo,monospace;font-size:9.5px;font-weight:700;border-radius:5px;padding:2px 6px;margin:2px 3px 0 0;background:#f2f0fc;color:#4b3f7a}
.bf-chip.new{background:#e7f6ee;color:#2f9e5c}
.bf-chip.skip{background:#fdf6e7;color:#a8752a}
.bf-row{font-size:10px;color:#5b5578;margin-top:5px;line-height:1.5}
.bf-dep{display:inline-block;font-family:ui-monospace,Menlo,monospace;font-size:8.5px;font-weight:800;color:#fff;border-radius:4px;padding:1px 5px;margin-right:5px}
.bf-tri{font-family:ui-monospace,Menlo,monospace;font-size:9.5px;background:#f7f6fc;color:#4b3f7a;border-radius:5px;padding:3px 6px;margin-top:3px;display:block;line-height:1.45}
.bf-src{color:#a9a3c4;font-size:8.5px}
.bf-none{font-size:10px;color:#c9c4dd;font-style:italic}
.bf-two{display:flex;gap:10px;flex-wrap:wrap;margin-top:12px}
.bf-b{flex:1 1 260px;border-radius:12px;padding:11px 13px;font-size:11.5px;line-height:1.6}
.bf-why{background:#f2f0fc;border-left:4px solid #667eea;color:#4b3f7a}
.bf-win{background:#eafaf0;border-left:4px solid #39b36a;color:#1e6b40}
.bf text{font-family:system-ui,Segoe UI,Roboto,sans-serif;pointer-events:none}
</style>
<div class="bf">
 <div class="bf-h">🎬 Breadth-first search, one hop at a time</div>
 <div class="bf-s">This is the algorithm you are about to write in <span class="bf-code">graph_search</span>. Press <b>next hop</b> and watch the three variables in the panels below — they are exactly the three lines you have to fill in.</div>

 <div class="bf-idea">💧 <b>The idea in one sentence:</b> drop a stone in a pond. <b>Hop 1</b> is everything one relation away from the start. <b>Hop 2</b> is everything one relation away from <i>those</i>, minus anything already wet. And so on until the ripples stop.<br>
 That "minus anything already wet" is the whole trick: because a node is recorded <b>the first time</b> a ripple reaches it, the depth you store is automatically the <b>shortest</b> chain of relations from the start — you never have to search for it.</div>

 <div class="bf-lbl">start the traversal from</div>
 <div class="bf-ctrl">__BTNS__</div>
 <div class="bf-ctrl">
  <button class="bf-btn" id="bfNext" onclick="bfStep()">⏭ next hop</button>
  <button class="bf-btn go" onclick="bfAuto()">▶ run to the end</button>
  <button class="bf-btn alt" onclick="bfReset()">↺ reset</button>
 </div>
 <div class="bf-legend">
  <span class="bf-lg"><i class="bf-dot" style="background:#c0392b"></i>start (hop 0)</span>
  <span class="bf-lg"><i class="bf-dot" style="background:#e0796d"></i>hop 1</span>
  <span class="bf-lg"><i class="bf-dot" style="background:#e0a23c"></i>hop 2</span>
  <span class="bf-lg"><i class="bf-dot" style="background:#9a63d4"></i>hop 3</span>
  <span class="bf-lg"><i class="bf-dot" style="background:#4c8dd8"></i>hop 4</span>
  <span class="bf-lg"><i class="bf-dot" style="background:#b9b4cc"></i>not reached yet</span>
  <span class="bf-lg"><i class="bf-dash"></i>edge skipped — both ends already seen</span>
 </div>
 <div class="bf-svgwrap"><svg id="bfSvg" viewBox="0 0 810 512" style="width:100%;min-width:640px;height:auto"></svg></div>
 <div class="bf-narr" id="bfNarr">👆 Pick a starting entity, then press <b>next hop</b>. Nothing has been visited yet: <span class="bf-code">seen</span> holds only the start, at depth 0.</div>

 <div class="bf-grid">
  <div class="bf-p">
   <div class="bf-pt">🌊 frontier</div>
   <div class="bf-pd">the layer being expanded right now — the loop's <span class="bf-code">frontier</span> list</div>
   <div id="bfFrontier"></div></div>
  <div class="bf-p">
   <div class="bf-pt">📓 seen &nbsp;<span style="font-weight:400;color:#a9a3c4">node → depth</span></div>
   <div class="bf-pd">every node found so far, at its <b>shortest</b> depth — <span class="bf-code">TODO 1</span> checks this dict to avoid revisiting</div>
   <div id="bfSeen"></div></div>
  <div class="bf-p">
   <div class="bf-pt">🔗 path, for the nodes just found</div>
   <div class="bf-pd">the edge chain stored by <span class="bf-code">TODO 2</span> — this is what Part 5 cites</div>
   <div id="bfPath"></div></div>
 </div>

 <div class="bf-two">
  <div class="bf-b bf-why"><b>❓ Why breadth-first and not depth-first?</b><br>
   Depth-first would dive down one branch to the end before trying the next, so the first path it finds to a node is whatever it stumbled into — often a long, silly detour. BFS reaches every node by the <b>fewest relations possible</b>, and in this corpus <i>hop count is a claim</i>: "Helios is 4 relations from the fire" is a statement you can defend. A 9-hop wander through PCB-K3 is not.</div>
  <div class="bf-b bf-win"><b>✅ Why this beats a search box</b><br>
   When the frontier empties, the traversal is <b>done</b> — and "done" means <i>every</i> reachable node was found, not the best 5. There is no <span class="bf-code">top_k</span> to tune and no relevance score to threshold, so "which other programmes are exposed?" gets an answer set that is <b>complete by construction</b>. That is the one thing Part 2's loop could never give you.</div>
 </div>
</div>
<script>
(function(){
 const N = __NODES__, E = __EDGES__;
 const svg = document.getElementById('bfSvg');
 const NS = 'http://www.w3.org/2000/svg';
 const DC = ['#c0392b','#e0796d','#e0a23c','#9a63d4','#4c8dd8','#39b36a','#7d5fd0'];
 const pos = {}; N.forEach(n => pos[n.id] = n);
 const adj = {}; N.forEach(n => adj[n.id] = []);
 E.forEach((e,i) => {adj[e.s].push({to:e.o, i:i, e:e}); adj[e.o].push({to:e.s, i:i, e:e});});

 // ---- draw the graph once -------------------------------------------------
 const gE = document.createElementNS(NS,'g'), gN = document.createElementNS(NS,'g');
 svg.appendChild(gE); svg.appendChild(gN);
 const lines = E.map(e => {
   const l = document.createElementNS(NS,'line');
   l.setAttribute('x1',pos[e.s].x); l.setAttribute('y1',pos[e.s].y);
   l.setAttribute('x2',pos[e.o].x); l.setAttribute('y2',pos[e.o].y);
   gE.appendChild(l); return l;});
 const shapes = {};
 N.forEach(n => {
   const g = document.createElementNS(NS,'g');
   const w = Math.max(70, n.id.length*6.4 + 16);
   const r = document.createElementNS(NS,'rect');
   r.setAttribute('x',n.x-w/2); r.setAttribute('y',n.y-13);
   r.setAttribute('width',w); r.setAttribute('height',26); r.setAttribute('rx',13);
   const t = document.createElementNS(NS,'text');
   t.setAttribute('x',n.x); t.setAttribute('y',n.y+4); t.setAttribute('text-anchor','middle');
   t.setAttribute('font-size','10.5'); t.setAttribute('font-weight','700'); t.setAttribute('fill','#fff');
   t.textContent = n.id.replace('Project ','');
   g.appendChild(r); g.appendChild(t); gN.appendChild(g);
   shapes[n.id] = {g:g, r:r};});

 // ---- the algorithm, in exactly the shape of graph_search() ---------------
 let start = '__START__', seen = {}, frontier = [], hop = 0,
     tree = {}, skippedEdges = {}, lastNew = [], lastSkips = 0, timer = null;

 function init(s){
   if (timer) {clearInterval(timer); timer = null;}
   start = s; seen = {}; seen[start] = {depth:0, path:[]};
   frontier = [start]; hop = 0; tree = {}; skippedEdges = {}; lastNew = []; lastSkips = 0;
   document.querySelectorAll('.bf-st').forEach(b =>
     b.classList.toggle('on', b.textContent === s));
   paint('▶ <b>hop 0.</b> <span class="bf-code">seen</span> holds the start node at depth 0 and '
       + '<span class="bf-code">frontier</span> holds it too. Nothing else exists yet.');
 }

 function step(){
   if (!frontier.length) return false;
   hop += 1;
   const next = []; lastNew = []; lastSkips = 0;
   frontier.forEach(u => adj[u].forEach(a => {
     if (a.to in seen) {                                  // TODO 1 — already reached
       if (!(a.i in tree)) skippedEdges[a.i] = 1;
       lastSkips += 1; return;
     }
     seen[a.to] = {depth: hop,                            // TODO 2 — depth + the chain
                   path: seen[u].path.concat([{frm:u, to:a.to, r:a.e.r, src:a.e.src}])};
     tree[a.i] = hop;
     next.push(a.to); lastNew.push(a.to);                 // TODO 3 — next frontier
   }));
   const expanded = frontier.length;
   frontier = next;
   const progs = Object.keys(seen).filter(k => pos[k].kind === 'programme');
   let msg = '▶ <b>hop ' + hop + '.</b> Expanded <b>' + expanded + '</b> frontier node'
           + (expanded === 1 ? '' : 's') + ' → discovered <b>' + lastNew.length + '</b> new node'
           + (lastNew.length === 1 ? '' : 's')
           + ', and hit <b>' + lastSkips + '</b> neighbour' + (lastSkips === 1 ? '' : 's')
           + ' already in <span class="bf-code">seen</span> (skipped — that is <span class="bf-code">TODO 1</span>, '
           + 'and it is what stops the traversal walking in circles).';
   if (!frontier.length)
     msg += '<br><br>🏁 <b>The frontier is empty, so the traversal is over.</b> '
          + Object.keys(seen).length + ' of ' + N.length + ' nodes were reachable from <b>' + start
          + '</b>, and every one of them carries the shortest chain that reached it. '
          + 'Programmes reached: <b>' + (progs.map(p => p.replace('Project ','')).join(', ') || 'none') + '</b>. '
          + 'Nothing was ranked and nothing was dropped — this list is <i>complete</i>.';
   paint(msg);
   return true;
 }

 // ---- rendering -----------------------------------------------------------
 function paint(msg){
   lines.forEach((l,i) => {
     const e = E[i];
     if (i in tree){
       l.setAttribute('stroke', DC[Math.min(tree[i], DC.length-1)]);
       l.setAttribute('stroke-width','3'); l.setAttribute('opacity','1');
       l.setAttribute('stroke-dasharray','');
     } else if (i in skippedEdges){
       l.setAttribute('stroke','#e0a23c'); l.setAttribute('stroke-width','1.8');
       l.setAttribute('opacity','.9'); l.setAttribute('stroke-dasharray','4 3');
     } else {
       l.setAttribute('stroke','#c9c4dd'); l.setAttribute('stroke-width','1.2');
       l.setAttribute('opacity','.3'); l.setAttribute('stroke-dasharray','');
     }});
   N.forEach(n => {
     const s = shapes[n.id], inF = frontier.indexOf(n.id) >= 0;
     if (n.id in seen){
       s.g.setAttribute('opacity','1');
       s.r.setAttribute('fill', DC[Math.min(seen[n.id].depth, DC.length-1)]);
       s.r.setAttribute('stroke', inF ? '#2c2350' : 'none');
       s.r.setAttribute('stroke-width', inF ? '2.5' : '0');
     } else {
       s.g.setAttribute('opacity','.22');
       s.r.setAttribute('fill','#b9b4cc'); s.r.setAttribute('stroke-width','0');
     }});

   document.getElementById('bfNarr').innerHTML = msg;
   document.getElementById('bfNext').disabled = !frontier.length;

   document.getElementById('bfFrontier').innerHTML = frontier.length
     ? frontier.map(n => '<span class="bf-chip new">' + n + '</span>').join('')
       + '<div class="bf-row">' + frontier.length + ' node(s) to expand on the next hop</div>'
     : '<span class="bf-none">empty — every reachable node has been found</span>';

   const byDepth = {};
   Object.keys(seen).forEach(k => {(byDepth[seen[k].depth] = byDepth[seen[k].depth] || []).push(k);});
   document.getElementById('bfSeen').innerHTML =
     Object.keys(byDepth).sort((a,b) => a-b).map(d =>
       '<div class="bf-row"><span class="bf-dep" style="background:'
       + DC[Math.min(d, DC.length-1)] + '">hop ' + d + '</span>'
       + byDepth[d].map(n => '<span class="bf-chip">' + n + '</span>').join('') + '</div>').join('')
     + '<div class="bf-row" style="color:#a9a3c4">' + Object.keys(seen).length
     + ' of ' + N.length + ' nodes recorded' + (lastSkips ? ' · ' + lastSkips
     + ' revisit(s) skipped this hop' : '') + '</div>';

   document.getElementById('bfPath').innerHTML = lastNew.length
     ? lastNew.slice(0,3).map(n => '<div class="bf-row"><b>' + n + '</b>'
         + seen[n].path.map(s2 => '<span class="bf-tri">' + s2.frm + ' --' + s2.r + '--> '
             + s2.to + ' <span class="bf-src">[' + s2.src + ']</span></span>').join('')
         + '</div>').join('')
       + (lastNew.length > 3 ? '<div class="bf-row" style="color:#a9a3c4">+'
          + (lastNew.length-3) + ' more found this hop</div>' : '')
     : '<span class="bf-none">no new node found yet — press next hop</span>';
 }

 window.bfStart = function(s){init(s);};
 window.bfReset = function(){init(start);};
 window.bfStep  = function(){if (timer){clearInterval(timer); timer = null;} step();};
 window.bfAuto  = function(){
   if (timer) {clearInterval(timer); timer = null; return;}
   timer = setInterval(function(){if (!step()){clearInterval(timer); timer = null;}}, 1100);};
 init(start);
})();
</script>
'''.replace("__NODES__", _json.dumps(_bf_nodes))
   .replace("__EDGES__", _json.dumps(_bf_edges))
   .replace("__BTNS__", _bf_btns)
   .replace("__START__", _BFS_STARTS[0])))

In [ ]:
def graph_search(start, max_depth=3):
    """BFS from `start`. Returns {node: {"depth": int, "path": [edge dicts]}}."""
    if start not in G:
        return {}
    seen = {start: dict(depth=0, path=[])}
    frontier = [start]

    for d in range(1, max_depth + 1):
        next_frontier = []
        for node in frontier:
            for step in get_neighbors(node):
                nb = step["neighbor"]
                # 🎯 TODO 1: already reached (at an equal or shorter depth) → skip it
                if ...:      # 🎯 is nb already a key in `seen`?
                    continue
                # 🎯 TODO 2: record the depth and the full edge chain that reached it.
                #   the chain so far is seen[node]["path"]; this hop is dict(frm=node, **step)
                # 🎯 depth is the loop variable d · path is seen[node]["path"] + this hop which is: [dict(frm=node, **step)
                seen[nb] = dict(depth=..., path=...)
                # 🎯 TODO 3: it becomes part of the next frontier
                next_frontier.append(...)   # 🎯 nb
        frontier = next_frontier
        if not frontier:
            break
    return seen

reach = graph_search("Sanko Photonics", max_depth=3)
for _n, _info in sorted(reach.items(), key=lambda kv: (kv[1]["depth"], kv[0])):
    print(f"  hop {_info['depth']}  {_n:<24} ({NODE_TYPE.get(_n,'?')})")

### 4.3 — The blast radius

Filter that traversal to programmes and you have the answer the CEO actually wanted — the one the
retriever could not produce at any `top_k`.

**⚠️ One honest caveat about the traversal you just wrote.** We built `G` as an *undirected* graph, so
BFS is happy to walk edges backwards. Keep expanding past the programmes and it will "reach" PCB-K3
via `Orion → PCB-K3` — as if a wafer fire threatened mainboards. It does not; that edge was traversed
against the direction of causation.

Two ways to handle it, and you should know both: **filter the output** by node type (what
`blast_radius` does below — simple, and enough here), or **model direction properly** with a
`DiGraph` and traverse only along `supplies`/`depends_on` in the causal sense. Real systems do the
second, and it is the most common source of quietly wrong GraphRAG answers: *an edge that exists is
not an edge that propagates.*

In [ ]:
def blast_radius(epicentre, max_depth=4):
    """Which programmes are reachable from a disruption, and by what path?"""
    reach = graph_search(epicentre, max_depth=max_depth)
    return {n: info for n, info in reach.items() if NODE_TYPE.get(n) == "programme"}

exposed = blast_radius("Sanko cleanroom fire", max_depth=4)
for _n, _i in sorted(exposed.items(), key=lambda kv: kv[1]["depth"]):
    chain = " → ".join([_i["path"][0]["frm"]] + [s["neighbor"] for s in _i["path"]])
    print(f"{_n:<16} (hop {_i['depth']})   {chain}")

In [ ]:
#@title 💥 Blast radius — sorted by how many hops from the fire { display-mode: "form" }
from IPython.display import HTML, display
_cards = ""
# order: the ones a human analyst would find first, then the ones that hide
_ordered = sorted(exposed.items(),
                  key=lambda kv: ("Apex Components" not in [s["neighbor"] for s in kv[1]["path"]],
                                  kv[1]["depth"], kv[0]))
for _n, _i in _ordered:
    _via = [s["neighbor"] for s in _i["path"]]
    _hidden = "Apex Components" not in _via          # reached through a supplier nobody is watching
    _c = "#c0392b" if _hidden else "#e0a23c"
    _seq = [_i["path"][0]["frm"]] + _via
    _srcs = [s["source"] for s in _i["path"]]
    _chain = ""
    for _k, _node in enumerate(_seq):
        _chain += f'<span class="br-n">{_esc(_node)}</span>'
        if _k < len(_srcs):
            _chain += f'<span class="br-a">→<sup>{_srcs[_k]}</sup></span>'
    _found = ("INVISIBLE — routed via Meridian Optics" if _hidden
              else "findable — routed via Apex, the supplier already under discussion")
    _fc = "#c0392b" if _hidden else "#2f9e5c"
    _cards += (f'<div class="br-c" style="border-left:5px solid {_c}">'
               f'<div class="br-hd"><b>{_esc(_n)}</b><span class="br-hop" style="background:{_c}">hop {_i["depth"]}</span>'
               f'<span class="br-f" style="color:{_fc}">{_found}</span></div>'
               f'<div class="br-p">{_chain}</div></div>')
display(HTML('''
<style>
.br{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.br-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.br-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.br-c{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px;margin:7px 0}
.br-hd{display:flex;align-items:center;gap:9px;font-size:13px;color:#2c2350}
.br-hop{font-size:9.5px;font-weight:800;color:#fff;border-radius:6px;padding:2px 8px}
.br-f{margin-left:auto;font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.4px}
.br-p{margin-top:7px;font-size:10.5px;line-height:2}
.br-n{background:#f2f0fc;color:#4b3f7a;border-radius:6px;padding:3px 7px;font-weight:700}
.br-a{color:#b9a9e6;margin:0 3px;font-weight:800}
.br-a sup{color:#c9c4dd;font-family:ui-monospace,Menlo,monospace;font-size:8px}
.br-foot{margin-top:12px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="br">
 <div class="br-h">💥 Every programme downstream of one cleanroom fire</div>
 <div class="br-s">Each row is a chain of relations, with the source document for each hop written above the arrow. Note that <b>hop count is not the interesting variable</b> — the <i>route</i> is.</div>
 __CARDS__
 <div class="br-foot">🔑 <b>Two kinds of finding, at the same distance.</b> Orion, Nova and Vega are all reached through <b>Apex Components</b> — the supplier already named in every incident report, so a diligent analyst gets there eventually. <b>Helios</b> is exactly as exposed and sits exactly as far from the fire, but it is reached through <b>Meridian Optics</b>: a different supplier, a different component, no shared part number with anything in the Orion story, and not one document connecting it to the incident. Its status report (D03) still says <i>green</i>. That is what sub-tier supply-chain risk looks like in practice — and it is a graph query, not a search query.</div>
</div>'''.replace("__CARDS__", _cards)))

In [ ]:
#@title ⚠️ An edge that exists is not an edge that propagates — see it happen { display-mode: "form" }
from IPython.display import HTML, display
import json as _json

# Which way does a DISRUPTION travel along each relation? This is NOT the direction the triple is
# written in. "Project Orion depends_on OS-17" is written programme → component, but risk travels
# component → programme. Getting this table wrong is the classic quietly-wrong GraphRAG answer.
_FLOW = {"supplies":           "fwd",   # Apex supplies OS-17      → a bad Apex poisons OS-17
         "supplies_wafers_to": "fwd",   # Sanko supplies Apex      → a bad Sanko poisons Apex
         "depends_on":         "bwd",   # Orion depends_on OS-17   → a bad OS-17 poisons Orion
         "disrupted_by":       "bwd",   # Apex disrupted_by halt   → the halt poisons Apex
         "caused_by":          "bwd"}   # halt caused_by fire      → the fire poisons the halt

_dir_nodes = [dict(id=n, x=_POS[n][0], y=_POS[n][1], kind=NODE_TYPE.get(n, "other")) for n in _POS]
_dir_edges = [dict(s=s, o=o, r=r, src=src, flow=_FLOW[r]) for (s, r, o, src) in EDGES]

display(HTML('''
<style>
.dr{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:960px;color:#2c2350}
.dr-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.dr-s{font-size:12px;color:#6b6685;margin:0 0 12px;line-height:1.55}
.dr-idea{background:#fff;border:1px solid #e7e4f6;border-left:4px solid #e0796d;border-radius:12px;padding:11px 13px;font-size:11.5px;line-height:1.6;margin-bottom:12px}
.dr-tbl{display:flex;gap:6px;flex-wrap:wrap;margin-top:7px}
.dr-rel{font-family:ui-monospace,Menlo,monospace;font-size:9.5px;font-weight:700;border-radius:6px;padding:3px 7px}
.dr-f{background:#e7f6ee;color:#2f9e5c}
.dr-b{background:#eef0ff;color:#4b3f7a}
.dr-ctrl{display:flex;gap:7px;flex-wrap:wrap;align-items:center;margin-bottom:9px}
.dr-btn{cursor:pointer;border:none;border-radius:10px;padding:8px 13px;font-size:12px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2)}
.dr-btn.fire{background:linear-gradient(135deg,#e0796d,#c0392b)}
.dr-btn.alt{background:#fff;color:#4b3f7a;border:1px solid #d9d5ee}
.dr-tog{display:inline-flex;background:#fff;border:1px solid #d9d5ee;border-radius:10px;overflow:hidden}
.dr-tog button{cursor:pointer;border:none;background:#fff;color:#4b3f7a;font-size:11.5px;font-weight:800;padding:8px 12px}
.dr-tog button.on{background:#667eea;color:#fff}
.dr-cap{display:inline-flex;align-items:center;gap:7px;background:#fff;border:1px solid #d9d5ee;border-radius:10px;padding:4px 8px;font-size:11.5px;font-weight:700;color:#4b3f7a}
.dr-cap button{cursor:pointer;border:none;background:#f2f0fc;color:#4b3f7a;border-radius:6px;width:22px;height:22px;font-size:13px;font-weight:800}
.dr-legend{display:flex;gap:10px;flex-wrap:wrap;font-size:10px;color:#5b5578;margin:2px 0 8px;align-items:center}
.dr-lg{display:flex;align-items:center;gap:4px}
.dr-dot{width:9px;height:9px;border-radius:50%;display:inline-block}
.dr-ring{width:10px;height:10px;border-radius:50%;display:inline-block;border:2px dashed #c0392b}
.dr-svgwrap{background:#fff;border:1px solid #e7e4f6;border-radius:14px;padding:6px;overflow-x:auto}
.dr-narr{margin-top:10px;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #9a63d4;border-radius:11px;padding:10px 13px;font-size:11.5px;line-height:1.6;min-height:40px}
.dr-code{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;background:#f7f6fc;color:#4b3f7a;border-radius:5px;padding:1px 5px}
.dr-grid{display:flex;gap:8px;flex-wrap:wrap;margin-top:10px}
.dr-pc{box-sizing:border-box;flex:1 1 138px;min-width:132px;max-width:200px;background:#fff;border:1px solid #e7e4f6;border-radius:11px;padding:9px 11px}
.dr-pn{font-size:12px;font-weight:800;color:#2c2350}
.dr-pd{font-size:9.5px;color:#a9a3c4;font-family:ui-monospace,Menlo,monospace;margin-top:2px}
.dr-pv{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.3px;margin-top:5px;line-height:1.35}
.dr-two{display:flex;gap:10px;flex-wrap:wrap;margin-top:12px}
.dr-bx{flex:1 1 260px;border-radius:12px;padding:11px 13px;font-size:11.5px;line-height:1.6}
.dr-bad{background:#fdf3f1;border-left:4px solid #e0796d;color:#7a3d34}
.dr-good{background:#eafaf0;border-left:4px solid #39b36a;color:#1e6b40}
.dr-foot{margin-top:10px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:4px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
.dr text{font-family:system-ui,Segoe UI,Roboto,sans-serif;pointer-events:none}
</style>
<div class="dr">
 <div class="dr-h">⚠️ The same traversal, run twice: once naively, once correctly</div>
 <div class="dr-s">You wrote BFS over an <b>undirected</b> graph, so it walks every edge in both directions. Some of those directions are nonsense. Flip the toggle and watch the answer change.</div>

 <div class="dr-idea">🧭 <b>The trap:</b> the direction a disruption travels is <b>not</b> the direction the triple is written in.
  <span class="dr-code">Project Orion depends_on OS-17</span> is written programme → component, but risk travels
  <b>component → programme</b>. So "follow the arrows as written" is just as wrong as ignoring them. Every relation needs its own answer:
  <div class="dr-tbl">__RELS__</div>
  <div style="margin-top:7px;font-size:11px;color:#7a3d34">Nothing in the graph itself tells you this table — <b>you</b> have to supply it. That is why an edge that <i>exists</i> is not an edge that <i>propagates</i>.</div></div>

 <div class="dr-ctrl">
  <span class="dr-tog">
   <button id="drU" class="on" onclick="drMode('undirected')">undirected — what you wrote</button>
   <button id="drC" onclick="drMode('causal')">causal direction only</button>
  </span>
  <span class="dr-cap"><button onclick="drCap(-1)">−</button>max_depth = <span id="drCapN">4</span><button onclick="drCap(1)">+</button></span>
  <button class="dr-btn fire" onclick="drTrace()">🔦 trace the false positive</button>
  <button class="dr-btn alt" onclick="drReset()">↺ reset</button>
 </div>
 <div class="dr-legend">
  <span class="dr-lg"><i class="dr-dot" style="background:#c0392b"></i>the fire (hop 0)</span>
  <span class="dr-lg"><i class="dr-dot" style="background:#e0796d"></i>1</span>
  <span class="dr-lg"><i class="dr-dot" style="background:#e0a23c"></i>2</span>
  <span class="dr-lg"><i class="dr-dot" style="background:#9a63d4"></i>3</span>
  <span class="dr-lg"><i class="dr-dot" style="background:#4c8dd8"></i>4</span>
  <span class="dr-lg"><i class="dr-dot" style="background:#39b36a"></i>5+</span>
  <span class="dr-lg"><i class="dr-ring"></i>reached only by walking an edge backwards</span>
 </div>
 <div class="dr-svgwrap"><svg id="drSvg" viewBox="0 0 810 512" style="width:100%;min-width:640px;height:auto"></svg></div>
 <div class="dr-narr" id="drNarr"></div>
 <div class="dr-grid" id="drProgs"></div>

 <div class="dr-two">
  <div class="dr-bx dr-bad"><b>❌ What the naive version claims</b><br>
   Left to run to exhaustion, undirected BFS reaches <b>every node in the graph</b> — so every programme is "exposed", which is the same as saying nothing is. The reachable set stopped being an answer and became a list of the company.</div>
  <div class="dr-bx dr-good"><b>✅ What the causal version claims</b><br>
   Following only the direction disruption actually travels, the traversal <b>runs out on its own</b> at the programmes. No depth cap needed, no filtering needed: the answer set is exactly the four programmes whose supply chains touch Sanko.</div>
 </div>
 <div class="dr-foot" id="drFoot"></div>
</div>
<script>
(function(){
 const N = __NODES__, E = __EDGES__, START = 'Sanko cleanroom fire';
 const svg = document.getElementById('drSvg');
 const NS = 'http://www.w3.org/2000/svg';
 const DC = ['#c0392b','#e0796d','#e0a23c','#9a63d4','#4c8dd8','#39b36a','#39b36a','#39b36a'];
 const pos = {}; N.forEach(n => pos[n.id] = n);
 const adj = {}; N.forEach(n => adj[n.id] = []);
 E.forEach((e,i) => {adj[e.s].push({to:e.o, i:i, e:e}); adj[e.o].push({to:e.s, i:i, e:e});});

 const gE = document.createElementNS(NS,'g'), gN = document.createElementNS(NS,'g');
 svg.appendChild(gE); svg.appendChild(gN);
 const lines = E.map(e => {
   const l = document.createElementNS(NS,'line');
   l.setAttribute('x1',pos[e.s].x); l.setAttribute('y1',pos[e.s].y);
   l.setAttribute('x2',pos[e.o].x); l.setAttribute('y2',pos[e.o].y);
   gE.appendChild(l); return l;});
 const shapes = {};
 N.forEach(n => {
   const g = document.createElementNS(NS,'g');
   const w = Math.max(70, n.id.length*6.4 + 16);
   const r = document.createElementNS(NS,'rect');
   r.setAttribute('x',n.x-w/2); r.setAttribute('y',n.y-13);
   r.setAttribute('width',w); r.setAttribute('height',26); r.setAttribute('rx',13);
   const t = document.createElementNS(NS,'text');
   t.setAttribute('x',n.x); t.setAttribute('y',n.y+4); t.setAttribute('text-anchor','middle');
   t.setAttribute('font-size','10.5'); t.setAttribute('font-weight','700'); t.setAttribute('fill','#fff');
   t.textContent = n.id.replace('Project ','');
   g.appendChild(r); g.appendChild(t); gN.appendChild(g);
   shapes[n.id] = {g:g, r:r};});

 // does a disruption at `u` travel along edge `e` to the other end?
 function flows(u, a){
   return a.e.flow === 'fwd' ? (a.e.s === u) : (a.e.o === u);
 }
 // BFS. mode 'causal' honours the propagation direction; 'undirected' walks anything.
 function bfs(mode, cap){
   const seen = {}; seen[START] = {d:0, path:[]};
   let frontier = [START];
   for (let d = 1; d <= cap; d++){
     const next = [];
     frontier.forEach(u => adj[u].forEach(a => {
       if (a.to in seen) return;
       if (mode === 'causal' && !flows(u, a)) return;
       seen[a.to] = {d:d, path: seen[u].path.concat([{frm:u, to:a.to, i:a.i,
                     legal: flows(u, a), r:a.e.r, src:a.e.src}])};
       next.push(a.to);}));
     if (!next.length) break;
     frontier = next;
   }
   return seen;
 }

 let mode = 'undirected', cap = 4, traced = null;
 const PROGS = N.filter(n => n.kind === 'programme').map(n => n.id).sort();
 const truth = bfs('causal', 99);            // the ground truth: unbounded, direction-aware

 function paint(){
   const seen = bfs(mode, cap);
   const inTree = {}; Object.keys(seen).forEach(k => seen[k].path.forEach(s => {inTree[s.i] = seen[k].d;}));
   const tracePath = traced && seen[traced] ? seen[traced].path : null;
   const traceIdx = {}; if (tracePath) tracePath.forEach(s => {traceIdx[s.i] = s.legal ? 1 : 2;});

   lines.forEach((l,i) => {
     if (i in traceIdx){
       l.setAttribute('stroke', traceIdx[i] === 2 ? '#c0392b' : '#2f9e5c');
       l.setAttribute('stroke-width','4'); l.setAttribute('opacity','1');
       l.setAttribute('stroke-dasharray', traceIdx[i] === 2 ? '5 3' : '');
     } else if (i in inTree){
       l.setAttribute('stroke', DC[Math.min(inTree[i], DC.length-1)]);
       l.setAttribute('stroke-width','2.6'); l.setAttribute('opacity', tracePath ? '.25' : '1');
       l.setAttribute('stroke-dasharray','');
     } else {
       l.setAttribute('stroke','#c9c4dd'); l.setAttribute('stroke-width','1.2');
       l.setAttribute('opacity','.28'); l.setAttribute('stroke-dasharray','');
     }});
   N.forEach(n => {
     const s = shapes[n.id];
     if (n.id in seen){
       s.g.setAttribute('opacity','1');
       s.r.setAttribute('fill', DC[Math.min(seen[n.id].d, DC.length-1)]);
       const bogus = !(n.id in truth);
       s.r.setAttribute('stroke', bogus ? '#c0392b' : 'none');
       s.r.setAttribute('stroke-width', bogus ? '2.5' : '0');
       s.r.setAttribute('stroke-dasharray', bogus ? '4 2' : '');
     } else {
       s.g.setAttribute('opacity','.2');
       s.r.setAttribute('fill','#b9b4cc'); s.r.setAttribute('stroke-width','0');
     }});

   const hit = PROGS.filter(p => p in seen);
   const bogus = hit.filter(p => !(p in truth));
   document.getElementById('drProgs').innerHTML = PROGS.map(p => {
     const got = p in seen, real = p in truth;
     const col = !got ? '#c9c4dd' : (real ? '#2f9e5c' : '#c0392b');
     const verdict = !got ? 'not reached at this depth'
                   : (real ? '✓ genuinely downstream' : '✗ false positive — path turns around');
     return '<div class="dr-pc" style="border-left:4px solid ' + col + '">'
          + '<div class="dr-pn">' + p.replace('Project ','') + '</div>'
          + '<div class="dr-pd">' + (got ? 'hop ' + seen[p].d : '—') + '</div>'
          + '<div class="dr-pv" style="color:' + col + '">' + verdict + '</div></div>';}).join('');

   let msg = '<b>' + (mode === 'causal' ? 'Causal direction only' : 'Undirected')
           + '</b>, <span class="dr-code">max_depth = ' + cap + '</span> → reached <b>'
           + (Object.keys(seen).length) + ' of ' + N.length + '</b> nodes and <b>' + hit.length
           + '</b> of ' + PROGS.length + ' programmes';
   msg += bogus.length
     ? ', of which <b style="color:#c0392b">' + bogus.length + ' should not be there</b>: '
       + bogus.map(p => p.replace('Project ','')).join(', ')
       + '. Each was reached by walking at least one edge against the direction a disruption travels.'
     : ' — and <b style="color:#2f9e5c">every one of them is genuinely exposed</b>.';
   if (mode === 'undirected' && cap <= 4)
     msg += '<br><br>💡 This is the setting the notebook ran. It looks correct — try pressing <b>+</b>.';
   if (mode === 'undirected' && cap >= 6)
     msg += '<br><br>🕳️ <b>Luna and Atlas are not exposed to a wafer fire.</b> Luna arrives via '
          + '<span class="dr-code">Orion → PCB-K3</span>, read backwards; Atlas via Vega\\'s telemetry SDK. '
          + 'Filtering the output to <i>programmes</i> does not remove them — they <b>are</b> programmes.';
   if (mode === 'causal' && cap >= 5)
     msg += '<br><br>✅ Raising the cap changes nothing now: the traversal already ran out of legal edges at hop 4. '
          + 'A correct model does not need a depth cap to protect it.';
   document.getElementById('drNarr').innerHTML = msg;

   document.getElementById('drFoot').innerHTML = tracePath
     ? '🔦 <b>' + traced + '</b>, hop ' + seen[traced].d + ': '
       + '<span class="dr-code">' + tracePath[0].frm + '</span>'
       + tracePath.map(s => (s.legal
           ? '<span style="color:#b9a9e6;font-weight:800"> → </span>'
           : '<span style="color:#c0392b;font-weight:800"> ✗→ </span>')
         + '<span class="dr-code"' + (s.legal ? '' : ' style="background:#fdecea;color:#c0392b"')
         + '>' + s.to + '</span>').join('')
       + '<br>The red hop is <b>' + tracePath.filter(s => !s.legal).map(s => s.frm + ' ' + s.r + ' ' + s.to).join(', ')
       + '</b> — a real edge, read in the wrong direction. Orion <i>needs</i> PCB-K3; a shortage of optics '
       + 'does not travel out of Orion and back down into the mainboard supply.'
     : '🔑 <b>The lesson.</b> Filtering the result by node type is a patch, not a fix: it only worked above because '
       + '<span class="dr-code">max_depth=4</span> happened to stop one hop before the first backwards edge mattered. '
       + 'Model the direction and the traversal is correct at <i>any</i> depth — which is what a '
       + '<span class="dr-code">nx.DiGraph</span> plus this relation table buys you.';
 }

 window.drMode  = function(m){mode = m; traced = null;
   document.getElementById('drU').classList.toggle('on', m === 'undirected');
   document.getElementById('drC').classList.toggle('on', m === 'causal'); paint();};
 window.drCap   = function(x){cap = Math.max(1, Math.min(7, cap + x));
   document.getElementById('drCapN').textContent = cap; paint();};
 window.drTrace = function(){mode = 'undirected'; cap = 6; traced = 'Project Luna';
   document.getElementById('drU').classList.add('on');
   document.getElementById('drC').classList.remove('on');
   document.getElementById('drCapN').textContent = cap; paint();};
 window.drReset = function(){mode = 'undirected'; cap = 4; traced = null;
   document.getElementById('drU').classList.add('on');
   document.getElementById('drC').classList.remove('on');
   document.getElementById('drCapN').textContent = cap; paint();};
 paint();
})();
</script>
'''.replace("__NODES__", _json.dumps(_dir_nodes))
   .replace("__EDGES__", _json.dumps(_dir_edges))
   .replace("__RELS__", "".join(
       f'<span class="dr-rel dr-{"f" if v == "fwd" else "b"}">{k}: '
       f'{"subject → object" if v == "fwd" else "object → subject"}</span>'
       for k, v in _FLOW.items()))))

---
# Part 5 — Graph retrieval is not enough: get the text back

Here is where a lot of GraphRAG demos quietly cheat. They traverse a graph, print triples, and call
it an answer.

A triple is a **pointer**, not evidence. `Apex --disrupted_by--> Apex production halt` does not tell
the COO *when*, *how long*, *what was said*, or *how confident to be*. And it cannot be audited: if
the extraction was wrong, nothing downstream will ever notice.

The real pattern closes the loop back to the corpus:

> **traverse the graph → collect the source ids on the path → fetch the original passages → answer
> from those.** The graph decides *what is relevant*; the text remains the *evidence*.

In [ ]:
#@title 🔗 A triple is a pointer, not evidence — build the context hop by hop { display-mode: "form" }
from IPython.display import HTML, display
import json as _json

# The path is computed here from the graph itself, so this cell works before you write
# retrieve_sources() below — and does not depend on your version being right.
_g5_nodes = nx.shortest_path(G, "Project Orion", "Project Helios")
_g5_hops = []
for _a, _b in zip(_g5_nodes, _g5_nodes[1:]):
    _e = G.edges[_a, _b]
    _d0 = BY_ID[_e["source"]]
    _g5_hops.append(dict(
        triple=(f"{_a} --{_e['relation']}--> {_b}" if _e["subject"] == _a
                else f"{_b} --{_e['relation']}--> {_a}"),
        src=_e["source"], date=_d0["date"], kind=_d0["kind"],
        title=_d0["title"], text=_d0["text"]))

# Six things the COO will actually ask. Some are answerable from the structure alone;
# the rest need the sentence the edge was extracted from.
_g5_qs = [
    dict(q="Which component is Orion blocked on?",            by="triple", at=1),
    dict(q="Who supplies OS-17?",                             by="triple", at=2),
    dict(q="Who else does Sanko Photonics supply?",           by="triple", at=4),
    dict(q="Is there a qualified second source for OS-17?",   by="doc", doc="D20",
         quote="no qualified second source exists as of this revision"),
    dict(q="Is Meridian sole-source on those wafers too?",    by="doc", doc="D33",
         quote="Both hold sole-source status for the wafer grades listed"),
    dict(q="What do I cite in the board pack on the 20th?",   by="doc", doc="ANY",
         quote="every claim carries a document id and a date"),
]

display(HTML('''
<style>
.gd{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:960px;color:#2c2350}
.gd-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.gd-s{font-size:12px;color:#6b6685;margin:0 0 12px;line-height:1.55}
.gd-ctrl{display:flex;gap:7px;flex-wrap:wrap;align-items:center;margin-bottom:10px}
.gd-btn{cursor:pointer;border:none;border-radius:10px;padding:8px 13px;font-size:12px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2)}
.gd-btn.go{background:linear-gradient(135deg,#39b36a,#2f9e5c)}
.gd-btn.bug{background:linear-gradient(135deg,#e0796d,#c0392b)}
.gd-btn.alt{background:#fff;color:#4b3f7a;border:1px solid #d9d5ee}
.gd-btn:disabled{opacity:.4;cursor:default}
.gd-tog{display:inline-flex;background:#fff;border:1px solid #d9d5ee;border-radius:10px;overflow:hidden}
.gd-tog button{cursor:pointer;border:none;background:#fff;color:#4b3f7a;font-size:11.5px;font-weight:800;padding:8px 12px}
.gd-tog button.on{background:#667eea;color:#fff}
.gd-chain{display:flex;gap:4px;flex-wrap:wrap;align-items:center;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 12px;margin-bottom:10px}
.gd-n{font-size:10.5px;font-weight:700;border-radius:7px;padding:4px 8px;background:#f2f0fc;color:#4b3f7a;opacity:.35}
.gd-n.on{opacity:1;background:#667eea;color:#fff}
.gd-ar{color:#c9c4dd;font-size:11px;font-weight:800}
.gd-ar.on{color:#39b36a}
.gd-two{display:flex;gap:10px;flex-wrap:wrap}
.gd-col{flex:1 1 320px}
.gd-ct{font-size:11px;font-weight:800;color:#3b2d6b;margin-bottom:2px}
.gd-cd{font-size:9.5px;color:#a9a3c4;line-height:1.4;margin-bottom:7px}
.gd-ctx{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 12px;min-height:170px;max-height:340px;overflow-y:auto}
.gd-tri{font-family:ui-monospace,Menlo,monospace;font-size:9.5px;font-weight:700;background:#f2f0fc;color:#4b3f7a;border-radius:5px;padding:4px 7px;margin-top:4px;display:block;line-height:1.45}
.gd-doc{border-left:3px solid #39b36a;padding-left:9px;margin:4px 0 8px}
.gd-dh{font-family:ui-monospace,Menlo,monospace;font-size:9px;color:#a9a3c4}
.gd-dt{font-size:10.5px;color:#5b5578;line-height:1.5;margin-top:2px}
.gd-empty{font-size:10.5px;color:#c9c4dd;font-style:italic}
.gd-q{display:flex;gap:8px;align-items:flex-start;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #e8dfe0;border-radius:10px;padding:7px 10px;margin-bottom:5px}
.gd-q.ok{border-left-color:#39b36a}
.gd-qm{font-weight:800;font-size:13px;width:13px;flex:0 0 13px;color:#c4a9a4}
.gd-q.ok .gd-qm{color:#39b36a}
.gd-qt{font-size:11px;line-height:1.4}
.gd-qw{font-size:9.5px;color:#a9a3c4;margin-top:2px;line-height:1.4}
.gd-score{display:flex;align-items:baseline;gap:8px;margin-bottom:7px}
.gd-sn{font-size:22px;font-weight:800;font-variant-numeric:tabular-nums}
.gd-narr{margin-top:10px;background:#fff;border:1px solid #e7e4f6;border-left:4px solid #9a63d4;border-radius:11px;padding:10px 13px;font-size:11.5px;line-height:1.6}
.gd-code{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;background:#f7f6fc;color:#4b3f7a;border-radius:5px;padding:1px 5px}
.gd-audit{margin-top:10px;border-radius:12px;padding:12px 14px;font-size:11.5px;line-height:1.6;background:#fdf3f1;border-left:4px solid #e0796d;color:#7a3d34;display:none}
.gd-audit.show{display:block}
.gd-side{display:flex;gap:10px;flex-wrap:wrap;margin-top:8px}
.gd-sb{flex:1 1 250px;background:#fff;border-radius:10px;padding:9px 11px;font-size:11px;line-height:1.55;color:#3a3357}
</style>
<div class="gd">
 <div class="gd-h">🔗 Assemble the context one hop at a time — and count what it can answer</div>
 <div class="gd-s">This is the Orion → Helios path from the graph. Add the hops one by one under each mode and watch the <b>right-hand column</b>: the triples answer the relational questions perfectly and then stop dead.</div>

 <div class="gd-ctrl">
  <span class="gd-tog">
   <button id="gdT" class="on" onclick="gdMode('triples')">triples only</button>
   <button id="gdS" onclick="gdMode('sources')">triples + source passages</button>
  </span>
  <button class="gd-btn" id="gdNext" onclick="gdStep()">⏭ add the next hop</button>
  <button class="gd-btn go" onclick="gdAll()">▶ add all six</button>
  <button class="gd-btn bug" onclick="gdAudit()">🐛 what if the extraction was wrong?</button>
  <button class="gd-btn alt" onclick="gdReset()">↺ reset</button>
 </div>

 <div class="gd-chain" id="gdChain"></div>

 <div class="gd-two">
  <div class="gd-col">
   <div class="gd-ct">📥 what the LLM actually receives</div>
   <div class="gd-cd">the context string you are about to build in <span class="gd-code">retrieve_sources</span></div>
   <div class="gd-ctx" id="gdCtx"></div>
  </div>
  <div class="gd-col">
   <div class="gd-ct">🙋 what the COO can ask it</div>
   <div class="gd-cd">a question counts as answered only if the context <i>establishes</i> it</div>
   <div class="gd-score"><span class="gd-sn" id="gdScore">0/6</span><span class="gd-cd" id="gdScoreL" style="margin:0"></span></div>
   <div id="gdQs"></div>
  </div>
 </div>

 <div class="gd-narr" id="gdNarr"></div>

 <div class="gd-audit" id="gdAudit">
  <b>🐛 Suppose the extractor got one edge wrong.</b> It read D33 and emitted
  <span class="gd-code">Sanko Photonics --supplies_wafers_to--> Voltix Energy</span> instead of Meridian Optics.
  A single wrong word in one triple.
  <div class="gd-side">
   <div class="gd-sb" style="border-left:4px solid #e0796d"><b>Triples only</b><br>
    The traversal now walks to Voltix, then BAT-4, then <b>Luna and Vega</b>. The model writes a fluent,
    confident briefing naming the wrong programmes. There is no passage in the context to disagree with it,
    no score that looks low, and no id to check. <b>The error is undetectable from the output.</b></div>
   <div class="gd-sb" style="border-left:4px solid #39b36a"><b>Triples + sources</b><br>
    The bad edge still carries <span class="gd-code">[D33]</span>. Open it and the sentence reads
    <i>"…supplies photonic wafers to two qualified regional integrators: Apex Components and Meridian Optics."</i>
    The contradiction is one click away — for a reviewer, and for an automated consistency check.</div>
  </div>
  <div style="margin-top:9px">🔑 Provenance is not decoration. It is the only thing that makes an extraction
   pipeline <b>falsifiable</b> — which is why the rule is that <i>every</i> edge keeps its source id,
   and why the traversal has to carry that id all the way through.</div>
 </div>
</div>
<script>
(function(){
 const HOPS = __HOPS__, QS = __QS__;
 let step = 0, mode = 'triples';

 function docsSoFar(){
   const seen = [], out = [];
   HOPS.slice(0, step).forEach(h => {if (seen.indexOf(h.src) < 0){seen.push(h.src); out.push(h);}});
   return out;
 }
 function answered(q){
   if (q.by === 'triple') return step >= q.at;
   if (mode !== 'sources') return false;
   const ids = docsSoFar().map(h => h.src);
   return q.doc === 'ANY' ? ids.length > 0 : ids.indexOf(q.doc) >= 0;
 }

 function paint(){
   // the path, lit up to the current hop
   let chain = '<span class="gd-n' + (step >= 0 ? ' on' : '') + '">Project Orion</span>';
   HOPS.forEach((h, i) => {
     const on = i < step;
     chain += '<span class="gd-ar' + (on ? ' on' : '') + '">→</span>'
            + '<span class="gd-n' + (on ? ' on' : '') + '">' + h.node + '</span>';});
   document.getElementById('gdChain').innerHTML = chain;

   // the context block
   const ctx = document.getElementById('gdCtx');
   if (!step){ ctx.innerHTML = '<span class="gd-empty">nothing added yet — press "add the next hop"</span>'; }
   else if (mode === 'triples'){
     ctx.innerHTML = HOPS.slice(0, step).map(h => '<span class="gd-tri">' + h.triple + '</span>').join('')
       + '<div class="gd-cd" style="margin-top:8px">' + step + ' triples · '
       + HOPS.slice(0, step).reduce((a,h) => a + h.triple.length, 0) + ' characters of context</div>';
   } else {
     ctx.innerHTML = HOPS.slice(0, step).map(h => '<span class="gd-tri">' + h.triple + '</span>').join('')
       + docsSoFar().map(h => '<div class="gd-doc"><div class="gd-dh">' + h.src + ' · ' + h.date
           + ' · ' + h.kind + '</div><div class="gd-dt">' + h.text + '</div></div>').join('')
       + '<div class="gd-cd">' + step + ' triples + ' + docsSoFar().length + ' passages · '
       + (HOPS.slice(0, step).reduce((a,h) => a + h.triple.length, 0)
          + docsSoFar().reduce((a,h) => a + h.text.length, 0)) + ' characters of context</div>';
   }

   // the COO's questions
   const n = QS.filter(answered).length;
   document.getElementById('gdQs').innerHTML = QS.map(q => {
     const ok = answered(q);
     return '<div class="gd-q' + (ok ? ' ok' : '') + '"><span class="gd-qm">' + (ok ? '✓' : '✗')
          + '</span><span><div class="gd-qt">' + q.q + '</div><div class="gd-qw">'
          + (q.by === 'triple' ? 'answerable from the structure alone'
             : 'needs the sentence: “' + q.quote + '”') + '</div></span></div>';}).join('');
   const col = n === QS.length ? '#39b36a' : (n >= 4 ? '#e0a23c' : '#e0796d');
   document.getElementById('gdScore').innerHTML = '<span style="color:' + col + '">' + n + '/' + QS.length + '</span>';
   document.getElementById('gdScoreL').textContent = mode === 'triples'
     ? 'with triples alone' : 'with the passages behind them';
   document.getElementById('gdNext').disabled = step >= HOPS.length;

   // narration
   let msg;
   if (!step) msg = 'The graph has already decided <b>what is relevant</b> — this exact six-hop path. '
                  + 'The only question left is what you hand the model once it has.';
   else if (mode === 'triples' && step >= HOPS.length)
     msg = '🧱 <b>Stuck at 3 of 6, and adding hops will not move it.</b> The triples answer every '
         + '<i>relational</i> question perfectly — that is what they are for. But "sole-source?", '
         + '"what contract?", "what do I cite?" are questions about <b>what a document says</b>, and no '
         + 'amount of structure contains a sentence. A triple tells you two things are related; it never '
         + 'tells you <i>when</i>, <i>how badly</i>, <i>how sure</i>, or <i>who wrote it down</i>.';
   else if (mode === 'sources' && step >= HOPS.length)
     msg = '✅ <b>6 of 6 — and note where the extra three came from.</b> Not from a better traversal and not '
         + 'from more hops: from the <span class="gd-code">source</span> id each edge was already carrying. '
         + 'The path selected five documents out of twenty-five; the corpus supplied the evidence. '
         + '<b>Structure for reasoning, text for grounding.</b>';
   else msg = 'Hop ' + step + ' added' + (mode === 'sources'
         ? ' — with it, document <b>' + HOPS[step-1].src + '</b> ("' + HOPS[step-1].title + '").'
         : ' as a bare triple. No date, no wording, no id.');
   document.getElementById('gdNarr').innerHTML = msg;
 }

 window.gdMode  = function(m){mode = m;
   document.getElementById('gdT').classList.toggle('on', m === 'triples');
   document.getElementById('gdS').classList.toggle('on', m === 'sources'); paint();};
 window.gdStep  = function(){if (step < HOPS.length) step++; paint();};
 window.gdAll   = function(){step = HOPS.length; paint();};
 window.gdReset = function(){step = 0; document.getElementById('gdAudit').classList.remove('show'); paint();};
 window.gdAudit = function(){document.getElementById('gdAudit').classList.toggle('show');};
 paint();
})();
</script>
'''.replace("__HOPS__", _json.dumps(
       [dict(triple=h["triple"], src=h["src"], date=h["date"], kind=h["kind"],
             title=h["title"], text=h["text"], node=n)
        for h, n in zip(_g5_hops, _g5_nodes[1:])]))
   .replace("__QS__", _json.dumps(_g5_qs))))

### 🎯 5.1 — `retrieve_sources`

Walk a traversal path, collect every edge's `source`, and return the actual documents — deduplicated,
in a stable order. Two lines.

In [ ]:
def retrieve_sources(path):
    """Given a list of edge dicts (a traversal path), return the documents behind them."""
    ids = []
    for step in path:
        # 🎯 TODO 1: append this edge's document id — step["source"] — to `ids`,
        #   skipping any you have already added. Order matters: keep first-seen
        #   order so the citations read in the same sequence as the path.
        ...          # 🎯 ids.append(step["source"]) — but only if it is not in `ids` already
    # look the documents up in the corpus. BY_ID maps document id -> document.
    return [BY_ID[i] for i in ids if i in BY_ID]   # 🎯 a list comprehension over `ids`, looking each one up in BY_ID

def path_to(target, start="Project Orion", max_depth=6):
    """Convenience: the edge chain the BFS used to reach `target` from `start`."""
    return graph_search(start, max_depth=max_depth).get(target, {}).get("path", [])

orion_to_helios = path_to("Project Helios")
print("PATH")
for _s in orion_to_helios:
    print(f"   {_s['triple']:<58} [{_s['source']}]")

print("\nGROUNDING DOCUMENTS")
for _d0 in retrieve_sources(orion_to_helios):
    print(f"   {_d0['id']} · {_d0['date']} · {_d0['title']}")

In [ ]:
#@title 🔗 Triple → passage: the grounding step that makes GraphRAG defensible { display-mode: "form" }
from IPython.display import HTML, display
_rows = ""
for _s in orion_to_helios:
    _d0 = BY_ID[_s["source"]]
    _rows += (f'<div class="gr-row"><div class="gr-tri">{_esc(_s["triple"])}</div>'
              f'<div class="gr-ar">is asserted by</div>'
              f'<div class="gr-doc"><div class="gr-dh">{_d0["id"]} · {_d0["date"]} · {_esc(_d0["kind"])}</div>'
              f'<div class="gr-dt">{_esc(_d0["text"][:210])}…</div></div></div>')
display(HTML('''
<style>
.gr{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.gr-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.gr-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.gr-row{display:flex;gap:11px;align-items:center;flex-wrap:wrap;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px;margin:7px 0}
.gr-tri{flex:0 1 250px;font-family:ui-monospace,Menlo,monospace;font-size:10.5px;font-weight:700;background:#f2f0fc;color:#4b3f7a;border-radius:8px;padding:7px 9px;line-height:1.45}
.gr-ar{flex:0 0 auto;font-size:10px;color:#b9a9e6;font-weight:800;text-transform:uppercase;letter-spacing:.4px}
.gr-doc{flex:1 1 300px;border-left:3px solid #39b36a;padding-left:10px}
.gr-dh{font-family:ui-monospace,Menlo,monospace;font-size:9.5px;color:#a9a3c4}
.gr-dt{font-size:11px;color:#5b5578;line-height:1.5;margin-top:3px}
.gr-bad{margin-top:14px;display:flex;gap:12px;flex-wrap:wrap}
.gr-box{flex:1 1 250px;border-radius:12px;padding:11px 13px;font-size:11.5px;line-height:1.6}
.gr-no{background:#fdf3f1;border-left:4px solid #e0796d;color:#7a3d34}
.gr-yes{background:#eafaf0;border-left:4px solid #39b36a;color:#1e6b40}
</style>
<div class="gr">
 <div class="gr-h">🔗 Every hop, and the sentence that licenses it</div>
 <div class="gr-s">The traversal chose <i>what to look at</i>. The corpus still supplies <i>what is true</i>. Each row is one edge of the Orion→Helios path next to the passage it was extracted from.</div>
 __ROWS__
 <div class="gr-bad">
  <div class="gr-box gr-no"><b>❌ Graph-only GraphRAG</b><br>Feed the LLM bare triples. You get a fluent answer with no dates, no quantities, no nuance, no citations — and an extraction error becomes an invisible hallucination with a confident tone. You cannot audit what you cannot trace.</div>
  <div class="gr-box gr-yes"><b>✅ Graph-as-router GraphRAG</b><br>The graph selects the <i>relevant slice</i> of the corpus; the passages behind it are the evidence. You get citations, dates, and a human-checkable chain. Structure for reasoning, text for grounding.</div>
 </div>
</div>'''.replace("__ROWS__", _rows)))

In [ ]:
graph_evidence = retrieve_sources(orion_to_helios) + retrieve_sources(
    path_to("Project Nova")) + [BY_ID["D01"], BY_ID["D30"], BY_ID["D31"], BY_ID["D32"]]
graph_evidence = list({d["id"]: d for d in graph_evidence}.values())

graph_answer = llm(
    f"Evidence:\n{build_context(graph_evidence)}\n\n"
    f"Known dependency paths from the knowledge graph:\n"
    + "\n".join(s["triple"] for s in orion_to_helios)
    + f"\n\nQuestion: {CEO_QUESTION}",
    system=ANSWER_SYSTEM)
ANSWERS["GraphRAG (path + sources)"] = graph_answer
print(graph_answer)

In [ ]:
fact_coverage(graph_answer, "GraphRAG · fact coverage",
              subtitle="Graph traversal chose the evidence; the original passages grounded it.")

In [ ]:
#@title 🏗️ Architecture — GraphRAG: traverse structure, ground in text { display-mode: "form" }
architecture(
    "Parts 4–5 · GraphRAG — retrieval with no similarity score in it",
    [("q",     "📍", "entry entity",   "an exact node name &middot; here <b>you</b> chose it"),
     ("graph", "🕸️", "BFS traversal",  "<code>graph_search</code> &middot; exhaustive, unranked, no top-k"),
     ("graph", "🔗", "path + edges",   "each edge carries the id of the document it came from"),
     ("text",  "📄", "fetch passages", "<code>retrieve_sources</code> &middot; back to the corpus"),
     ("llm",   "🤖", "LLM",            "structure to reason over, text to cite"),
     ("ans",   "📝", "answer",         "with a defensible chain behind every hop")],
    subtitle="Note what is absent from this row: no embedding, no cosine, no threshold anywhere "
             "in the retrieval path.",
    new=(0, 1, 2, 3),
    note="🔑 The graph decides <b>what is relevant</b>; the corpus still supplies <b>what is true</b>. "
         "Delete the two middle boxes and you get the demo version that answers in bare triples — "
         "fluent, uncitable, and wrong in a way nothing downstream can detect.")


---
# Part 6 — Give the agent both tools and let it choose

Be honest about what just happened: in Part 5 **you** chose the entry point. You knew to start the
traversal at *Project Orion* and to look for *Project Helios*. A deployed system gets a question and
nothing else.

So the last architectural step is a **router**: at each turn the agent decides whether this
sub-question is a *text* question or a *structure* question, and reaches for the matching tool.

| the sub-question is… | example | right tool | why |
|---|---|---|---|
| **descriptive** — what happened, when, how bad | *"What happened at Apex Components?"* | `semantic_search` | the answer is a sentence someone wrote |
| **relational** — who else, what connects, what depends on | *"Which programmes use parts from Apex?"* | `graph_neighbors` | the answer is a join, not a sentence |
| **propagation** — blast radius, downstream impact | *"Who is downstream of the Sanko fire?"* | `graph_expand` | the answer is a multi-hop closure |

In [ ]:
#@title 🧭 The router — matching question shape to retriever shape { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r'''
<style>
.rt{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.rt-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.rt-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.55}
.rt-row{display:flex;gap:12px;align-items:stretch;flex-wrap:wrap}
.rt-q{flex:0 0 128px;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:11px;display:flex;flex-direction:column;justify-content:center;text-align:center}
.rt-qt{font-size:12.5px;font-weight:800;color:#3b2d6b}
.rt-qd{font-size:10px;color:#8b86a6;margin-top:3px;line-height:1.35}
.rt-fan{flex:1;display:flex;flex-direction:column;gap:8px}
.rt-t{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px;display:flex;gap:11px;align-items:flex-start}
.rt-ic{width:30px;height:30px;border-radius:9px;display:flex;align-items:center;justify-content:center;font-size:15px;color:#fff;flex:0 0 30px}
.rt-tt{font-weight:800;font-size:12px;color:#2c2350;font-family:ui-monospace,Menlo,monospace}
.rt-td{font-size:10.8px;color:#5b5578;line-height:1.5;margin-top:3px}
.rt-ex{font-size:10.5px;color:#764ba2;font-style:italic;margin-top:4px}
.rt-foot{margin-top:15px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
.rt-warn{margin-top:8px;font-size:11.5px;color:#7a3d34;background:#fdf3f1;border-left:3px solid #e0796d;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="rt">
 <div class="rt-h">🧭 One agent, three retrievers, one decision per turn</div>
 <div class="rt-s">The router does not ask "which tool is better?" — that question has no answer. It asks <b>"what shape is this sub-question?"</b> and matches shape to retriever.</div>
 <div class="rt-row">
  <div class="rt-q"><div class="rt-qt">sub-question<br>at this turn</div><div class="rt-qd">given the question and everything found so far</div></div>
  <div class="rt-fan">
   <div class="rt-t"><span class="rt-ic" style="background:#667eea">🔍</span><span>
     <span class="rt-tt">semantic_search(query)</span>
     <div class="rt-td">Descriptive needs: what happened, when, how severe, what was said. Fuzzy matching over prose, tolerant of paraphrase.</div>
     <div class="rt-ex">“What happened at Apex Components?”</div></span></div>
   <div class="rt-t"><span class="rt-ic" style="background:#9a63d4">🕸️</span><span>
     <span class="rt-tt">graph_neighbors(entity)</span>
     <div class="rt-td">Relational needs: exact, exhaustive, one hop. No ranking and no misses — either the edge exists or it does not.</div>
     <div class="rt-ex">“Which programmes use parts supplied by Apex?”</div></span></div>
   <div class="rt-t"><span class="rt-ic" style="background:#e0796d">💥</span><span>
     <span class="rt-tt">graph_expand(entity, depth)</span>
     <div class="rt-td">Propagation needs: everything downstream of a shock, graded by distance, with the path kept for citation.</div>
     <div class="rt-ex">“Who is exposed to the Sanko fire?”</div></span></div>
   <div class="rt-t"><span class="rt-ic" style="background:#39b36a">✅</span><span>
     <span class="rt-tt">answer</span>
     <div class="rt-td">The stopping decision. A router without this option investigates forever.</div></span></div>
  </div>
 </div>
 <div class="rt-foot">🔑 <b>Both graph tools end in the corpus.</b> They return triples <i>and</i> the passages behind them, so the evidence pile stays citable no matter which route the agent took.</div>
 <div class="rt-warn">⚠️ <b>The failure mode to design against:</b> a router that always picks the same tool. Usually it is the prompt's fault — vague tool descriptions, or no explicit statement of what each tool is <i>bad</i> at. Tell the model when <b>not</b> to use something.</div>
</div>
'''))

### 🎯 6.1 — Write the routing rules

The JSON plumbing is provided. What you write is the part that actually decides behaviour: **the tool
descriptions**. Two rules of thumb that matter more than they look:

1. State what each tool is **bad at**, not only what it is good at. "Cannot find relationships that
   are not written down in one passage" steers the model far harder than "searches documents".
2. Give the model the **entity vocabulary** of the graph. Graph tools take exact node names; a router
   that hallucinates `"Apex Corp"` gets an empty result and no error message.

In [ ]:
GRAPH_ENTITIES = sorted(G.nodes)

def choose_tool(question, evidence, notes, history=()):
    """Decide the next action. Returns {"tool","argument","why"}."""
    # 🎯 TODO 1: describe each tool — what it is FOR, and what it CANNOT do.
    #   This string is the ENTIRE basis on which the model routes. Vague
    #   descriptions produce a router that picks semantic_search every time.
    #   Reminder of what the three retrievers actually are:
    #     semantic_search  → dense search over the 25 documents
    #     graph_neighbors  → exact one-hop relations of ONE entity in G
    #     graph_expand     → multi-hop closure from ONE entity, with paths
    tool_docs = """
- semantic_search(query: str)
    TODO — good for …?   cannot do …?

- graph_neighbors(entity: str)
    TODO — good for …?   cannot do …?

- graph_expand(entity: str)
    TODO — good for …?   cannot do …?
    (say something about WHERE to expand from: the root of the problem, or a leaf?)

- answer
    TODO — when is stopping right, and when is it never a valid choice?
"""
    # 🎯 TODO 2: hand the model (a) your tool descriptions from above and (b) the
    #   EXACT graph vocabulary — GRAPH_ENTITIES, as a readable comma-separated list.
    #   Graph tools accept literal node names only: a router that invents
    #   "Apex Corp" gets an empty result and no error message.
    #   ⚠️ Unlike every other blank in this notebook, these two sit inside an
    #   f-string — leaving them unfilled does NOT raise. You get the word
    #   "Ellipsis" in the prompt and a router that quietly guesses. Fill both.
    out = llm_json(
        f"INVESTIGATION QUESTION:\n{question}\n\n"
        f"EVIDENCE GATHERED SO FAR:\n{_evidence_digest(evidence, limit=300)}\n\n"
        f"STRUCTURED FACTS ALREADY KNOWN:\n{chr(10).join('- ' + n for n in notes[-25:]) or '(none)'}\n\n"
        f"ACTIONS ALREADY TAKEN — never repeat one of these, it returns nothing new:\n"
        f"{chr(10).join('- ' + h for h in history) or '(none)'}\n\n"
        f"AVAILABLE TOOLS:{...}\n"          # 🎯 tool_docs
        f"VALID GRAPH ENTITY NAMES (graph tools accept ONLY these, spelled exactly):\n"
        f"{...}\n\n"                        # 🎯 GRAPH_ENTITIES, joined into one readable line
        "Pick the ONE next action that closes the biggest remaining gap. Do not repeat an action "
        "whose result you already have. Prefer a graph tool whenever the gap is about which other "
        "entities are connected or exposed.\n"
        '{"tool": "semantic_search|graph_neighbors|graph_expand|answer", '
        '"argument": "query string or exact entity name, empty for answer", '
        '"why": "one short sentence"}')
    return dict(tool=out.get("tool", "answer"),
                argument=(out.get("argument") or "").strip(),
                why=out.get("why", ""))

print("✅ router defined ·", len(GRAPH_ENTITIES), "graph entities exposed to it")

In [ ]:
#@title 🔧 PROVIDED — run_tool + _with_graph_facts (run, don't edit) { display-mode: "form" }
def _with_graph_facts(evidence, notes):
    """Evidence pile + the graph facts, as one list the Part-2 helpers can read."""
    if not notes:
        return evidence
    return evidence + [dict(id="GRAPH", title="knowledge-graph relations", date="", kind="graph",
                            text=" · ".join(notes))]

def run_tool(tool, argument):
    """Execute one routed action. Returns (structured_facts: list[str], docs: list[dict]).
    Note both graph tools also return the SOURCE DOCUMENTS behind the edges they walked —
    structure for reasoning, text for grounding."""
    if tool == "semantic_search":
        hits = vector_search(argument, top_k=4)
        return [], [h["doc"] for h in hits]

    if tool == "graph_neighbors":
        rels = get_neighbors(argument)
        return [r["triple"] for r in rels], retrieve_sources(rels)

    if tool == "graph_expand":
        reach = graph_search(argument, max_depth=5)
        # State the *meaning* of the edge chain, not just the chain. A bare "A → B" reads as a
        # neutral association and loses an argument against a passage that says "B is unrelated".
        facts = [f"{node} is DOWNSTREAM of '{argument}' ({info['depth']} hops): "
                 + " → ".join([argument] + [s["neighbor"] for s in info["path"]])
                 for node, info in reach.items() if info["path"]]
        # Subtle but important: BFS records ONE path per node, so an edge between two nodes
        # that were both already reached is never walked — and its source document silently
        # disappears from the evidence. Here that edge is `Sanko --disrupted_by--> fire` (D32),
        # i.e. the report of the fire itself. Take the sources of the whole INDUCED SUBGRAPH,
        # not just the spanning tree the traversal happened to build.
        nodes = set(reach)
        induced = [dict(source=G.edges[u, v]["source"])
                   for u, v in G.edges(nodes) if u in nodes and v in nodes]
        return facts, retrieve_sources(induced)

    return [], []

print("✅ run_tool ready")

### 6.2 — Does it route sensibly? Test before you trust

Three questions with three different shapes. Check the router picks a different tool for each — if
it picks `semantic_search` three times, go back and sharpen the tool descriptions.

In [ ]:
PROBES = [
    ("What exactly happened at Apex Components, and when?",            "semantic_search"),
    ("Which programmes depend on components supplied by Apex Components?", "graph_neighbors / graph_expand"),
    ("Which programmes are downstream of the Sanko Photonics fire?",   "graph_expand"),
]
routing = [(q, choose_tool(q, [], []), exp) for q, exp in PROBES]
for q, d, exp in routing:
    print(f"❓ {q}\n   → {d['tool']}({d['argument']!r})\n     why: {d['why']}\n     expected: {exp}\n")

In [ ]:
#@title 🧭 Routing decisions from your run { display-mode: "form" }
from IPython.display import HTML, display
_TC = {"semantic_search": "#667eea", "graph_neighbors": "#9a63d4",
       "graph_expand": "#e0796d", "answer": "#39b36a"}
_rows = ""
for _q, _d, _exp in routing:
    _c = _TC.get(_d["tool"], "#9aa0b5")
    _ok = _d["tool"] in _exp
    _rows += (f'<div class="ro-r"><div class="ro-q">{_esc(_q)}</div>'
              f'<div class="ro-t" style="background:{_c}">{_d["tool"]}</div>'
              f'<div class="ro-a">{_esc(_d["argument"])[:44]}</div>'
              f'<div class="ro-w">{_esc(_d["why"])}</div>'
              f'<div class="ro-e" style="color:{"#2f9e5c" if _ok else "#e0a23c"}">'
              f'{"✓ as expected" if _ok else "≠ expected: " + _exp}</div></div>')
display(HTML('''
<style>
.ro{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.ro-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.ro-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.ro-r{background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px;margin:7px 0;display:grid;grid-template-columns:1fr auto;gap:4px 10px;align-items:center}
.ro-q{font-size:12.5px;font-weight:700;color:#2c2350}
.ro-t{font-family:ui-monospace,Menlo,monospace;font-size:10.5px;font-weight:800;color:#fff;border-radius:7px;padding:4px 10px;text-align:center}
.ro-a{grid-column:1/3;font-family:ui-monospace,Menlo,monospace;font-size:10.5px;color:#4b3f7a;background:#f7f6fc;border-radius:6px;padding:3px 8px;justify-self:start}
.ro-w{grid-column:1/2;font-size:10.8px;color:#8b86a6;font-style:italic;line-height:1.45}
.ro-e{grid-column:2/3;font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.3px}
.ro-foot{margin-top:12px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="ro">
 <div class="ro-h">🧭 Three question shapes, three routes</div>
 <div class="ro-s">Routing is not about which retriever is stronger. It is about which one can <i>express</i> the sub-question at all.</div>
 __ROWS__
 <div class="ro-foot">💡 If your router collapsed onto one tool, that is a <b>prompt-engineering</b> bug, not a model limitation. The single most effective fix is stating what each tool <b>cannot</b> do.</div>
</div>'''.replace("__ROWS__", _rows)))

### 🎯 6.3 — The combined agent

The same loop as Part 2, with one substitution: instead of always searching, **ask the router what to
do**. Three lines to fill.

- **`choose_tool(question, evidence, notes, history)`** → `{"tool","argument","why"}`. Pass
  `history` — the same lesson as `past_queries` in Part 2. Without it the router happily re-runs
  `graph_expand("OS-17")` until `max_steps` and learns nothing; that is the single most common
  agent bug you will hit in practice.
- **`enough_evidence(...)` again.** When the router says `answer`, *check it*. The component that
  proposes an action is a poor judge of whether the job is done, and a premature stop is invisible:
  the system produces a confident, well-formatted, incomplete answer.
- **`run_tool(tool, argument)`** → `(facts, docs)` — structured facts *and* the passages behind them.
- Merge both: `notes` accumulates the graph facts, `evidence` accumulates deduplicated documents.

In [ ]:
def combined_investigate(question, max_steps=6, verbose=True):
    """Agentic RAG with a graph in the toolbox."""
    evidence, seen_ids, notes, trace, history = [], set(), [], [], []

    for step in range(max_steps):
        # 🎯 TODO 1: ask the router what to do next.
        #   choose_tool(question, evidence, notes, history) -> {"tool","argument","why"}
        #   Pass `history`, or the router re-runs graph_expand("OS-17") until max_steps.
        decision = ...          # 🎯 choose_tool(question, evidence, notes, history)
        tool, arg = decision["tool"], decision["argument"]

        # 🎯 TODO 2: the stopping decision is a routing decision — but do NOT take the
        #   router's word for it. Re-use the strict stopping rule from Part 2:
        #   enough_evidence(question, _with_graph_facts(evidence, notes))
        #   The component that proposes an action is a poor judge of whether it is done.
        if tool == "answer" and not evidence:
            tool, arg = "semantic_search", question       # guard: something to answer *from*
        if tool == "answer":
            verdict = ...       # 🎯 enough_evidence(question, _with_graph_facts(evidence, notes))

            if not verdict["sufficient"] and step < max_steps - 1:
                notes.append(f"STILL MISSING: {verdict['why']}")   # tell the router why it may not stop
                if verbose:
                    print(f"STEP {step + 1}  ⛔ router wanted to stop — audit says no: {verdict['why']}")
                trace.append(dict(step=step + 1, tool="blocked", argument="",
                                  why=f"router said stop; strict check disagreed → {verdict['why']}",
                                  facts=[], new_ids=[]))
                continue
            if verbose:
                print(f"STEP {step + 1}  ✅ answer — {decision['why']}")
            trace.append(dict(step=step + 1, tool="answer", argument="", why=decision["why"],
                              facts=[], new_ids=[]))
            break

        # 🎯 TODO 3: execute the action and merge BOTH kinds of result.
        #   run_tool(tool, arg) -> (facts: list[str], docs: list[dict])
        #   `notes` accumulates the structured graph facts (deduplicated),
        #   `evidence` accumulates the documents (deduplicated on doc["id"]),
        #   and `history` records the action so the router does not repeat it.
        history.append(...)     # 🎯 a string the router can read back, e.g. f"{tool}({arg})"
        facts, docs = ...       # 🎯 run_tool(tool, arg)
        notes.extend(...)       # 🎯 the entries of `facts` not already in `notes`
        new_docs = ...          # 🎯 the docs whose ["id"] is not yet in seen_ids
        evidence.extend(new_docs)
        seen_ids.update(d["id"] for d in new_docs)

        # --- bookkeeping for the trace visual (provided — leave it alone) ---
        trace.append(dict(step=step + 1, tool=tool, argument=arg, why=decision["why"],
                          facts=facts, new_ids=[d["id"] for d in new_docs]))
        if verbose:
            print(f"STEP {step + 1}  {tool}({arg!r})")
            print(f"         → {len(facts)} facts, {len(new_docs)} new docs "
                  f"{[d['id'] for d in new_docs] or ''}")

    return evidence, notes, trace

print("✅ combined_investigate() defined")

In [ ]:
comb_evidence, comb_notes, comb_trace = combined_investigate(CEO_QUESTION, max_steps=6)
print(f"\n📚 {len(comb_evidence)} documents · 🕸️ {len(comb_notes)} structured facts")

In [ ]:
#@title 🧭 Combined trace — watch it switch retrievers mid-investigation { display-mode: "form" }
from IPython.display import HTML, display
_TC = {"semantic_search": "#667eea", "graph_neighbors": "#9a63d4",
       "graph_expand": "#e0796d", "answer": "#39b36a", "blocked": "#e0a23c"}
_rows = ""
for _t in comb_trace:
    _c = _TC.get(_t["tool"], "#9aa0b5")
    _f = "".join(f'<div class="ct-f">{_esc(x)}</div>' for x in _t["facts"][:6])
    if len(_t["facts"]) > 6:
        _f += f'<div class="ct-more">+{len(_t["facts"]) - 6} more relations</div>'
    _n = "".join(f'<span class="ct-doc">{i}</span>' for i in _t["new_ids"]) or '<span class="ct-none">—</span>'
    _rows += f'''<div class="ct-s">
      <div class="ct-n" style="background:{_c}">{_t["step"]}</div>
      <div class="ct-b">
        <div class="ct-hd"><span class="ct-tool" style="color:{_c}">{_t["tool"]}</span>
          <span class="ct-arg">{_esc(_t["argument"])}</span></div>
        <div class="ct-why">{_esc(_t["why"])}</div>
        {'<div class="ct-lbl">structured facts from the graph</div>' + _f if _t["facts"] else ''}
        <div class="ct-lbl">new source documents</div><div>{_n}</div>
      </div></div>'''
display(HTML('''
<style>
.ct{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.ct-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.ct-sub{font-size:12px;color:#6b6685;margin:0 0 15px;line-height:1.55}
.ct-s{display:flex;gap:12px;margin-bottom:9px}
.ct-n{flex:0 0 28px;height:28px;border-radius:50%;color:#fff;font-weight:800;display:flex;align-items:center;justify-content:center;font-size:12.5px}
.ct-b{flex:1;background:#fff;border:1px solid #e7e4f6;border-radius:12px;padding:10px 13px}
.ct-hd{display:flex;gap:9px;align-items:baseline;flex-wrap:wrap}
.ct-tool{font-family:ui-monospace,Menlo,monospace;font-size:12px;font-weight:800}
.ct-arg{font-family:ui-monospace,Menlo,monospace;font-size:11px;color:#4b3f7a;background:#f7f6fc;border-radius:5px;padding:2px 7px}
.ct-why{font-size:10.8px;color:#8b86a6;font-style:italic;margin-top:3px;line-height:1.45}
.ct-lbl{font-size:9.5px;font-weight:800;text-transform:uppercase;letter-spacing:.4px;color:#b9a9e6;margin-top:7px}
.ct-f{font-family:ui-monospace,Menlo,monospace;font-size:10px;color:#4b3f7a;background:#f7f6fc;border-radius:5px;padding:3px 7px;margin-top:3px}
.ct-more{font-size:10px;color:#a9a3c4;font-style:italic;margin-top:3px}
.ct-doc{display:inline-block;font-family:ui-monospace,Menlo,monospace;font-size:10px;font-weight:700;background:#e7f6ee;color:#2f9e5c;border-radius:5px;padding:2px 7px;margin:3px 3px 0 0}
.ct-none{color:#c9c4dd;font-size:10.5px}
.ct-foot{margin-top:8px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="ct">
 <div class="ct-h">🧭 The combined investigation</div>
 <div class="ct-sub">One agent, alternating between prose retrieval and structure traversal, with each step chosen from what the last one returned.</div>
 __ROWS__
 <div class="ct-foot">🔑 The system is not "GraphRAG" or "vector RAG". It is an agent for which <b>the graph is one tool among several</b> — which is how these actually get built.</div>
</div>'''.replace("__ROWS__", _rows)))

In [ ]:
# Graph facts are EXHAUSTIVE in a way passages never are, and the synthesis prompt has to say so.
# Without this line the model reads the triples like prose and quietly drops the least familiar
# entity — which is precisely the one the graph was added to find.
GRAPH_ANSWER_SYSTEM = ANSWER_SYSTEM + (
    " Part of the evidence is a list of structured facts from a knowledge graph. Those are exact "
    "and exhaustive: treat them as a CHECKLIST, not as prose to skim. Account for every programme "
    "appearing in them explicitly, giving its dependency path.\n"
    "Critically: if the graph shows a programme is DOWNSTREAM of the disruption, that programme IS "
    "exposed — even when no document says so, and even when it shares no supplier and no component "
    "with the programmes already under discussion. That indirect, sub-tier path is precisely why "
    "the graph was consulted. Never conclude a programme is unaffected merely because a passage "
    "states it shares no parts; a passage cannot see past its own tier.")

combined_answer = llm(
    f"Evidence passages:\n{build_context(comb_evidence)}\n\n"
    f"Structured facts from the dependency knowledge graph:\n"
    + "\n".join("- " + n for n in comb_notes)
    + f"\n\nQuestion: {CEO_QUESTION}",
    system=GRAPH_ANSWER_SYSTEM)
ANSWERS["Agentic + GraphRAG"] = combined_answer
print(combined_answer)

In [ ]:
fact_coverage(combined_answer, "Agentic + GraphRAG · fact coverage",
              subtitle="Iterative retrieval found the causal chain; graph traversal found the exposures; the corpus grounded both.")

In [ ]:
#@title 🧠 The conflict hiding in that last answer — passages vs. structure { display-mode: "form" }
architecture(
    "Part 6 · Agentic + GraphRAG — one loop, three retrievers",
    [("q",     "❓", "the question", "no entry entity given &middot; it has to find its own"),
     ("route", "🧭", "router",       "<code>choose_tool</code> &middot; what <b>shape</b> is this gap?"),
     ("vec",   "🔀", [("vec", "semantic_search"), ("graph", "graph_neighbors"),
                      ("graph", "graph_expand")],
                     "one action per turn &middot; both graph tools also return their source passages"),
     ("text",  "🗂️", "evidence + facts", "passages <b>and</b> triples, deduplicated"),
     ("llm",   "⚖️", "enough?",      "the router may say stop &mdash; the strict check can overrule it"),
     ("ans",   "📝", "answer",       "the synthesis prompt calls the triples a <b>checklist</b>, not prose")],
    subtitle="Part 2's loop, with the tool chosen per turn. The graph is not the system — "
             "it is one tool inside it.",
    loop="not sufficient &rarr; back to the router, told which actions have already been taken",
    new=(1, 2),
    note="⚠️ Two boxes exist only because the component that <i>proposes</i> an action is a poor judge of "
         "whether the job is done: the history handed to the router, and the stopping rule that can veto it.")

explain("Two evidence types, and they disagree about Helios",
        "Your context window just contained both of these:<br><br>"
        "📄 <b>Passage D12</b> — “<i>Project Helios… no optical sensor of the OS series is used on this "
        "platform, and no part is shared with the Orion or Nova programmes.</i>”<br>"
        "🕸️ <b>Graph fact</b> — “<i>Project Helios is DOWNSTREAM of the Apex production halt (5 hops)</i>”"
        "<br><br>Both are <b>true</b>. The passage is right about tier 1 and blind past it; the graph sees "
        "the whole chain. Early drafts of this notebook produced answers that read <i>“Helios uses IR-9 "
        "from Meridian, <b>not</b> Apex — not exposed”</i>: the model believed the prose, because prose "
        "argues and triples merely sit there.", tone="risk", icon="⚔️")
explain("The fix is in how you present structure, not in the model",
        "Two changes to <code>GRAPH_ANSWER_SYSTEM</code> and <code>run_tool</code> did it: emit "
        "“<b>X is DOWNSTREAM of Y</b>” rather than a neutral “X → Y”, and state the semantics out loud — "
        "<i>reachable in the graph means exposed, even when a passage says the parts are unrelated.</i> "
        "Retrieval put the fact in the window; only <b>context engineering</b> got it into the answer.",
        tone="good", icon="🛠️")
explain("The generalisable rule",
        "Whenever you mix retrieval types in one prompt, say what each one <i>means</i> and which wins "
        "in a conflict. Unlabelled evidence is resolved by whichever source sounds most like an "
        "argument — and structured facts always lose that contest.", tone="info", icon="🔑")

---
# Part 7 — Local vs global: questions with no relevant document

Every system so far answered a **local** question — one with a specific answer sitting in a specific
corner of the corpus. Now try the question a board actually asks:

> **"What are our biggest cross-programme concentration risks?"**

Nothing in the corpus is *about* that. There is no "concentration risk" document to retrieve. This is
the distinction Microsoft's GraphRAG work calls **local search** (start from entities, traverse
locally) versus **global search** (summarise clusters of the graph, then reason over the summaries).

In [ ]:
print("🔍 vector search for a global question:\n")
for h in vector_search("biggest cross-programme concentration risks", top_k=4):
    print(f"   {h['score']:.2f}  {h['doc']['id']}  {h['doc']['title']}")

Note the **absolute scores**, not just the ranking. They are low and flat — the retriever is
returning "the least irrelevant" documents, which is what a retriever always does when nothing is
relevant. A confidently-worded answer built on those passages is a hallucination waiting to happen.

**🔧 PROVIDED — precomputed communities.** In a real GraphRAG pipeline these come from community
detection (Leiden/Louvain) over the graph, plus one LLM summary per community. We hand you the
result; the mechanism is one clustering call.

In [ ]:
#@title 🔧 PROVIDED — graph communities + their summaries (run, don't edit) { display-mode: "form" }
COMMUNITIES = [
    dict(name="Photonic supply chain",
         members=["Sanko Photonics", "Apex Components", "Meridian Optics", "OS-17", "OS-22",
                  "IR-9", "Project Orion", "Project Nova", "Project Vega", "Project Helios"],
         summary=("A single tier-2 vendor, Sanko Photonics, sits underneath BOTH qualified optical "
                  "integrators (Apex Components and Meridian Optics) following the 2025 wafer "
                  "sourcing consolidation. Four programmes — Orion, Nova, Vega and Helios — depend "
                  "on parts whose supply chains converge on that one company. The convergence is "
                  "invisible at tier 1: the four programmes appear to use three different parts "
                  "from two different suppliers.")),
    dict(name="Energy storage",
         members=["Voltix Energy", "BAT-4", "Project Luna", "Project Vega"],
         summary=("The BAT-4 battery module is single-sourced from Voltix Energy and is used by "
                  "Luna and Vega. Dual-sourcing was evaluated and deferred on cost grounds. No "
                  "incident has ever occurred here, which is precisely why this risk is invisible "
                  "in every incident-driven report and every retrieval over them.")),
    dict(name="Shared electronics manufacturing",
         members=["Kestrel Fabrication", "PCB-K3", "Project Orion", "Project Nova", "Project Luna"],
         summary=("The PCB-K3 mainboard from Kestrel Fabrication is common to Orion, Nova and Luna. "
                  "A disruption at Kestrel would hit three programmes simultaneously, including "
                  "both revenue-critical ones.")),
    dict(name="Cloud and software dependencies",
         members=["CloudSync Ltd", "CS-9", "Project Vega", "Project Atlas"],
         summary=("The CS-9 telemetry SDK from CloudSync Ltd underpins Vega and Atlas. CloudSync "
                  "had a short gateway outage in March 2026 with no data loss, but the dependency "
                  "is contractual and single-sourced.")),
]

def global_search(question):
    """Reason over community summaries instead of over passages."""
    body = "\n\n".join(
        f"COMMUNITY: {c['name']}\nMembers: {', '.join(c['members'])}\nSummary: {c['summary']}"
        for c in COMMUNITIES)
    return llm(f"Knowledge-graph community summaries:\n\n{body}\n\nQuestion: {question}\n\n"
               "Answer at the level of the portfolio. Rank the risks and justify the ranking by how "
               "many programmes each one touches and whether a second source exists.",
               system=ANSWER_SYSTEM)

print(f"✅ {len(COMMUNITIES)} communities covering {len({m for c in COMMUNITIES for m in c['members']})} entities")

In [ ]:
GLOBAL_QUESTION = "What are our biggest cross-programme concentration risks?"

local_attempt  = llm(f"Evidence:\n{build_context([h['doc'] for h in vector_search(GLOBAL_QUESTION, top_k=5)])}"
                     f"\n\nQuestion: {GLOBAL_QUESTION}", system=ANSWER_SYSTEM)
global_attempt = global_search(GLOBAL_QUESTION)

print("── LOCAL (top-5 passages) ─────────────────────\n" + local_attempt)
print("\n── GLOBAL (community summaries) ───────────────\n" + global_attempt)

In [ ]:
#@title 🌍 Local vs global — two different retrieval questions { display-mode: "form" }
from IPython.display import HTML, display
_comm = "".join(
    f'<div class="gl-c"><div class="gl-cn">{_esc(c["name"])}</div>'
    f'<div class="gl-cm">{len(c["members"])} entities · '
    f'{len([m for m in c["members"] if m.startswith("Project")])} programmes</div>'
    f'<div class="gl-cs">{_esc(c["summary"][:180])}…</div></div>' for c in COMMUNITIES)
display(HTML('''
<style>
.gl{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:920px;color:#2c2350}
.gl-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.gl-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.gl-two{display:flex;gap:12px;flex-wrap:wrap;margin-bottom:14px}
.gl-b{flex:1 1 250px;border-radius:12px;padding:12px 14px;font-size:11.5px;line-height:1.6}
.gl-l{background:#fdf3f1;border-left:4px solid #e0796d;color:#7a3d34}
.gl-g{background:#eafaf0;border-left:4px solid #39b36a;color:#1e6b40}
.gl-grid{display:flex;gap:9px;flex-wrap:wrap}
.gl-c{flex:1 1 200px;background:#fff;border:1px solid #e7e4f6;border-top:3px solid #9a63d4;border-radius:12px;padding:10px 12px}
.gl-cn{font-size:12px;font-weight:800;color:#2c2350}
.gl-cm{font-size:10px;color:#9a63d4;font-weight:700;margin:2px 0 5px}
.gl-cs{font-size:10.5px;color:#5b5578;line-height:1.5}
.gl-foot{margin-top:13px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
</style>
<div class="gl">
 <div class="gl-h">🌍 Local search answers "about X". Global search answers "about the whole".</div>
 <div class="gl-s">The retrieval unit changes. Local search retrieves <b>passages near an entity</b>; global search retrieves <b>summaries of regions of the graph</b>.</div>
 <div class="gl-two">
  <div class="gl-b gl-l"><b>❌ Local, on a global question</b><br>Top-k returns whatever is least irrelevant, at low and undifferentiated scores. The model answers fluently from an arbitrary five documents and cannot know that it is missing three quarters of the portfolio. <b>Low similarity scores across the board are the tell.</b></div>
  <div class="gl-b gl-g"><b>✅ Global, on a global question</b><br>Every community is represented, so coverage is structural rather than lucky. Risks that produced <i>no incident and no document</i> — like the BAT-4 single-source — surface anyway, because they are visible in the shape of the graph.</div>
 </div>
 <div class="gl-grid">__COMM__</div>
 <div class="gl-foot">💡 <b>Cost note for the managers in the room:</b> global search is not free. Communities have to be detected and summarised offline, and re-summarised as the corpus changes. Build it when your organisation genuinely asks portfolio-level questions — not because the architecture diagram looks impressive.</div>
</div>'''.replace("__COMM__", _comm)))

architecture(
    "Part 7 · Global search — retrieve regions, not passages",
    [("q",     "❓", "the question",  "portfolio-level &middot; no document is <i>about</i> it"),
     ("off",   "🏗️", "communities",   "detected and summarised <b>offline</b>, once per corpus version"),
     ("graph", "🧩", "every summary", "all of them &middot; nothing is selected, ranked or dropped"),
     ("llm",   "🤖", "LLM",           "reason and rank across regions of the graph"),
     ("ans",   "📝", "answer",        "coverage is structural rather than lucky")],
    subtitle="The retrieval unit itself changes. There is no query embedding and no top-k in this row at all.",
    new=(1, 2),
    note="💰 The offline box is the entire cost of this architecture: communities must be re-detected and "
         "re-summarised every time the corpus moves. Build it when the questions really are portfolio-level.")


In [ ]:
#@title 🏆 The scoreboard — every system you built, scored on your run { display-mode: "form" }
from IPython.display import HTML, display

_SYS = ["Classical RAG", "Agentic RAG", "GraphRAG (path + sources)",
        "Agentic + GraphRAG"]
_SUB = {"Classical RAG": "one shot · top-5",
        "Agentic RAG": "iterative retrieval",
        "GraphRAG (path + sources)": "traversal, entry point chosen by hand",
        "Agentic + GraphRAG": "routed agent, both retrievers"}
_hdr = "".join(f'<th class="sc-th"><div class="sc-rot">{g["label"].split(":")[0].split("(")[0].strip()}</div></th>' for g in GOLD)
_rows = ""
for _s in _SYS:
    if _s not in ANSWERS:
        continue
    _got = score_answer(ANSWERS[_s]); _n = sum(_got.values())
    _cells = "".join(
        f'<td class="sc-td"><span class="sc-m {"y" if _got[g["key"]] else "n"}">'
        f'{"✓" if _got[g["key"]] else "✗"}</span></td>' for g in GOLD)
    _col = "#39b36a" if _n == len(GOLD) else ("#e0a23c" if _n >= 4 else "#e0796d")
    _rows += (f'<tr><td class="sc-name"><b>{_s}</b><div class="sc-sub">{_SUB.get(_s,"")}</div></td>'
              f'{_cells}<td class="sc-tot" style="color:{_col}">{_n}/{len(GOLD)}</td></tr>')

_cost = (f'<b>{LLM_CALLS["n"]}</b> LLM calls · <b>{SEARCH_CALLS["vector"]}</b> vector searches · '
         f'<b>{SEARCH_CALLS["keyword"]}</b> keyword searches · <b>{SEARCH_CALLS["graph"]}</b> graph lookups '
         f'· model <code>{MODEL}</code>')

display(HTML('''
<style>
.sc{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);border:1px solid #ecebff;border-radius:18px;padding:20px;max-width:960px;color:#2c2350}
.sc-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.sc-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.55}
.sc-wrap{overflow-x:auto;background:#fff;border:1px solid #e7e4f6;border-radius:13px;padding:8px}
table.sc-t{border-collapse:collapse;width:100%;min-width:620px}
.sc-th{font-size:9.5px;color:#6b6685;font-weight:800;text-align:center;padding:0 3px 8px;vertical-align:bottom;max-width:74px;line-height:1.25}
.sc-rot{white-space:normal}
.sc-name{font-size:12px;color:#2c2350;padding:8px 10px 8px 4px;border-top:1px solid #f0eef9;min-width:170px}
.sc-sub{font-size:10px;color:#a9a3c4;font-weight:400;margin-top:1px}
.sc-td{text-align:center;border-top:1px solid #f0eef9;padding:8px 3px}
.sc-m{display:inline-flex;width:21px;height:21px;border-radius:6px;align-items:center;justify-content:center;font-weight:800;font-size:12px;color:#fff}
.sc-m.y{background:#39b36a}.sc-m.n{background:#e8dfe0;color:#c4a9a4}
.sc-tot{text-align:center;font-weight:800;font-size:14px;border-top:1px solid #f0eef9;padding:8px 6px;font-variant-numeric:tabular-nums}
.sc-cost{margin-top:11px;font-size:11.5px;color:#4b3f7a;background:#f2f0fc;border-left:3px solid #667eea;border-radius:9px;padding:10px 12px;line-height:1.6}
.sc-cost code{background:#fff;border-radius:4px;padding:1px 5px;font-size:10.5px}
</style>
<div class="sc">
 <div class="sc-h">🏆 Same question, same corpus, same model — different architectures</div>
 <div class="sc-s">Scored automatically against the six facts a complete answer to the CEO must contain. These are the numbers from <b>your</b> run, so expect small variation between runs — the <i>shape</i> is the stable part.</div>
 <div class="sc-wrap"><table class="sc-t"><thead><tr><th></th>__HDR__<th class="sc-th">total</th></tr></thead>
 <tbody>__ROWS__</tbody></table></div>
 <div class="sc-cost">💰 <b>What it cost:</b> __COST__<br>
   Recall is bought with calls. That is the trade-off to put in front of a budget owner — and the reason the right answer is often "agentic retrieval for the hard 5% of queries, one-shot RAG for the rest".</div>
</div>'''.replace("__HDR__", _hdr).replace("__ROWS__", _rows).replace("__COST__", _cost)))

---
# 🎓 Wrap-up — four architectures, four different jobs

You asked one question four ways. Each architecture failed at something the next one fixed:

| | **Classical RAG** | **Agentic RAG** | **GraphRAG** | **Agentic + Graph** |
|---|---|---|---|---|
| **Retrieval unit** | passage | passage, repeatedly | relation / path | whichever fits the turn |
| **Answers well** | "what does the corpus say about X?" | "trace this causal chain" | "who else, what connects, blast radius" | all of the above |
| **Fails at** | anything spanning documents | facts that exist only as a join | narrative, dates, nuance, causes | nothing here — but it costs the most |
| **Cost per query** | 1 LLM call | 2–3 per step | ~free traversal, + extraction offline | highest |
| **Build cost** | embeddings | + a loop and a stopping rule | + entity/relation extraction, maintained | + routing prompt |
| **Breaks when** | questions get multi-hop | the fact is not written down | the graph is stale or wrong | — |

### The four lessons underneath the exercise

1. **Classical RAG works when one retrieval step can surface the answer.** Most questions are like
   this. Do not over-engineer them.
2. **Agentic RAG helps when the information need evolves as evidence arrives** — when the query you
   need next is unwritable until the previous result comes back.
3. **GraphRAG helps when relationships *are* the information** — shared dependencies, blast radius,
   "who else". An embedding cannot approximate a join.
4. **The strongest system combines them, and the combination is a routing problem** — plus the
   discipline of always coming back to the source text.

### The one question to ask instead of "should we use GraphRAG?"

> **What kinds of questions does my organisation actually need to answer — and do they require
> search, iterative investigation, explicit relationships, or all three?**

If your questions look like *"what does the policy say about X"*, a graph is expensive decoration.
If they look like *"what else breaks if this vendor fails"*, no amount of embedding tuning will get
you there — because **that fact is not written down anywhere, and it never will be**.

### 🎯 Transfer test — classify four new questions

The framework is only worth something if it generalises. For each question below, decide which
retrieval architecture is the *cheapest one that can actually answer it*. Reason it out before
running the cell — the asserts encode the intended answers.

In [ ]:
# 🎯 TODO: fill in one of  "classical" | "agentic" | "graph" | "global"  for each.
#          Decide before you run the cell — the asserts below encode the intended answers.
triage = {
    # 🎯 is the answer inside one passage, or spread between several?
    "What is our standard payment term with Voltix Energy?":                      "???",

    # 🎯 do you want a ranked guess here, or an exhaustive join?
    "Which programmes would be affected if Kestrel Fabrication went bankrupt?":   "???",

    # 🎯 could you write the second query before seeing the first result?
    "Why did the Q1 margin on Nova come in below plan?":                          "???",

    # 🎯 is any single document actually *about* this question?
    "Which suppliers should we dual-source first, across the whole portfolio?":   "???",
}

assert triage["What is our standard payment term with Voltix Energy?"] == "classical", \
    "one passage states it — a loop and a graph are pure overhead here"
assert triage["Which programmes would be affected if Kestrel Fabrication went bankrupt?"] == "graph", \
    "exhaustive one-hop dependency lookup: exactly what get_neighbors does, with no ranking risk"
assert triage["Why did the Q1 margin on Nova come in below plan?"] == "agentic", \
    "a causal chain — each query becomes writable only after the previous result"
assert triage["Which suppliers should we dual-source first, across the whole portfolio?"] == "global", \
    "portfolio-level: no single document is about it, so summarise communities and rank"
print("✅ all four triaged correctly\n")
for q, a in triage.items():
    print(f"  {a:<10} ← {q}")

### 🧠 Final checks

Answer these from what you saw, not from memory of the slides:

- **Which system found Helios, and why could no other system have found it?**
  *(Only the graph. The Orion–Helios relationship is a six-hop path across six documents; it exists in
  no passage, so there is nothing for a similarity score to rank.)*
- **You double `top_k` to 20 in Part 1. Which of the six facts does that recover?**
  *(Possibly the Apex one, by luck. Not Sanko, and never Helios — recall is not the binding
  constraint, expressiveness is.)*
- **Your graph says `Apex --supplies--> OS-17`, extracted from a 2025 register. Procurement
  re-sources OS-17 tomorrow. What happens, and how would you notice?**
  *(The system keeps asserting it, confidently, with a citation. You notice only if edges carry
  valid-from dates and are re-extracted — graph maintenance is the hidden cost of GraphRAG.)*
- **Which part of the exercise would have failed identically with a frontier model ten times larger?**
  *(All of Part 1 and Part 3. Retrieval decides what enters the context window; a model cannot reason
  over what it was never shown.)*
- **The CEO asks the same question next quarter about a different programme. What do you have to
  rebuild?** *(Nothing — that is the payoff of building an architecture rather than an answer.)*

### Where this goes next

- **Agentic RAG** — Self-RAG (Asai et al., 2023); FLARE (Jiang et al., 2023) — active retrieval
  driven by the model's own uncertainty.
- **Multi-hop QA** — IRCoT (Trivedi et al., 2023) — interleaving chain-of-thought with retrieval,
  which is the formal version of your `investigate()` loop.
- **GraphRAG** — Edge et al., 2024, *From Local to Global: A Graph RAG Approach to Query-Focused
  Summarization* (Microsoft) — the source of the local/global distinction in Part 7.
- **The build step we skipped** — entity and relation extraction with an LLM, plus entity
  resolution. In practice this is where most of the engineering effort and most of the errors live.
- **Next block** — evaluation: how would you *measure* that your six-fact scorecard generalises
  beyond one question? (Hint: the scorecard in this notebook is a hand-built eval set of size one.)